# Roshni: offline diabetic eye triage for primary-care diabetes clinics in Pakistan

**Track:** AI + Health. **Team:** Varun Bajpai (India), Laziza Farrukhovna (Uzbekistan), Zainab Jamshed (Pakistan).  
**User:** a doctor or nurse at a primary-care diabetes clinic in Pakistan that can take retinal photos but has no eye specialist on site. **Moment:** right after the photo is taken at a routine diabetes visit.  
**Output:** REFER to an eye specialist, RETAKE the photo, or RESCREEN in 12 months, with a note in English and Urdu.

> This project is a prototype and is not medical advice.

### Before you run
**Kaggle (recommended, no big download):** New Notebook → *Add Input* → search **APTOS 2019 Blindness Detection** (join the competition and accept its rules first) → *Settings*: Accelerator **GPU**, Internet **On** (needs a phone-verified account). Then **Save Version → Save & Run All (Commit)**: it runs in the background, so you can close the tab. When it finishes, open the version's **Output** and download `roshni_outputs.zip`.

**Colab:** Runtime → GPU. You will be asked for your `kaggle.json` API token (Kaggle → Settings → Create New Token), and the competition rules must already be accepted on Kaggle. The download is about 10 GB.

Full run: roughly 45 to 60 minutes on a free GPU, including a quick check, the stress test and the web-demo export.

**What it produces:** results with baselines, a held-out test and an external test; a stress test on deliberately worse photos (planned before any result); the write-up and video script filled from the results; and `web/`, a demo that runs entirely in the browser (publish it with GitHub Pages).

## 1. Code
Written to disk so the notebook is self-contained.

In [ ]:
%%writefile drscreen.py
"""Roshni: diabetic-eye screening for a primary-care diabetes clinic in Pakistan.

One retinal photo in, one of three next steps out:
  REFER     - referable diabetic retinopathy suspected (moderate NPDR or worse)
  RETAKE    - photo not good enough to give an all-clear; take it again
  RESCREEN  - no referable disease seen on a good photo; screen again in 12 months

Safety rules built into the code:
  * A poor photo can never produce an all-clear, but it never blocks a referral either.
  * The referral threshold is chosen on the validation split to reach a sensitivity target
    fixed in advance (90%), then frozen before any test image is scored.
  * Near-duplicate photos are grouped before splitting, so a copy of a test image can never
    sit in training (a common way to get results that are too good to be true).
  * The external test set (DeepDRiD, China) is never used for training or for any threshold.

Prototype for a hackathon. Not medical advice and not a diagnosis.
"""
from __future__ import annotations

import os
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

GRADES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
REFERABLE_FROM = 2           # moderate NPDR or worse = "more-than-mild DR", the usual referral line
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


# ============================================================ preprocessing
def crop_fundus(img: Image.Image, tol: int = 10) -> Image.Image:
    """Cut away the black border around the round fundus image."""
    a = np.asarray(img.convert("L"))
    mask = a > tol
    if mask.sum() < 0.05 * a.size:          # almost all dark: leave it (the quality gate will judge it)
        return img
    ys, xs = np.where(mask)
    return img.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))


def prep_image(path_or_img, size: int = 384) -> Image.Image:
    img = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    img = crop_fundus(img.convert("RGB"))
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (0, 0, 0))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.BILINEAR)


def _cache_one(args):
    src, dst, size = args
    if not os.path.exists(dst):
        try:
            prep_image(src, size).save(dst, quality=95)
        except Exception as e:                # a broken file should not stop the run
            return f"FAILED {src}: {e}"
    return None


def cache_images(srcs, out_dir, size=384, workers=4):
    """Crop, pad and resize every image once, so training reads small JPEGs."""
    from multiprocessing import Pool
    os.makedirs(out_dir, exist_ok=True)
    dsts = [os.path.join(out_dir, os.path.splitext(os.path.basename(s))[0] + ".jpg") for s in srcs]
    jobs = [(s, d, size) for s, d in zip(srcs, dsts)]
    t0 = time.time()
    with Pool(workers) as pool:
        errs = [e for e in pool.imap(_cache_one, jobs, chunksize=8) if e]
    print(f"Cached {len(dsts)} images to {out_dir} in {time.time() - t0:.0f}s; {len(errs)} failed")
    for e in errs[:5]:
        print(" ", e)
    return dsts


# ============================================================ near-duplicates
def dhash(path, n: int = 16) -> np.ndarray:
    a = np.asarray(Image.open(path).convert("L").resize((n + 1, n), Image.BILINEAR), dtype=np.float32)
    return (a[:, 1:] > a[:, :-1]).ravel()


def duplicate_groups(paths, max_dist: int = 10):
    """Group photos whose 256-bit difference hashes are within `max_dist` bits.
    Returns (group id per image, list of near-duplicate pairs)."""
    H = np.stack([dhash(p) for p in paths]).astype(np.float32)
    n_bits = H.shape[1]
    parent = list(range(len(paths)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    pairs = []
    for s in range(0, len(H), 1024):
        blk = H[s:s + 1024]
        same = blk @ H.T + (1 - blk) @ (1 - H).T            # bits that agree
        dist = n_bits - same
        ii, jj = np.where(dist <= max_dist)
        for i, j in zip(ii + s, jj):
            if i < j:
                pairs.append((int(i), int(j), int(dist[i - s, j])))
                ri, rj = find(i), find(j)
                if ri != rj:
                    parent[ri] = rj
    groups = np.array([find(i) for i in range(len(paths))])
    return groups, pairs


def group_split(labels, groups, seed: int = 0, n_splits: int = 7):
    """Seven stratified folds: one test, one validation, five training. Near-duplicate groups stay whole, so the
    actual shares can differ from 1/7 each (the real split is reported from aptos_split.csv)."""
    from sklearn.model_selection import StratifiedGroupKFold
    sgk = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.zeros(len(labels), int)
    for k, (_, idx) in enumerate(sgk.split(np.zeros(len(labels)), labels, groups)):
        folds[idx] = k
    split = np.where(folds == 0, "test", np.where(folds == 1, "val", "train"))
    return split


# ============================================================ simple baseline features
def color_features(path) -> np.ndarray:
    """16-bin histograms per colour channel over the fundus (black border ignored) + mean/std."""
    a = np.asarray(Image.open(path).convert("RGB").resize((128, 128)), dtype=np.float32)
    m = a.sum(axis=2) > 30
    if m.sum() < 50:
        m = np.ones(a.shape[:2], bool)
    feats = []
    for ch in range(3):
        v = a[..., ch][m]
        h, _ = np.histogram(v, bins=16, range=(0, 255))
        feats += list(h / max(1, h.sum())) + [v.mean() / 255, v.std() / 255]
    return np.array(feats, np.float32)


# ============================================================ torch parts
def _torch():
    import torch
    return torch


def get_transforms(size: int, train: bool):
    from torchvision import transforms as T
    if train:
        return T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
                          T.ColorJitter(0.2, 0.2, 0.1), T.ToTensor(), T.Normalize(MEAN, STD)])
    return T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])


def make_dataset(paths, labels, size, train, degrade_fn=None):
    """degrade_fn (optional) makes each photo worse in a fixed way before scoring; used only by the stress test."""
    from torch.utils.data import Dataset
    tf = get_transforms(size, train)

    class _DS(Dataset):
        def __len__(self):
            return len(paths)

        def __getitem__(self, i):
            img = Image.open(paths[i]).convert("RGB")
            if img.size != (size, size):
                img = img.resize((size, size))
            if degrade_fn is not None:
                img = degrade_fn(img)
            return tf(img), int(labels[i]) if labels is not None else -1
    return _DS()


def build_model(arch: str = "efficientnet_b0", pretrained: bool = True, n_classes: int = 5):
    import torch.nn as nn
    import torchvision
    if arch == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    elif arch == "resnet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    else:
        raise ValueError(arch)
    m.arch = arch
    return m


def _embed(model, x):
    """Penultimate-layer features (used by the linear-probe baseline and the quality gate)."""
    torch = _torch()
    if getattr(model, "arch", "") == "resnet50":
        f = torch.nn.Sequential(*list(model.children())[:-1])(x)
    else:
        f = model.avgpool(model.features(x))
    return torch.flatten(f, 1)


def predict(model, ds, batch_size=32, device=None, workers=2, with_embeddings=False):
    torch = _torch()
    from torch.utils.data import DataLoader
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers)
    P, E = [], []
    with torch.no_grad():
        for x, _ in dl:
            x = x.to(device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                if with_embeddings:
                    e = _embed(model, x)
                    logits = model.classifier(e) if hasattr(model, "classifier") else model.fc(e)
                    E.append(e.float().cpu().numpy())
                else:
                    logits = model(x)
            P.append(torch.softmax(logits.float(), 1).cpu().numpy())
    P = np.concatenate(P)
    return (P, np.concatenate(E)) if with_embeddings else P


def referable_prob(P: np.ndarray) -> np.ndarray:
    return P[:, REFERABLE_FROM:].sum(axis=1)


def train_model(model, train_ds, val_ds, val_grades, epochs=12, lr=3e-4, batch_size=32,
                device=None, workers=2, out_path="model_best.pt", log=print):
    """Train; keep the epoch with the best validation AUC for referable DR (validation only)."""
    torch = _torch()
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=workers, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * max(1, len(dl)))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
    yv = (np.asarray(val_grades) >= REFERABLE_FROM).astype(int)
    best, hist = -1.0, []
    for ep in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y)
            n += len(y)
        pv = referable_prob(predict(model, val_ds, batch_size, device, workers))
        auc = roc_auc_score(yv, pv) if len(set(yv)) > 1 else float("nan")
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(1, n), val_auc=auc, seconds=time.time() - t0))
        log(f"epoch {ep + 1}/{epochs}: loss {tot / max(1, n):.3f}, validation AUC (referable) {auc:.3f}, {time.time() - t0:.0f}s")
        if auc > best:
            best = auc
            torch.save(model.state_dict(), out_path)
    model.load_state_dict(torch.load(out_path, map_location=device))
    return pd.DataFrame(hist)


# ============================================================ thresholds and metrics
def threshold_for_sensitivity(y_bin, p, target=0.90) -> float:
    """Highest threshold whose sensitivity on these (validation) cases is at least `target`."""
    y_bin, p = np.asarray(y_bin), np.asarray(p)
    pos = np.sort(p[y_bin == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    k = int(np.ceil(target * len(pos))) - 1            # need at least k+1 positives above the line
    return float(pos[k])


def bootstrap_auc(y, p, n=1000, seed=0):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) > 1:
            vals.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (float(c - h), float(c + h))


def binary_report(y_bin, p, thr, name="") -> dict:
    from sklearn.metrics import roc_auc_score
    y_bin, p = np.asarray(y_bin).astype(int), np.asarray(p)
    pred = p >= thr
    tp, fn = int((pred & (y_bin == 1)).sum()), int((~pred & (y_bin == 1)).sum())
    fp, tn = int((pred & (y_bin == 0)).sum()), int((~pred & (y_bin == 0)).sum())
    auc = roc_auc_score(y_bin, p) if len(set(y_bin)) > 1 else float("nan")
    return dict(model=name, n=len(y_bin), referable=int(y_bin.sum()), threshold=float(thr),
                AUC=auc, AUC_CI=bootstrap_auc(y_bin, p) if len(set(y_bin)) > 1 else (np.nan, np.nan),
                sensitivity=tp / max(1, tp + fn), sensitivity_CI=wilson(tp, tp + fn),
                specificity=tn / max(1, tn + fp), specificity_CI=wilson(tn, tn + fp),
                PPV=tp / max(1, tp + fp), NPV=tn / max(1, tn + fn),
                missed_FN=fn, false_alarms_FP=fp, referred_share=float(pred.mean()))


def grade_report(grades, P) -> dict:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix
    g = np.asarray(grades).astype(int)
    pred = P.argmax(1)
    pr = referable_prob(P)
    per_grade = {GRADES[k]: dict(n=int((g == k).sum())) for k in range(5)}
    return dict(QWK=float(cohen_kappa_score(g, pred, weights="quadratic")),
                confusion=confusion_matrix(g, pred, labels=list(range(5))).tolist(),
                per_grade=per_grade, mean_referable_prob_by_grade=[float(pr[g == k].mean()) if (g == k).any() else None
                                                                   for k in range(5)])


def flagged_by_grade(grades, p, thr) -> pd.DataFrame:
    g = np.asarray(grades).astype(int)
    rows = []
    for k in range(5):
        m = g == k
        rows.append(dict(grade=f"{k} {GRADES[k]}", photos=int(m.sum()),
                         sent_to_specialist=int((p[m] >= thr).sum()),
                         share_sent=float((p[m] >= thr).mean()) if m.any() else np.nan))
    return pd.DataFrame(rows)


def fit_logreg(X, y, C=1.0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))
    return m.fit(X, y)


# ============================================================ the clinic decision
def decide(p_ref, p_poor, ref_thr, poor_thr):
    """Refer whenever referable DR is suspected (a poor photo never blocks a referral).
    Otherwise a poor photo gets RETAKE instead of an all-clear."""
    p_ref, p_poor = np.asarray(p_ref), np.asarray(p_poor)
    return np.where(p_ref >= ref_thr, "REFER", np.where(p_poor >= poor_thr, "RETAKE", "RESCREEN"))


def workflow_report(grades, decisions) -> dict:
    g = np.asarray(grades).astype(int)
    d = np.asarray(decisions)
    ref = g >= REFERABLE_FROM
    return dict(photos=len(g),
                REFER=int((d == "REFER").sum()), RETAKE=int((d == "RETAKE").sum()), RESCREEN=int((d == "RESCREEN").sum()),
                referable_given_all_clear=int((ref & (d == "RESCREEN")).sum()),
                referable_caught_by_retake=int((ref & (d == "RETAKE")).sum()),
                sight_threatening_given_all_clear=int(((g >= 3) & (d == "RESCREEN")).sum()))


# ============================================================ stress test: worse photos
# Planned before any result was seen. Each change is applied to the 384 px photo, at two fixed strengths,
# to mimic what cheaper cameras, rushed photography or sending photos by phone do to image quality.
# It is a stress test of the frozen tool, not a validation on real low-cost cameras.
DEGRADATIONS = {
    #  name               what it imitates                                   mild   strong
    "blur":            ("out of focus or camera shake (Gaussian blur, px)",  1.5,   3.0),
    "low_resolution":  ("a cheaper, lower-resolution sensor (shrunk to px)", 192,   96),
    "underexposed":    ("too dark: poor lighting or a small pupil (brightness x)", 0.6, 0.35),
    "compressed":      ("sent over a messaging app (JPEG quality)",          30,    8),
}


def degrade(img: Image.Image, kind: str, level) -> Image.Image:
    """Make a photo worse in one fixed, repeatable way (no randomness)."""
    import io
    from PIL import ImageEnhance, ImageFilter
    img = img.convert("RGB")
    if kind == "clean":
        return img
    if kind == "blur":
        return img.filter(ImageFilter.GaussianBlur(float(level)))
    if kind == "low_resolution":
        w, h = img.size
        return img.resize((int(level), int(level)), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if kind == "underexposed":
        return ImageEnhance.Brightness(img).enhance(float(level))
    if kind == "compressed":
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=int(level))
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    raise ValueError(kind)


def stress_conditions():
    """[(change, strength, level)], starting with the original photos."""
    out = [("clean", "none", None)]
    for kind, (_, mild, strong) in DEGRADATIONS.items():
        out += [(kind, "mild", mild), (kind, "strong", strong)]
    return out


# ============================================================ one photo, as the clinic app runs it
def triage_one(model, gate, img, size, ref_thr, poor_thr, tf=None):
    """Screen one photo exactly as the clinic app does: open, crop, resize, score, quality check, decide.
    img is a PIL image or a file path. Returns (decision, p_referable, p_poor, grade_probabilities, embedding)."""
    torch = _torch()
    model.eval()
    tf = tf or get_transforms(size, False)
    dev = next(model.parameters()).device
    x = tf(prep_image(img, size)).unsqueeze(0).to(dev)
    with torch.no_grad():
        e = _embed(model, x)
        head = model.classifier if hasattr(model, "classifier") else model.fc
        P = torch.softmax(head(e), 1).cpu().numpy()
    e = e.cpu().numpy()
    p_ref = float(referable_prob(P)[0])
    p_poor = float(gate.predict_proba(e)[0, 1]) if gate is not None else 0.0
    return str(decide([p_ref], [p_poor], ref_thr, poor_thr)[0]), p_ref, p_poor, P[0], e


def _cpu_name() -> str:
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    import platform
    return platform.processor() or platform.machine() or "unknown"


def cpu_benchmark(model, gate, paths, size, ref_thr, poor_thr, n=20, heatmaps=5) -> dict:
    """Time the clinic step on the CPU alone (no graphics card), starting from the original photo file.
    The first photo is a warm-up and is not timed."""
    import copy
    torch = _torch()
    m = copy.deepcopy(model).cpu().eval()
    paths = list(paths)[: n + 1]
    if len(paths) < 2:
        raise ValueError("need at least 2 photos to time")
    tf = get_transforms(size, False)
    triage_one(m, gate, paths[0], size, ref_thr, poor_thr, tf)
    times = []
    for p in paths[1:]:
        t = time.perf_counter()
        triage_one(m, gate, p, size, ref_thr, poor_thr, tf)
        times.append(time.perf_counter() - t)
    hm = []
    for p in paths[1: 1 + heatmaps]:
        t = time.perf_counter()
        gradcam(m, Image.open(p).convert("RGB"), size, device="cpu")
        hm.append(time.perf_counter() - t)
    return dict(photos_timed=len(times), median_seconds_per_photo=float(np.median(times)),
                slowest_seconds_per_photo=float(np.max(times)),
                heatmap_median_seconds=float(np.median(hm)) if hm else float("nan"),
                cpu=_cpu_name(), cpu_threads=int(torch.get_num_threads()),
                parameters=int(sum(q.numel() for q in m.parameters())))


# ============================================================ explanation
def gradcam(model, pil_img, size=384, device=None):
    """Heat map of where the model looked when scoring referable DR (Grad-CAM on the last conv block), drawn on the photo."""
    cam = gradcam_map(model, pil_img, size, device)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize((size, size), Image.BILINEAR)
    base = prep_image(pil_img, size)
    heat = np.asarray(cam_img, dtype=np.float32) / 255
    rgb = np.asarray(base, dtype=np.float32)
    overlay = rgb * (1 - 0.45 * heat[..., None]) + np.array([255, 40, 0]) * (0.45 * heat[..., None])
    return Image.fromarray(np.uint8(np.clip(overlay, 0, 255)))


def gradcam_map(model, pil_img, size=384, device=None):
    """The raw Grad-CAM map (last conv block, scaled to 0..1) for the referable score."""
    torch = _torch()
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    x = get_transforms(size, False)(prep_image(pil_img, size)).unsqueeze(0).to(device)
    layer = model.features[-1] if hasattr(model, "features") else model.layer4
    store = {}
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("g", go[0]))
    try:
        logits = model(x)
        score = torch.logsumexp(logits[:, REFERABLE_FROM:], 1) - torch.logsumexp(logits, 1)
        model.zero_grad()
        score.sum().backward()
        w = store["g"].mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * store["a"]).sum(1))[0].detach().cpu().numpy()
    finally:
        h1.remove()
        h2.remove()
    cam = cam - cam.min()
    return cam / (cam.max() + 1e-8)


# ============================================================ report text
# Languages for the note. A language is offered to users only once a native speaker on the team has checked it,
# except the defaults for the Pakistan setting, which the submission checker refuses to ship unchecked.
# To add a language later (for example Uzbek, checked by Laziza): add an entry here with checked=False, add its
# sentence to every entry in REPORT (tests fail until all four are there), have the reviewer approve the wording,
# then set checked=True. Nothing else in the code changes.
LANGS = {
    "en": dict(name="English", dir="ltr", reviewer="whole team", checked=True),
    "ur": dict(name="اردو", dir="rtl", reviewer="Zainab", checked=True),    # reviewed and corrected by Zainab, 15 Nov 2026
    "hi": dict(name="हिन्दी", dir="ltr", reviewer="Varun", checked=False),   # hidden until Varun checks it
    "uz": dict(name="Oʻzbekcha", dir="ltr", reviewer="Laziza", checked=False),   # hidden until Laziza checks it
}
# Shown by default for the Pakistan clinic setting.
DEFAULT_LANGS = ("en", "ur")


def shown_langs():
    """Languages the app offers: every checked language, plus the defaults (which must be checked before submission)."""
    return [c for c, v in LANGS.items() if v["checked"] or c in DEFAULT_LANGS]


# Same meaning in every language: the action, why, and that the doctor confirms it.
REPORT = {
    "REFER": {
        "en": "Referable diabetic retinopathy suspected (moderate or worse). Recommended next step, for the doctor to confirm: refer to an eye specialist.",
        "hi": "मधुमेह से आँख के पर्दे (रेटिना) की मध्यम या उससे गंभीर बीमारी (डायबिटिक रेटिनोपैथी) की आशंका है। अनुशंसित अगला कदम, डॉक्टर की पुष्टि के बाद: नेत्र विशेषज्ञ के पास भेजें।",
        "ur": "ذیابیطس کے باعث آنکھ کے پردے (ریٹینا) میں درمیانی یا شدید درجے کی بیماری کا شبہ ہے۔ ڈاکٹر کی تصدیق کے بعد، اگلا قدم: مریض کو آنکھوں کے ماہر کے پاس بھیجیں۔",
        "uz": "Yuborishni talab qiluvchi diabetik retinopatiya gumon qilinmoqda (oʻrtacha yoki ogʻirroq). Shifokor tasdiqlashi uchun tavsiya etilgan keyingi qadam: koʻz mutaxassisiga yuborish.",
    },
    "RETAKE": {
        "en": "The photo is not clear enough to rule out disease. Recommended next step: retake the photo; if it is still unclear, refer to an eye specialist.",
        "hi": "तस्वीर इतनी साफ़ नहीं है कि बीमारी को नकारा जा सके। अनुशंसित अगला कदम: तस्वीर दोबारा लें; अगर फिर भी साफ़ न हो, तो नेत्र विशेषज्ञ के पास भेजें।",
        "ur": "تصویر اتنی واضح نہیں کہ بیماری کی موجودگی یا عدم موجودگی کا قابلِ اعتماد اندازہ لگایا جا سکے۔ اگلا قدم: تصویر دوبارہ لیں۔ اگر دوسری تصویر بھی واضح نہ ہو تو مریض کو آنکھوں کے ماہر کے پاس بھیجیں۔",
        "uz": "Rasm kasallikni istisno qilish uchun yetarlicha aniq emas. Tavsiya etilgan keyingi qadam: rasmni qayta oling; hali ham noaniq boʻlsa, koʻz mutaxassisiga yuboring.",
    },
    "RESCREEN": {
        "en": "No referable diabetic retinopathy was detected on this photo; this does not rule out eye disease. Recommended next step: continue routine eye check-ups as the doctor advises (usually every 12 months).",
        "hi": "इस तस्वीर में ऐसी डायबिटिक रेटिनोपैथी नहीं दिखी जिसके लिए विशेषज्ञ के पास भेजना ज़रूरी हो; इससे आँख की बीमारी की संभावना खत्म नहीं होती। अनुशंसित अगला कदम: डॉक्टर की सलाह के अनुसार नियमित आँखों की जाँच जारी रखें (आमतौर पर हर 12 महीने)।",
        "ur": "اس تصویر میں ذیابیطس سے متعلق ایسی آنکھ کی بیماری کی نشاندہی نہیں ہوئی جس پر ریفرل کی حد پوری ہوتی ہو۔ اگلا قدم: ڈاکٹر کی ہدایت کے مطابق معمول کے معائنے جاری رکھیں۔",
        "uz": "Bu rasmda yuborishni talab qiluvchi diabetik retinopatiya aniqlanmadi; bu koʻz kasalligini istisno qilmaydi. Tavsiya etilgan keyingi qadam: shifokor maslahatiga koʻra muntazam koʻz tekshiruvlarini davom ettiring (odatda har 12 oyda).",
    },
    "DISCLAIMER": {
        "en": "Prototype for a student hackathon. Not medical advice and not a diagnosis.",
        "hi": "यह छात्रों के हैकाथॉन का एक प्रोटोटाइप है। यह चिकित्सीय सलाह या निदान नहीं है।",
        "ur": "یہ طلبہ کے ہیکاتھون کے لیے تیار کیا گیا ایک تجرباتی نمونہ ہے۔ یہ طبی مشورہ یا حتمی تشخیص فراہم نہیں کرتا۔",
        "uz": "Talabalar xakatoni uchun prototip. Tibbiy maslahat yoki tashxis emas.",
    },
}


def slip_html(decision: str, langs=DEFAULT_LANGS) -> str:
    """The printable note for the doctor and patient, one block per language (Urdu right-to-left)."""
    blocks = []
    for code in langs:
        name, direction = LANGS[code]["name"], LANGS[code]["dir"]
        blocks.append(f"<div dir='{direction}' lang='{code}' style='margin:6px 0;line-height:1.5'>"
                      f"<b>{name}:</b> {REPORT[decision][code]}<br>"
                      f"<span style='opacity:0.7;font-size:0.9em'>{REPORT['DISCLAIMER'][code]}</span></div>")
    return "".join(blocks)


@dataclass
class Frozen:
    """Everything the clinic tool needs, fixed before the test sets were scored."""
    arch: str
    size: int
    ref_threshold: float
    poor_threshold: float
    target_sensitivity: float


In [ ]:
%%writefile run_dr.py
"""End-to-end experiment: APTOS 2019 (India) for training, DeepDRiD (China) as the untouched external test.

Used by the notebook; can also run from the command line:
  python run_dr.py --aptos-csv .../train.csv --aptos-dir .../train_images --ddr-root DeepDRiD --out results
"""
from __future__ import annotations

import argparse
import json
import os
import pickle
import time
from dataclasses import asdict, dataclass

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import drscreen as ds

INK, MUTED, GRID = "#1f2328", "#6a737d", "#e1e4e8"
C_MAIN, C_EXT, C_PROBE, C_COLOR = "#2f6fdb", "#7a3fd1", "#d9822b", "#8c959f"
C_GRAYBAR = "#6e7781"   # the "without" comparison: deliberately neutral


@dataclass
class RunConfig:
    aptos_csv: str
    aptos_dir: str
    ddr_root: str | None = None
    oiaddr_root: str | None = None       # v2 only: OIA-DDR (Li et al. 2019) photos added to training and validation
    out: str = "results"
    cache: str = "cache"
    size: int = 384
    arch: str = "efficientnet_b0"
    pretrained: bool = True
    epochs: int = 12
    lr: float = 3e-4
    batch_size: int = 32
    workers: int = 2
    target_sensitivity: float = 0.90     # fixed before any result was seen
    poor_threshold: float = 0.50         # balanced logistic regression, default cut
    dup_max_dist: int = 10
    seed: int = 0
    max_images: int | None = None        # for quick code checks only


# ------------------------------------------------------------------ data loading
def load_aptos(csv, img_dir):
    df = pd.read_csv(csv)
    ext = ".png"
    if df["id_code"].astype(str).str.contains(r"\.").any():
        ext = ""
    df["path"] = [os.path.join(img_dir, f"{i}{ext}") for i in df["id_code"].astype(str)]
    missing = (~df["path"].map(os.path.exists)).sum()
    if missing:
        alt = [p[:-4] + ".jpg" for p in df["path"]] if ext == ".png" else df["path"]
        df["path"] = [a if not os.path.exists(p) and os.path.exists(a) else p for p, a in zip(df["path"], alt)]
        missing = (~df["path"].map(os.path.exists)).sum()
    if missing:
        raise FileNotFoundError(f"{missing} APTOS images not found under {img_dir}")
    return df.rename(columns={"diagnosis": "grade"})[["id_code", "path", "grade"]]


IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")


def load_oiaddr(root, log=print):
    """OIA-DDR grading photos (Li et al. 2019; 147 hospitals, 42 camera models; CC BY-NC-SA 4.0), from whatever folder
    layout the copy has. Labels are read from, in order: '<image> <grade>' text files, CSV files with an image column
    and a grade column, or class folders named 0 to 5. Grade 5 (ungradable) is dropped. Our own split is used later."""
    import re
    index = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                index.setdefault(f, os.path.join(d, f))
                index.setdefault(os.path.splitext(f)[0], os.path.join(d, f))
    labels = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            if f.lower().endswith(".txt"):
                for line in open(p, errors="ignore"):
                    m = re.match(r"^\s*(\S+\.(?:jpg|jpeg|png|tif|tiff))\s+(\d)\s*$", line, re.I)
                    if m:
                        labels.setdefault(os.path.basename(m.group(1)), int(m.group(2)))
            elif f.lower().endswith(".csv"):
                try:
                    c = pd.read_csv(p)
                except Exception:  # noqa: BLE001
                    continue
                gcol = next((k for k in c.columns if str(k).lower() in ("diagnosis", "label", "grade", "level", "dr_grade", "class")), None)
                icol = next((k for k in c.columns if c[k].astype(str).map(lambda v: v in index).mean() > 0.5), None)
                if gcol is not None and icol is not None:
                    for i, g in zip(c[icol].astype(str), c[gcol]):
                        if str(g).strip().isdigit():
                            labels.setdefault(os.path.basename(index[i]) if i in index else i, int(g))
    how = "label files"
    if not labels:          # class folders: .../0/x.jpg ... .../5/y.jpg
        how = "class folders"
        for k, p in index.items():
            parent = os.path.basename(os.path.dirname(p))
            if parent.isdigit() and os.path.splitext(k)[1]:
                labels.setdefault(k, int(parent))
    rows = [dict(id_code=k, path=index[k], grade=g) for k, g in labels.items() if k in index]
    df = pd.DataFrame(rows, columns=["id_code", "path", "grade"]).drop_duplicates("path")
    n_all = len(df)
    df = df[df.grade.between(0, 4)].reset_index(drop=True)
    log(f"OIA-DDR: {n_all} labelled photos found under {root} (from {how}); {n_all - len(df)} ungradable dropped; "
        f"grades {df.grade.value_counts().sort_index().to_dict()}")
    if len(df) < 100:
        raise FileNotFoundError(f"Fewer than 100 labelled OIA-DDR photos found under {root}: check the dataset is attached")
    return df


def load_deepdrid(root, part):
    """part: 'training' or 'validation'. Returns one row per photo with eye-level grade, quality and source."""
    folder = os.path.join(root, "regular_fundus_images", f"regular-fundus-{part}")
    lab = pd.read_csv(os.path.join(folder, f"regular-fundus-{part}.csv"))
    src_csv = os.path.join(folder, f"regular-fundus-source-{part}.csv")
    if os.path.exists(src_csv):
        lab = lab.merge(pd.read_csv(src_csv)[["image_id", "Source"]], on="image_id", how="left")
    else:
        lab["Source"] = "unknown"
    lab["grade"] = lab["left_eye_DR_Level"].fillna(lab["right_eye_DR_Level"]).astype(int)
    lab["poor_quality"] = (lab["Overall quality"] == 0).astype(int)
    lab["path"] = [os.path.join(folder, "Images", str(p), f"{i}.jpg") for p, i in zip(lab["patient_id"], lab["image_id"])]
    lab = lab[lab["path"].map(os.path.exists)].reset_index(drop=True)
    lab["part"] = part
    return lab[["patient_id", "image_id", "path", "grade", "poor_quality", "Source", "part"]]


# ------------------------------------------------------------------ figures
def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def fig_roc(curves, path):
    from sklearn.metrics import roc_curve
    fig, ax = plt.subplots(figsize=(6.4, 5.6), dpi=150)
    _style(ax)
    for label, y, p, thr, color in curves:
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr * 100, tpr * 100, color=color, lw=2, label=label)
        pred = p >= thr
        sens = (pred & (y == 1)).sum() / max(1, (y == 1).sum())
        fpr_pt = (pred & (y == 0)).sum() / max(1, (y == 0).sum())
        ax.plot([fpr_pt * 100], [sens * 100], "o", ms=7, color=color, markeredgecolor="white", markeredgewidth=1.5)
    ax.plot([0, 100], [0, 100], color=GRID, lw=1, ls="--")
    ax.set_xlabel("False alarms: healthy eyes sent to a specialist (%)", color=INK)
    ax.set_ylabel("Referable eyes caught (%)", color=INK)
    ax.set_title("Referable diabetic retinopathy: catches vs false alarms\n(dots = threshold frozen on validation)",
                 loc="left", fontsize=10.5, color=INK)
    ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def fig_gallery(rows, path, title, size=160):
    rows = rows[:8]
    if len(rows) == 0:
        return
    fig, axes = plt.subplots(1, len(rows), figsize=(2.0 * len(rows), 2.5), dpi=150)
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, rows):
        ax.imshow(Image.open(r["path"]).convert("RGB").resize((size, size)))
        ax.set_title(r["caption"], fontsize=7, color=INK)
        ax.axis("off")
    fig.suptitle(title, fontsize=10, color=INK, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def stress_test(model, gate, ref_thr, poor_thr, sets, cfg, log=print) -> pd.DataFrame:
    """Score the frozen tool on deliberately worse copies of held-out photos. Nothing is re-tuned."""
    from functools import partial
    rows = []
    for set_name, d in sets:
        y = (d["grade"].to_numpy() >= ds.REFERABLE_FROM).astype(int)
        for kind, strength, level in ds.stress_conditions():
            fn = None if kind == "clean" else partial(ds.degrade, kind=kind, level=level)
            P, E = ds.predict(model, ds.make_dataset(list(d["cached"]), None, cfg.size, False, degrade_fn=fn),
                              cfg.batch_size, workers=cfg.workers, with_embeddings=True)
            p_ref = ds.referable_prob(P)
            p_poor = gate.predict_proba(E)[:, 1] if gate is not None else np.zeros(len(d))
            rep = ds.binary_report(y, p_ref, ref_thr)
            w = ds.workflow_report(d["grade"], ds.decide(p_ref, p_poor, ref_thr, poor_thr))
            w0 = ds.workflow_report(d["grade"], ds.decide(p_ref, np.zeros(len(d)), ref_thr, poor_thr))
            rows.append(dict(photos=set_name, change=kind, strength=strength, level=level, n=len(d), referable=int(y.sum()),
                             AUC=rep["AUC"], sensitivity=rep["sensitivity"], specificity=rep["specificity"],
                             retake_share=w["RETAKE"] / max(1, w["photos"]),
                             referable_all_clear_with_check=w["referable_given_all_clear"],
                             referable_all_clear_without_check=w0["referable_given_all_clear"],
                             grade3_4_all_clear_with_check=w["sight_threatening_given_all_clear"],
                             grade3_4_all_clear_without_check=w0["sight_threatening_given_all_clear"]))
            log(f"  stress {set_name:>8} {kind:>15} {strength:>6}: sensitivity {rep['sensitivity']:.3f}, "
                f"specificity {rep['specificity']:.3f}, referable all-clears {w['referable_given_all_clear']} "
                f"with check / {w0['referable_given_all_clear']} without")
    return pd.DataFrame(rows)


def fig_stress(st: pd.DataFrame, path):
    """Per photo set: sensitivity and specificity (left), and referable eyes given an all-clear
    with vs without the quality check (right). Horizontal bars so every condition label is readable."""
    sets = list(dict.fromkeys(st["photos"]))
    fig, axes = plt.subplots(len(sets), 2, figsize=(12, 4.2 * len(sets)), dpi=150, squeeze=False)
    for i, name in enumerate(sets):
        d = st[st.photos == name].reset_index(drop=True)
        labels = ["original" if c == "clean" else f"{c.replace('_', ' ')}, {s}" for c, s in zip(d.change, d.strength)]
        y = np.arange(len(d))[::-1]
        h = 0.36
        ax = axes[i, 0]
        _style(ax)
        ax.grid(axis="y", visible=False)
        ax.barh(y + h / 2, d.sensitivity * 100, height=h, color=C_MAIN, label="sensitivity (referable eyes caught)")
        ax.barh(y - h / 2, d.specificity * 100, height=h, color=C_PROBE, label="specificity (healthy eyes cleared)")
        ax.set_yticks(y, labels, fontsize=8)
        ax.set_xlim(0, 100)
        ax.set_xlabel("%", color=MUTED, fontsize=9)
        ax.set_title(f"{name}: accuracy at the frozen threshold", loc="left", fontsize=10, color=INK, pad=24)
        ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
        ax = axes[i, 1]
        _style(ax)
        ax.grid(axis="y", visible=False)
        a, b = d.referable_all_clear_without_check, d.referable_all_clear_with_check
        ax.barh(y + h / 2, a, height=h, color=C_GRAYBAR, label="without quality check")
        ax.barh(y - h / 2, b, height=h, color=C_MAIN, label="with quality check")
        top = max(a.max(), b.max(), 1)
        for yy, va, vb in zip(y, a, b):
            ax.text(va + top * 0.01, yy + h / 2, str(int(va)), va="center", fontsize=7, color=MUTED)
            ax.text(vb + top * 0.01, yy - h / 2, str(int(vb)), va="center", fontsize=7, color=INK)
        ax.set_xlim(0, top * 1.12)
        ax.set_yticks(y, labels, fontsize=8)
        ax.set_xlabel(f"referable eyes given an all-clear (of {int(d.referable.iloc[0])})", color=MUTED, fontsize=9)
        ax.set_title(f"{name}: false all-clears", loc="left", fontsize=10, color=INK, pad=24)
        ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def fig_stress_examples(photo_path, size, path):
    conds = ds.stress_conditions()
    base = Image.open(photo_path).convert("RGB").resize((size, size))
    fig, axes = plt.subplots(1, len(conds), figsize=(1.7 * len(conds), 2.2), dpi=150)
    for ax, (kind, strength, level) in zip(axes, conds):
        ax.imshow(ds.degrade(base, kind, level))
        ax.set_title("original" if kind == "clean" else f"{kind.replace('_', ' ')}\n{strength}", fontsize=7, color=INK)
        ax.axis("off")
    fig.suptitle("The same photo under each stress-test change", fontsize=9, color=INK, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


# ------------------------------------------------------------------ main experiment
def run(cfg: RunConfig, log=print) -> dict:
    import torch
    t0 = time.time()
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    os.makedirs(cfg.out, exist_ok=True)
    out = {"config": asdict(cfg)}

    # 1. APTOS: cache, find near-duplicates, split by duplicate group
    ap = load_aptos(cfg.aptos_csv, cfg.aptos_dir)
    if cfg.max_images:
        ap = pd.concat([d.head(max(2, cfg.max_images // 5)) for _, d in ap.groupby("grade")]).reset_index(drop=True)
    ap["cached"] = ds.cache_images(list(ap["path"]), os.path.join(cfg.cache, f"aptos_{cfg.size}"), cfg.size, max(1, cfg.workers))
    groups, pairs = ds.duplicate_groups(list(ap["cached"]), cfg.dup_max_dist)
    ap["dup_group"] = groups
    conflicts = sum(1 for i, j, _ in pairs if ap.grade[i] != ap.grade[j])
    out["duplicates"] = dict(pairs=len(pairs), pairs_with_different_grades=conflicts,
                             images_in_groups=int((pd.Series(groups).value_counts() > 1).sum()))
    log(f"Near-duplicate pairs: {len(pairs)} ({conflicts} with different grades). Kept together when splitting.")
    ap["split"] = ds.group_split(ap["grade"].to_numpy(), groups, cfg.seed)
    ap.to_csv(os.path.join(cfg.out, "aptos_split.csv"), index=False)
    log(ap.groupby("split")["grade"].value_counts().unstack(fill_value=0).to_string())
    tr, va, te = (ap[ap.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    ybin = lambda d: (d["grade"].to_numpy() >= ds.REFERABLE_FROM).astype(int)
    ox = None
    if cfg.oiaddr_root:      # v2: more cameras in training and validation; APTOS test and DeepDRiD unchanged
        ox = load_oiaddr(cfg.oiaddr_root, log)
        if cfg.max_images:
            ox = pd.concat([d.head(max(2, cfg.max_images // 5)) for _, d in ox.groupby("grade")]).reset_index(drop=True)
        ox["cached"] = ds.cache_images(list(ox["path"]), os.path.join(cfg.cache, f"oiaddr_{cfg.size}"), cfg.size, max(1, cfg.workers))
        g2, p2 = ds.duplicate_groups(list(ox["cached"]), cfg.dup_max_dist)
        ox["dup_group"] = g2
        ox["split"] = ds.group_split(ox["grade"].to_numpy(), g2, cfg.seed)
        ox.to_csv(os.path.join(cfg.out, "oiaddr_split.csv"), index=False)
        out["oiaddr"] = dict(photos=len(ox), near_duplicate_pairs=len(p2),
                             split=ox.split.value_counts().to_dict(), grades=ox.grade.value_counts().sort_index().to_dict())
        log("OIA-DDR split:\n" + ox.groupby("split")["grade"].value_counts().unstack(fill_value=0).to_string())
        tr = pd.concat([tr.assign(source="APTOS"), ox[ox.split == "train"].assign(source="OIA-DDR")], ignore_index=True)
        va = pd.concat([va.assign(source="APTOS"), ox[ox.split == "val"].assign(source="OIA-DDR")], ignore_index=True)
        log(f"Training photos: {len(tr)} (APTOS + OIA-DDR); validation photos: {len(va)} (both sources)")

    # 2. Baselines
    results = {}
    Fc = {s: np.stack([ds.color_features(p) for p in d["cached"]]) for s, d in (("train", tr), ("val", va), ("test", te))}
    m = ds.fit_logreg(Fc["train"], ybin(tr))
    pv, pt = m.predict_proba(Fc["val"])[:, 1], m.predict_proba(Fc["test"])[:, 1]
    thr = ds.threshold_for_sensitivity(ybin(va), pv, cfg.target_sensitivity)
    results["colour_baseline"] = ds.binary_report(ybin(te), pt, thr, "Colour histogram + logistic regression")
    curves_te = [("Colour baseline (APTOS test)", ybin(te), pt, thr, C_COLOR)]

    probe_model = ds.build_model(cfg.arch, cfg.pretrained)
    emb = {}
    for s, d in (("train", tr), ("val", va), ("test", te)):
        _, emb[s] = ds.predict(probe_model, ds.make_dataset(list(d["cached"]), d["grade"].to_numpy(), cfg.size, False),
                               cfg.batch_size, workers=cfg.workers, with_embeddings=True)
    m = ds.fit_logreg(emb["train"], ybin(tr), C=0.1)
    pv, pt = m.predict_proba(emb["val"])[:, 1], m.predict_proba(emb["test"])[:, 1]
    thr = ds.threshold_for_sensitivity(ybin(va), pv, cfg.target_sensitivity)
    results["linear_probe"] = ds.binary_report(ybin(te), pt, thr, "Pretrained features, not fine-tuned + logistic regression")
    curves_te.append(("Linear probe (APTOS test)", ybin(te), pt, thr, C_PROBE))
    del probe_model
    for k in ("colour_baseline", "linear_probe"):
        log(f"{results[k]['model']}: AUC {results[k]['AUC']:.3f}, sensitivity {results[k]['sensitivity']:.3f}, specificity {results[k]['specificity']:.3f}")

    # 3. Fine-tuned model; threshold frozen on validation
    model = ds.build_model(cfg.arch, cfg.pretrained)
    hist = ds.train_model(model, ds.make_dataset(list(tr["cached"]), tr["grade"].to_numpy(), cfg.size, True),
                          ds.make_dataset(list(va["cached"]), va["grade"].to_numpy(), cfg.size, False),
                          va["grade"].to_numpy(), cfg.epochs, cfg.lr, cfg.batch_size, workers=cfg.workers,
                          out_path=os.path.join(cfg.out, "model_best.pt"), log=log)
    hist.to_csv(os.path.join(cfg.out, "training_history.csv"), index=False)
    Pv, Ev = ds.predict(model, ds.make_dataset(list(va["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers, with_embeddings=True)
    ref_thr = ds.threshold_for_sensitivity(ybin(va), ds.referable_prob(Pv), cfg.target_sensitivity)
    out["frozen"] = asdict(ds.Frozen(cfg.arch, cfg.size, ref_thr, cfg.poor_threshold, cfg.target_sensitivity))
    log(f"FROZEN referral threshold {ref_thr:.3f} (validation sensitivity target {cfg.target_sensitivity:.0%})")

    # 4. APTOS held-out test (scored once, after freezing)
    Pt = ds.predict(model, ds.make_dataset(list(te["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers)
    pt = ds.referable_prob(Pt)
    results["fine_tuned"] = ds.binary_report(ybin(te), pt, ref_thr, f"Fine-tuned {cfg.arch}")
    out["aptos_grades"] = ds.grade_report(te["grade"], Pt)
    out["aptos_flagged_by_grade"] = ds.flagged_by_grade(te["grade"], pt, ref_thr)
    curves_te.append((f"Fine-tuned (APTOS test, India)", ybin(te), pt, ref_thr, C_MAIN))
    te = te.assign(p_referable=pt, pred_grade=Pt.argmax(1))
    te.to_csv(os.path.join(cfg.out, "aptos_test_predictions.csv"), index=False)
    log(f"APTOS test: AUC {results['fine_tuned']['AUC']:.3f}, sensitivity {results['fine_tuned']['sensitivity']:.3f}, "
        f"specificity {results['fine_tuned']['specificity']:.3f}, missed {results['fine_tuned']['missed_FN']}")

    if ox is not None:      # OIA-DDR held-out test: new photos from the same hospitals as its training part
        ot = ox[ox.split == "test"].reset_index(drop=True)
        Po = ds.predict(model, ds.make_dataset(list(ot["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers)
        ot["p_referable"] = ds.referable_prob(Po)
        results["oiaddr_test"] = ds.binary_report(ybin(ot), ot["p_referable"].to_numpy(), ref_thr, "Fine-tuned, OIA-DDR test (China)")
        ot.drop(columns=["path", "cached"]).to_csv(os.path.join(cfg.out, "oiaddr_test_predictions.csv"), index=False)
        curves_te.append(("Fine-tuned (OIA-DDR test, China)", ybin(ot), ot["p_referable"].to_numpy(), ref_thr, C_PROBE))
        log(f"OIA-DDR test: AUC {results['oiaddr_test']['AUC']:.3f}, sensitivity {results['oiaddr_test']['sensitivity']:.3f}, "
            f"specificity {results['oiaddr_test']['specificity']:.3f}")

    # 5. External: DeepDRiD (never used to train the DR model or to set its threshold)
    if cfg.ddr_root:
        dd = pd.concat([load_deepdrid(cfg.ddr_root, "training"), load_deepdrid(cfg.ddr_root, "validation")], ignore_index=True)
        dd["cached"] = ds.cache_images(list(dd["path"]), os.path.join(cfg.cache, f"ddr_{cfg.size}"), cfg.size, max(1, cfg.workers))
        Pd, Ed = ds.predict(model, ds.make_dataset(list(dd["cached"]), None, cfg.size, False), cfg.batch_size,
                            workers=cfg.workers, with_embeddings=True)
        dd["p_referable"] = ds.referable_prob(Pd)
        dd["near_copy_in_training"] = False
        if ox is not None:      # DeepDRiD must stay unseen: look for near-copies among the OIA-DDR photos used
            used = ox[ox.split.isin(["train", "val"])]
            Hd = np.stack([ds.dhash(p) for p in dd["cached"]]).astype(np.float32)
            Ho = np.stack([ds.dhash(p) for p in used["cached"]]).astype(np.float32)
            dist = Hd.shape[1] - (Hd @ Ho.T + (1 - Hd) @ (1 - Ho).T)
            dd["near_copy_in_training"] = (dist <= cfg.dup_max_dist).any(1)
            out["deepdrid_near_copies_in_oiaddr"] = int(dd["near_copy_in_training"].sum())
            log(f"DeepDRiD photos with a near-copy among the OIA-DDR photos used: {out['deepdrid_near_copies_in_oiaddr']}")
        graded = dd["grade"] <= 4
        g = dd[graded]
        rep = ds.binary_report(ybin(g), g["p_referable"].to_numpy(), ref_thr, "Fine-tuned, external DeepDRiD (China)")
        rep["AUC_CI_by_patient"] = _patient_bootstrap_auc(ybin(g), g["p_referable"].to_numpy(), g["patient_id"].to_numpy())
        results["external_all"] = rep
        if dd["near_copy_in_training"].any():
            gc = g[~g["near_copy_in_training"]]
            results["external_no_near_copies"] = ds.binary_report(ybin(gc), gc["p_referable"].to_numpy(), ref_thr,
                                                                  "External DeepDRiD, near-copies of training photos removed")
        for q, name in ((0, "good"), (1, "poor")):
            gq = g[g.poor_quality == q]
            if len(gq) and len(set(ybin(gq))) > 1:
                results[f"external_{name}_photos"] = ds.binary_report(ybin(gq), gq["p_referable"].to_numpy(), ref_thr,
                                                                       f"External, {name}-quality photos")
        curves_te.append(("Fine-tuned (DeepDRiD, China, external)", ybin(g), g["p_referable"].to_numpy(), ref_thr, C_EXT))
        out["external_by_source"] = [dict(source=s, photos=len(d), **{k: v for k, v in ds.binary_report(
            ybin(d), d["p_referable"].to_numpy(), ref_thr).items() if k in ("sensitivity", "specificity", "missed_FN")})
            for s, d in g.groupby("Source") if len(d) >= 20 and len(set(ybin(d))) > 1]

        # 6. Quality gate: trained on DeepDRiD training photos' quality labels, tested on DeepDRiD validation
        trq, teq = dd.part.to_numpy() == "training", dd.part.to_numpy() == "validation"
        gate = ds.fit_logreg(Ed[trq], dd.loc[trq, "poor_quality"].to_numpy(), C=0.1)
        dd["p_poor"] = gate.predict_proba(Ed)[:, 1]
        from sklearn.metrics import roc_auc_score
        v = dd[teq]
        out["quality_gate"] = dict(AUC_detecting_poor_photos=float(roc_auc_score(v.poor_quality, v.p_poor)),
                                   flagged_share=float((v.p_poor >= cfg.poor_threshold).mean()),
                                   poor_photos_flagged=float((v.p_poor[v.poor_quality == 1] >= cfg.poor_threshold).mean()),
                                   good_photos_flagged=float((v.p_poor[v.poor_quality == 0] >= cfg.poor_threshold).mean()))
        vg = v[v.grade <= 4]
        with_gate = ds.decide(vg.p_referable, vg.p_poor, ref_thr, cfg.poor_threshold)
        no_gate = ds.decide(vg.p_referable, np.zeros(len(vg)), ref_thr, cfg.poor_threshold)
        out["workflow"] = dict(with_quality_gate=ds.workflow_report(vg.grade, with_gate),
                               without_quality_gate=ds.workflow_report(vg.grade, no_gate),
                               ungradable_photos=int((v.grade == 5).sum()),
                               ungradable_sent_to_retake_or_refer=int((ds.decide(v[v.grade == 5].p_referable, v[v.grade == 5].p_poor,
                                                                                ref_thr, cfg.poor_threshold) != "RESCREEN").sum()))
        dd["decision"] = ds.decide(dd.p_referable, dd.p_poor, ref_thr, cfg.poor_threshold)
        dd.to_csv(os.path.join(cfg.out, "deepdrid_predictions.csv"), index=False)
        with open(os.path.join(cfg.out, "quality_gate.pkl"), "wb") as f:
            pickle.dump(gate, f)
        log(f"External DeepDRiD: AUC {rep['AUC']:.3f}, sensitivity {rep['sensitivity']:.3f}, specificity {rep['specificity']:.3f}")
        log(f"Workflow on DeepDRiD validation: {out['workflow']}")

    # 7. Tables and figures
    summary = pd.DataFrame([{k: (f"{v:.3f}" if isinstance(v, float) else v) for k, v in r.items()
                             if k in ("model", "n", "referable", "AUC", "sensitivity", "specificity", "PPV", "NPV",
                                      "missed_FN", "false_alarms_FP", "referred_share")} for r in results.values()])
    summary.to_csv(os.path.join(cfg.out, "summary.csv"), index=False)
    fig_roc(curves_te, os.path.join(cfg.out, "roc.png"))
    misses = te[(te.grade >= ds.REFERABLE_FROM) & (te.p_referable < ref_thr)].sort_values("p_referable")
    fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}")
                 for r in misses.itertuples()], os.path.join(cfg.out, "missed_cases.png"),
                "Referable eyes the model missed (APTOS test) — look for why")
    if cfg.ddr_root:   # external misses: DeepDRiD photos (CC BY-SA 4.0), safe to show publicly with attribution
        mx = dd[(dd.grade >= ds.REFERABLE_FROM) & (dd.decision == "RESCREEN")].sort_values("p_referable")
        fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}, "
                                                 f"{'poor' if r.poor_quality else 'good'} photo")
                     for r in mx.itertuples()], os.path.join(cfg.out, "missed_external.png"),
                    "Referable eyes given an all-clear on external DeepDRiD photos (CC BY-SA 4.0)")
    fps = te[(te.grade < ds.REFERABLE_FROM) & (te.p_referable >= ref_thr)].sort_values("p_referable", ascending=False)
    fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}")
                 for r in fps.itertuples()], os.path.join(cfg.out, "false_alarms.png"),
                "Eyes sent to a specialist that did not need it (APTOS test)")
    # 8. Speed on the CPU alone (no graphics card), from the original photo file to a decision.
    #    Optional: if it fails, the run still finishes and the write-up says speed was not measured.
    try:
        if cfg.ddr_root:
            bench_paths, bench_gate = list(dd.loc[dd.part == "validation", "path"]), gate
        else:
            bench_paths, bench_gate = list(te["path"]), None
        cs = ds.cpu_benchmark(model, bench_gate, bench_paths, cfg.size, ref_thr, cfg.poor_threshold)
        cs["model_file_MB"] = os.path.getsize(os.path.join(cfg.out, "model_best.pt")) / 1e6
        out["cpu_speed"] = cs
        log(f"CPU only: {cs['median_seconds_per_photo']:.2f} s per photo (median of {cs['photos_timed']}), "
            f"heat map {cs['heatmap_median_seconds']:.2f} s, {cs['cpu_threads']} threads, {cs['cpu']}, "
            f"model file {cs['model_file_MB']:.1f} MB")
    except Exception as e:  # never lose a finished run over a speed test
        out["cpu_speed"] = dict(error=repr(e))
        log(f"CPU speed test skipped: {e!r}")

    # 9. Stress test: the frozen tool on deliberately worse copies of held-out photos (planned before results).
    #    Optional: if it fails, the run still finishes and the write-up says it was not run.
    try:
        sets = [("APTOS test", te)]
        if cfg.ddr_root:
            sets.append(("DeepDRiD validation", dd[(dd.part == "validation") & (dd.grade <= 4)].reset_index(drop=True)))
        log("Stress test (worse photos):")
        st = stress_test(model, gate if cfg.ddr_root else None, ref_thr, cfg.poor_threshold, sets, cfg, log)
        st.to_csv(os.path.join(cfg.out, "stress_test.csv"), index=False)
        out["stress_test"] = st.to_dict("records")
        fig_stress(st, os.path.join(cfg.out, "stress_test.png"))
        ex = sets[-1][1]
        ex = ex[ex.grade >= ds.REFERABLE_FROM] if (ex.grade >= ds.REFERABLE_FROM).any() else ex
        fig_stress_examples(ex["cached"].iloc[0], cfg.size, os.path.join(cfg.out, "stress_examples.png"))
    except Exception as e:  # never lose a finished run over the stress test
        out["stress_test_error"] = repr(e)
        log(f"Stress test skipped: {e!r}")

    out["results"] = results
    out["seconds"] = time.time() - t0
    with open(os.path.join(cfg.out, "results.json"), "w") as f:
        json.dump(out, f, indent=2, default=lambda o: o.to_dict("records") if isinstance(o, pd.DataFrame) else (o.tolist() if hasattr(o, "tolist") else str(o)))
    log(f"Done in {out['seconds'] / 60:.1f} min. Results in {cfg.out}/")
    out["summary"] = summary
    out["model"] = model
    return out


def _patient_bootstrap_auc(y, p, patients, n=1000, seed=0):
    """AUC interval resampling whole patients (each has up to 4 photos), so it is not falsely narrow."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    pats = np.unique(patients)
    idx = {q: np.where(patients == q)[0] for q in pats}
    vals = []
    for _ in range(n):
        pick = np.concatenate([idx[q] for q in rng.choice(pats, len(pats))])
        if len(set(y[pick])) > 1:
            vals.append(roc_auc_score(y[pick], p[pick]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


if __name__ == "__main__":
    a = argparse.ArgumentParser()
    a.add_argument("--aptos-csv", required=True)
    a.add_argument("--aptos-dir", required=True)
    a.add_argument("--ddr-root", default=None, help="DeepDRiD (external test)")
    a.add_argument("--oiaddr-root", default=None, help="v2 only: OIA-DDR photos added to training and validation")
    a.add_argument("--out", default="results")
    a.add_argument("--epochs", type=int, default=12)
    a.add_argument("--size", type=int, default=384)
    a.add_argument("--no-pretrained", action="store_true")
    a.add_argument("--max-images", type=int, default=None)
    a.add_argument("--workers", type=int, default=2)
    x = a.parse_args()
    r = run(RunConfig(aptos_csv=x.aptos_csv, aptos_dir=x.aptos_dir, ddr_root=x.ddr_root, oiaddr_root=x.oiaddr_root,
                      out=x.out, size=x.size, epochs=x.epochs, pretrained=not x.no_pretrained,
                      max_images=x.max_images, workers=x.workers))
    print(r["summary"].to_string(index=False))


In [ ]:
%%writefile app.py
"""Roshni demo (optional Gradio version; the main demo is the browser version in web/): upload a retinal photo,
get REFER / RETAKE / RESCREEN with a note in English and Urdu.

  python app.py --results results [--share]

Prototype for a student hackathon. Not medical advice and not a diagnosis.
"""
import argparse
import glob
import json
import os
import pickle

import gradio as gr
import numpy as np
import pandas as pd
import torch

import drscreen as ds

COLORS = {"REFER": "#b42318", "RETAKE": "#b54708", "RESCREEN": "#067647"}
TITLES = {"REFER": "Refer to an eye specialist", "RETAKE": "Retake the photo", "RESCREEN": "Rescreen in 12 months"}


def load_bundle(results_dir):
    with open(os.path.join(results_dir, "results.json")) as f:
        res = json.load(f)
    fr = res["frozen"]
    model = ds.build_model(fr["arch"], pretrained=False)
    model.load_state_dict(torch.load(os.path.join(results_dir, "model_best.pt"), map_location="cpu"))
    model.eval()
    gate = None
    gp = os.path.join(results_dir, "quality_gate.pkl")
    if os.path.exists(gp):
        with open(gp, "rb") as f:
            gate = pickle.load(f)
    summary = pd.read_csv(os.path.join(results_dir, "summary.csv")) if os.path.exists(os.path.join(results_dir, "summary.csv")) else None
    return model, gate, fr, summary, res


def build_app(results_dir: str, examples_dir: str | None = None) -> gr.Blocks:
    model, gate, fr, summary, res = load_bundle(results_dir)
    size, ref_thr, poor_thr = fr["size"], fr["ref_threshold"], fr["poor_threshold"]
    tf = ds.get_transforms(size, False)

    def screen(img, langs=ds.DEFAULT_LANGS):
        langs = [c for c in ds.shown_langs() if c in (langs or [])] or ["en"]
        if img is None:
            return None, "Upload a retinal photo first.", None, ""
        d, p_ref, p_poor, P, _ = ds.triage_one(model, gate, img, size, ref_thr, poor_thr, tf)
        P = P[None, :]
        grades = {f"{k} {g}": float(P[0, k]) for k, g in enumerate(ds.GRADES)}
        card = (f"<div style='border-left:6px solid {COLORS[d]};padding:10px 14px;border-radius:6px;"
                f"background:rgba(0,0,0,0.03)'><div style='font-size:22px;font-weight:700;color:{COLORS[d]}'>"
                f"{TITLES[d]}</div><div style='margin-top:6px'>Referable-disease score {p_ref:.2f} "
                f"(refer at {ref_thr:.2f} or above) · Poor-photo score "
                + (f"{p_poor:.2f} (retake at {poor_thr:.2f} or above)" if gate is not None else "not available")
                + "</div></div>")
        note = ("<div style='font-weight:600;margin-bottom:4px'>Note for the doctor and patient</div>"
                + ds.slip_html(d, langs))
        try:
            heat = ds.gradcam(model, img, size, device="cpu")
        except Exception:
            heat = None
        return heat, card, grades, note

    ex = []
    if examples_dir and os.path.isdir(examples_dir):
        ex = sorted(glob.glob(os.path.join(examples_dir, "*.jpg")) + glob.glob(os.path.join(examples_dir, "*.png")))[:8]

    with gr.Blocks(title="Roshni: diabetic eye screening") as demo:
        gr.Markdown("# Roshni: offline diabetic eye triage for primary-care diabetes clinics in Pakistan\n"
                    "For a clinic that can take retinal photos but has no eye specialist on site. Upload one photo taken at a "
                    "diabetes check-up; the tool suggests one next step for the doctor to confirm: **refer**, **retake the photo**, "
                    "or **rescreen in 12 months**. A photo judged too poor to read gets a retake instead of an all-clear, and never blocks a referral.  \n"
                    f"_Thresholds were frozen before testing (referral threshold set for {fr['target_sensitivity']:.0%} sensitivity on validation). "
                    "Prototype for a student hackathon; not medical advice and not a diagnosis._")
        with gr.Row():
            with gr.Column():
                inp = gr.Image(type="pil", label="Retinal photo")
                langs = gr.CheckboxGroup([(ds.LANGS[k]["name"], k) for k in ds.shown_langs()], value=list(ds.DEFAULT_LANGS),
                                         label="Languages on the note")
                btn = gr.Button("Screen this photo", variant="primary")
                if ex:
                    gr.Examples(ex, inp, label="Example photos")
            with gr.Column():
                card = gr.HTML()
                grades = gr.Label(num_top_classes=5, label="Estimated severity")
                note = gr.HTML()
                heat = gr.Image(label="Where the model looked (Grad-CAM)", interactive=False)
        btn.click(screen, [inp, langs], [heat, card, grades, note])
        if summary is not None:
            gr.Markdown("## How it did on photos it never saw")
            gr.Dataframe(summary, interactive=False, wrap=True)
        if "workflow" in res:
            w = res["workflow"]["with_quality_gate"]
            wo = res["workflow"]["without_quality_gate"]
            gr.Markdown(f"On {w['photos']} external photos (DeepDRiD validation): {w['REFER']} refer, {w['RETAKE']} retake, "
                        f"{w['RESCREEN']} rescreen. Referable photos given an all-clear: **{w['referable_given_all_clear']}** "
                        f"with the photo-quality check vs **{wo['referable_given_all_clear']}** without it.")
    return demo


if __name__ == "__main__":
    a = argparse.ArgumentParser()
    a.add_argument("--results", default="results")
    a.add_argument("--examples", default=None)
    a.add_argument("--share", action="store_true")
    x = a.parse_args()
    build_app(x.results, x.examples).launch(share=x.share)


In [ ]:
%%writefile fill_report.py
"""Fill README.md and VIDEO_SCRIPT.md from the run's own output files, so no number is typed by hand.

  python fill_report.py                      # reads results/, writes README.md and VIDEO_SCRIPT.md
  python fill_report.py --results quick_check --allow-quick --out-dir /tmp/test   # testing only

Tokens in the templates look like  [FROM_RESULTS_JSON: results.fine_tuned.AUC | .3f]
  path   dotted path into results.json (list items by number), or derived.* / table.* computed here
  format .3f  .2f  pct (91.2%)  pct0 (91%)  int  ci (0.812 to 0.861)  cipct  str
Text written by people comes from TEAM_INPUTS.md (one "## key" section each), never from results:
  [TEAM_TO_WRITE(key): instruction]          required; filled from the "## key" section, listed if still empty
  <!-- team:key -->default<!-- /team:key -->   optional; the "## key" section replaces the default text
If any results token cannot be filled, nothing is written.
"""
import argparse
import json
import math
import os
import re
import sys

import pandas as pd

TOKEN = re.compile(r"\[FROM_RESULTS_JSON:\s*([^\]|]+?)\s*(?:\|\s*([^\]]+?)\s*)?\]")
TEAM = re.compile(r"\[TEAM_TO_WRITE(?:\((\w+)\))?:[^\]]*\]")
BLOCK = re.compile(r"<!-- team:(\w+) -->(.*?)<!-- /team:\1 -->", re.S)
# word limits for the team's pieces (from the official template and our plan); over-long text is reported, not cut
LIMITS = {"problem": 80, "where_it_fails": 55, "what_we_learned": 50, "next_steps": 60, "adoption": 120,
          "video_miss_reason": 12}
HEADINGS = {"adoption": "**Who would test it first (a hypothesis, not a finding)**\n\n"}


def read_team(path):
    """{key: text} from TEAM_INPUTS.md. HTML comments (the instructions) and empty sections are ignored."""
    if not path or not os.path.exists(path):
        return {}
    text = re.sub(r"<!--.*?-->", "", open(path, encoding="utf-8").read(), flags=re.S)
    parts = re.split(r"^## (\w+)\s*$", text, flags=re.M)
    return {k: v.strip() for k, v in zip(parts[1::2], parts[2::2]) if v.strip()}


def apply_team(text, team):
    used = set()

    def block(m):
        key, default = m.group(1), m.group(2)
        if key in team:
            used.add(key)
            return HEADINGS.get(key, "") + team[key]
        return default

    def marker(m):
        key = m.group(1)
        if key and key in team:
            used.add(key)
            return team[key]
        return m.group(0)
    return TEAM.sub(marker, BLOCK.sub(block, text)), used


class Missing(Exception):
    pass


def get(data, path):
    cur = data
    for part in path.split("."):
        if isinstance(cur, list) and part.isdigit() and int(part) < len(cur):
            cur = cur[int(part)]
        elif isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            raise Missing(path)
    if cur is None or (isinstance(cur, float) and math.isnan(cur)):
        raise Missing(path + " (empty)")
    return cur


def fmt(v, spec):
    if spec in (None, "", "str"):
        return f"{v:.3f}" if isinstance(v, float) else str(v)
    if spec == "pct":
        return f"{100 * float(v):.1f}%"
    if spec == "pct0":
        return f"{100 * float(v):.0f}%"
    if spec == "int":
        return f"{int(round(float(v))):,}"
    if spec in ("ci", "cipct"):
        a, b = (float(x) for x in v)
        if any(math.isnan(x) for x in (a, b)):
            raise Missing("interval is empty")
        return f"{100 * a:.1f}% to {100 * b:.1f}%" if spec == "cipct" else f"{a:.3f} to {b:.3f}"
    return format(float(v), spec)


def table(rows, header):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    return "\n".join(out)


def build(res_dir):
    with open(os.path.join(res_dir, "results.json")) as f:
        R = json.load(f)
    r = R["results"]
    d = {}
    d["aptos_test_n"] = r["fine_tuned"]["n"]
    lc_p = os.path.join(res_dir, "local_calibration.json")
    if not os.path.exists(lc_p) and os.path.exists(os.path.join(res_dir, "deepdrid_predictions.csv")):
        import local_calibration
        local_calibration.run(res_dir)
    if os.path.exists(lc_p):
        lc = json.load(open(lc_p))
        s = next(x for x in lc["samples"] if x["patients"] == 50)
        d["lc_patients"], d["lc_photos"], d["lc_draws"] = s["patients"], s["photos_median"], s["draws"]
        d["lc_sens"], d["lc_spec"] = 100 * s["sensitivity"]["median"], 100 * s["specificity"]["median"]
        d["lc_sens_range"] = f"{100 * s['sensitivity']['p10']:.1f}% to {100 * s['sensitivity']['p90']:.1f}%"
        d["lc_spec_range"] = f"{100 * s['specificity']['p10']:.1f}% to {100 * s['specificity']['p90']:.1f}%"
        d["lc_allclear"], d["lc_sev_allclear"] = s["referable_given_all_clear"]["median"], s["sight_threatening_given_all_clear"]["median"]
        d["lc_refer"] = s["REFER"]["median"]
        d["lc_frozen_sens"], d["lc_frozen_spec"] = 100 * lc["frozen"]["sensitivity"], 100 * lc["frozen"]["specificity"]
        d["lc_frozen_allclear"], d["lc_frozen_refer"] = lc["frozen"]["referable_given_all_clear"], lc["frozen"]["REFER"]
        d["lc_eval_patients"] = lc["evaluation_patients"]
    tp_p = os.path.join(res_dir, "two_photo.json")
    if not os.path.exists(tp_p) and os.path.exists(os.path.join(res_dir, "deepdrid_predictions.csv")):
        import two_photo
        two_photo.run(res_dir)
    if os.path.exists(tp_p):
        tp = json.load(open(tp_p))
        one, two = tp["all"]["per_photo"], tp["all"]["per_eye_two_photos"]
        d["tp_one_sens"], d["tp_one_spec"] = 100 * one["sensitivity"], 100 * one["specificity"]
        d["tp_two_sens"], d["tp_two_spec"] = 100 * two["sensitivity"], 100 * two["specificity"]
        d["tp_two_ci"] = f"{100 * two['sensitivity_ci'][0]:.1f}% to {100 * two['sensitivity_ci'][1]:.1f}%"
        d["tp_eyes"], d["tp_patients"] = two["n"], tp["all"]["patients"]
        w = tp["validation"]["workflow_per_eye_two_photos"]
        d["tp_val_eyes"] = tp["validation"]["per_eye_two_photos"]["n"]
        d["tp_val_allclear"], d["tp_val_sev"], d["tp_val_retake"] = (w["referable_given_all_clear"],
                                                                       w["sight_threatening_given_all_clear"], w["RETAKE"])
    nr_p = os.path.join(res_dir, "non_retinal.json")
    if os.path.exists(nr_p):
        nr = json.load(open(nr_p))
        d["nr_n"], d["nr_allclear"] = nr["pictures"], nr["all_clears"]
        d["nr_allclear_words"] = "none" if nr["all_clears"] == 0 else f"{nr['all_clears']} of them"
        d["nr_retake"], d["nr_refer"] = nr["counts"]["RETAKE"], nr["counts"]["REFER"]
    if "seconds" in R:
        d["run_minutes"] = round(R["seconds"] / 60)
    sp_p = os.path.join(res_dir, "aptos_split.csv")
    if os.path.exists(sp_p):
        c = pd.read_csv(sp_p)["split"].value_counts()
        d["split_counts"] = (f"{c.get('train', 0):,} training, {c.get('val', 0):,} validation and "
                             f"{c.get('test', 0):,} test photos")
    d["aptos_test_referable"] = r["fine_tuned"]["referable"]
    hist_p = os.path.join(res_dir, "training_history.csv")
    if os.path.exists(hist_p):
        h = pd.read_csv(hist_p)
        d["epochs_run"] = len(h)
        d["best_epoch"] = int(h.loc[h["val_auc"].idxmax(), "epoch"])
    fl = {row["grade"][0]: row for row in R.get("aptos_flagged_by_grade", [])}
    if "3" in fl and "4" in fl:
        d["severe_sent"], d["severe_total"] = fl["3"]["sent_to_specialist"], fl["3"]["photos"]
        d["prolif_sent"], d["prolif_total"] = fl["4"]["sent_to_specialist"], fl["4"]["photos"]
    if "external_all" in r:
        d["ext_photos"] = r["external_all"]["n"]
        dd_p = os.path.join(res_dir, "deepdrid_predictions.csv")
        if os.path.exists(dd_p):
            dd = pd.read_csv(dd_p)
            d["ext_patients"] = int(dd.loc[dd["grade"] <= 4, "patient_id"].nunique())
    if "workflow" in R:
        w, wo = R["workflow"]["with_quality_gate"], R["workflow"]["without_quality_gate"]
        d["val_photos"] = w["photos"]
        a, b = wo["referable_given_all_clear"], w["referable_given_all_clear"]
        if b > a:
            raise RuntimeError("Quality check increased all-clears for referable eyes; that should be impossible. Check the run.")
        d["gate_sentence"] = (f"cut referable photos given an all-clear from {a} to {b}, at the cost of {w['RETAKE']} retakes"
                              if b < a else
                              f"did not change the number of referable photos given an all-clear ({b}), and asked for {w['RETAKE']} retakes")
        d["gate_short"] = (f"cut false all-clears from {a} to {b}, for {w['RETAKE']} retakes" if b < a else
                           f"did not reduce false all-clears ({b}), and asked for {w['RETAKE']} retakes")
    cs = R.get("cpu_speed") or {}
    if "median_seconds_per_photo" in cs:
        d["cpu_short"] = f"about {cs['median_seconds_per_photo']:.2f} s per photo on a CPU, with no graphics card"
        d["cpu_sentence"] = (f"on the CPU alone ({cs['cpu']}, {cs['cpu_threads']} threads, no graphics card), going from the "
                             f"original photo file to a decision took a median of {cs['median_seconds_per_photo']:.2f} s "
                             f"(slowest {cs['slowest_seconds_per_photo']:.2f} s, {cs['photos_timed']} photos); the heat map "
                             f"adds about {cs['heatmap_median_seconds']:.1f} s. The model file is {cs['model_file_MB']:.0f} MB "
                             f"({cs['parameters'] / 1e6:.1f} million parameters)")
    else:
        d["cpu_short"] = "speed on a CPU not measured in this run"
        d["cpu_sentence"] = "speed on a CPU was not measured in this run"
    st = pd.DataFrame(R.get("stress_test") or [])
    if len(st):
        name = "DeepDRiD validation" if (st.photos == "DeepDRiD validation").any() else "APTOS test"
        x = st[st.photos == name]
        clean, changed = x[x.change == "clean"].iloc[0], x[x.change != "clean"]
        worst = changed.loc[changed.sensitivity.idxmin()]
        wspec = changed.loc[changed.specificity.idxmin()]
        lab = lambda c: f"{c.change.replace('_', ' ')}, {c.strength}"
        w_lo, w_hi = int(changed.referable_all_clear_with_check.min()), int(changed.referable_all_clear_with_check.max())
        wo_hi = int(changed.referable_all_clear_without_check.max())
        d["stress_sentence"] = (
            f"on the {int(clean.n)} {name} photos, sensitivity was {100 * clean.sensitivity:.1f}% on the originals and "
            f"{100 * worst.sensitivity:.1f}% at worst ({lab(worst)}). With the quality check, referable photos given an all-clear "
            f"were {int(clean.referable_all_clear_with_check)} on the originals and between {w_lo} and {w_hi} across the "
            f"{len(changed)} changes, against up to {wo_hi} without it; the check sent up to "
            f"{100 * changed.retake_share.max():.0f}% of photos to a retake. Strong compression failed the other way: "
            f"specificity fell to {100 * wspec.specificity:.1f}% ({lab(wspec)}), so many photos without referable disease would be referred")
        d["stress_brief"] = (f"on external photos sensitivity fell to {100 * worst.sensitivity:.1f}% at worst ({lab(worst)}), yet with the quality check "
                             f"false all-clears stayed at {w_hi} or fewer, against up to {wo_hi} without it; strong compression "
                             f"cut specificity to {100 * wspec.specificity:.1f}%")
        d["stress_short"] = (f"On deliberately worse photos: at most {w_hi} false all-clears with the check, "
                             f"up to {wo_hi} without it.")
        T_stress = table([[("original" if c.change == "clean" else f"{c.change.replace('_', ' ')}, {c.strength}"), c.photos,
                           f"{100 * c.sensitivity:.1f}%", f"{100 * c.specificity:.1f}%", f"{100 * c.retake_share:.0f}%",
                           int(c.referable_all_clear_without_check), int(c.referable_all_clear_with_check)]
                          for c in st.itertuples()],
                         ["Photos", "Set", "Sensitivity", "Specificity", "Sent to retake",
                          "Referable all-clears, no check", "Referable all-clears, with check"])
    else:
        d["stress_sentence"] = "the stress test was not run in this run"
        d["stress_short"] = ""
        d["stress_brief"] = "not run in this run"
        T_stress = "_Stress test not run in this run._"
    ft, cb, lp = r["fine_tuned"], r["colour_baseline"], r["linear_probe"]
    d["baseline_sentence"] = (f"with an AUC of {ft['AUC']:.2f}, against {cb['AUC']:.2f} for a colour baseline "
                              f"and {lp['AUC']:.2f} for untuned features")

    T = {}
    rows = []
    for k in ("colour_baseline", "linear_probe", "fine_tuned", "external_all"):
        if k in r:
            x = r[k]
            rows.append([x["model"], f"{x['AUC']:.3f} ({x['AUC_CI'][0]:.3f} to {x['AUC_CI'][1]:.3f})",
                         f"{100 * x['sensitivity']:.1f}%", f"{100 * x['specificity']:.1f}%",
                         x["missed_FN"], x["false_alarms_FP"]])
    T["main"] = table(rows, ["Model", "AUC (95% CI)", "Sensitivity", "Specificity", "Missed (FN)", "False alarms (FP)"])
    if R.get("aptos_flagged_by_grade"):
        T["by_grade"] = table([[g["grade"], g["photos"], g["sent_to_specialist"], f"{100 * g['share_sent']:.0f}%"]
                               for g in R["aptos_flagged_by_grade"]], ["Grade", "Photos", "Sent to specialist", "Share sent"])
    if R.get("external_by_source"):
        T["by_source"] = table([[s["source"], s["photos"], f"{100 * s['sensitivity']:.1f}%", f"{100 * s['specificity']:.1f}%",
                                 s["missed_FN"]] for s in R["external_by_source"]],
                               ["Source", "Photos", "Sensitivity", "Specificity", "Missed (FN)"])
    if "workflow" in R:
        w, wo = R["workflow"]["with_quality_gate"], R["workflow"]["without_quality_gate"]
        names = [("REFER", "Refer"), ("RETAKE", "Retake"), ("RESCREEN", "Rescreen (all-clear)"),
                 ("referable_given_all_clear", "Referable photos given an all-clear"),
                 ("sight_threatening_given_all_clear", "Grade 3 to 4 photos given an all-clear")]
        T["workflow"] = table([[label, wo[k], w[k]] for k, label in names], ["", "Without quality check", "With quality check"])
    T["stress"] = T_stress
    if R.get("external_by_source"):
        ss = [100 * x["sensitivity"] for x in R["external_by_source"]]
        d["src_sens_min"], d["src_sens_max"] = min(ss), max(ss)
    # A real-world yardstick: LumineticsCore's pivotal trial pre-specified sensitivity above 85% and specificity
    # above 82.5% (Abramoff et al. 2018). Different reference standard, unit and population: a yardstick, not a test.
    BAR_SENS, BAR_SPEC = 0.85, 0.825
    bar = [("Held-out APTOS test, frozen cut-off", "photo", r["fine_tuned"]["sensitivity"], r["fine_tuned"]["specificity"]),
           ("Unseen hospitals (DeepDRiD), frozen cut-off", "photo", r["external_all"]["sensitivity"], r["external_all"]["specificity"])]
    if "lc_sens" in d:
        bar.append((f"Unseen hospitals, cut-off re-set on {d['lc_patients']} local patients (post-hoc)", "photo", d["lc_sens"] / 100, d["lc_spec"] / 100))
    if "tp_two_sens" in d:
        bar.append(("Unseen hospitals, two photos per eye (post-hoc)", "eye", d["tp_two_sens"] / 100, d["tp_two_spec"] / 100))
    T["bar"] = table([(n, u, f"{100 * se:.1f}%", f"{100 * sp:.1f}%",
                       "yes" if (se > BAR_SENS and sp > BAR_SPEC) else "no") for n, u, se, sp in bar],
                     ["Setting", "Unit", "Sensitivity", "Specificity", "Above both (85% and 82.5%)?"])
    d["bar_met_n"] = sum(se > BAR_SENS and sp > BAR_SPEC for _, _, se, sp in bar)
    d["bar_met_words"] = "None of our settings clears both yet" if d["bar_met_n"] == 0 else f"{d['bar_met_n']} of our settings clear both"
    v2p = os.path.join(os.path.dirname(os.path.abspath(res_dir)), "results_v2", "results.json")
    if os.path.exists(v2p):      # v2 (pre-registered, PREREGISTRATION_v2.md): reported next to v1 whatever the verdict
        V = json.load(open(v2p))
        R["v2"] = V
        ve, vw, va, v1e = V["results"]["external_all"], V["workflow"]["with_quality_gate"], V["results"]["fine_tuned"], r["external_all"]
        nc = V.get("deepdrid_near_copies_in_oiaddr", 0)
        tests = [("DeepDRiD sensitivity at least 66.1%", f"{100 * ve['sensitivity']:.1f}%", ve["sensitivity"] >= 0.661),
                 ("DeepDRiD specificity at least 85.0%", f"{100 * ve['specificity']:.1f}%", ve["specificity"] >= 0.850),
                 ("APTOS test AUC at least 0.950", f"{va['AUC']:.3f}", va["AUC"] >= 0.950),
                 ("At most 46 referable eyes given an all-clear (400-photo workflow)", str(vw["referable_given_all_clear"]),
                  vw["referable_given_all_clear"] <= 46),
                 ("Fewer than 16 DeepDRiD photos with a near-copy in the added photos", str(nc), nc < 16)]
        failed = [t for t in tests if not t[2]]
        d["v2_failed_n"], d["v2_tests_n"] = len(failed), len(tests)
        d["v2_verdict"] = ("it passed all of them, so v2 replaces v1 in the demo" if not failed else
                           f"it failed {len(failed)} of {len(tests)} pre-set tests, so v1 stays the demo model")
        d["v2_near_copies"] = nc
        T["v2_tests"] = table([(n, v, "pass" if ok else "**fail**") for n, v, ok in tests], ["Pre-registered test", "v2", "Result"])
        pct = lambda x: f"{100 * x:.1f}%"
        rows = [("DeepDRiD, all 1,600 photos: sensitivity", pct(v1e["sensitivity"]), pct(ve["sensitivity"])),
                ("DeepDRiD: specificity", pct(v1e["specificity"]), pct(ve["specificity"])),
                ("DeepDRiD: AUC", f"{v1e['AUC']:.3f}", f"{ve['AUC']:.3f}")]
        if "external_no_near_copies" in V["results"]:
            vn = V["results"]["external_no_near_copies"]
            rows.append((f"DeepDRiD without the {nc} near-copies: sensitivity / specificity", "(none removed)",
                         f"{pct(vn['sensitivity'])} / {pct(vn['specificity'])}"))
        rows += [("APTOS test (352): AUC", f"{r['fine_tuned']['AUC']:.3f}", f"{va['AUC']:.3f}"),
                 ("APTOS test: sensitivity / specificity", f"{pct(r['fine_tuned']['sensitivity'])} / {pct(r['fine_tuned']['specificity'])}",
                  f"{pct(va['sensitivity'])} / {pct(va['specificity'])}"),
                 ("400-photo workflow: refer / retake / rescreen",
                  f"{R['workflow']['with_quality_gate']['REFER']} / {R['workflow']['with_quality_gate']['RETAKE']} / {R['workflow']['with_quality_gate']['RESCREEN']}",
                  f"{vw['REFER']} / {vw['RETAKE']} / {vw['RESCREEN']}"),
                 ("400-photo workflow: referable photos given an all-clear (severe or proliferative)",
                  f"{R['workflow']['with_quality_gate']['referable_given_all_clear']} ({R['workflow']['with_quality_gate']['sight_threatening_given_all_clear']})",
                  f"{vw['referable_given_all_clear']} ({vw['sight_threatening_given_all_clear']})")]
        if "oiaddr_test" in V["results"]:
            o = V["results"]["oiaddr_test"]
            rows.append((f"OIA-DDR test ({o['n']}; same hospitals as its training photos): AUC, sensitivity / specificity", "not tested",
                         f"{o['AUC']:.3f}, {pct(o['sensitivity'])} / {pct(o['specificity'])}"))
        T["v2_compare"] = table(rows, ["", "v1 (APTOS only)", "v2 (APTOS + OIA-DDR)"])
    R["derived"], R["table"] = d, T
    return R


def checks(R, res_dir, allow_quick):
    c = R["config"]
    problems = []
    if not allow_quick and (c.get("max_images") or not c.get("pretrained", True) or c.get("epochs", 0) < 5):
        problems.append("These results come from a quick or test run (max_images set, no pretrained weights, or under 5 epochs). "
                        "Use the full run, or pass --allow-quick for testing only.")
    sp = os.path.join(res_dir, "summary.csv")
    if os.path.exists(sp):
        s = pd.read_csv(sp)
        by_name = {v["model"]: v for v in R["results"].values()}
        for _, row in s.iterrows():
            v = by_name.get(row["model"])
            for k in ("AUC", "sensitivity", "specificity"):
                if v is not None and k in row and abs(float(row[k]) - float(v[k])) > 0.0011:
                    problems.append(f"summary.csv and results.json disagree on {row['model']} {k}")
    for k, v in R["results"].items():
        for m in ("AUC", "sensitivity", "specificity"):
            if not 0 <= float(v[m]) <= 1:
                problems.append(f"{k} {m} out of range: {v[m]}")
    return problems


def gates(R):
    r = R["results"]
    ft, cb, lp = r["fine_tuned"], r["colour_baseline"], r["linear_probe"]
    print("\nDiagnostics (report them whatever they say; they are not a reason to change project):")
    print(f"  A. Fine-tuned beats both baselines on AUC: {ft['AUC'] > max(cb['AUC'], lp['AUC'])} "
          f"({ft['AUC']:.3f} vs {cb['AUC']:.3f}, {lp['AUC']:.3f}); on specificity: {ft['specificity'] > max(cb['specificity'], lp['specificity'])}")
    if "external_all" in r:
        e = r["external_all"]
        print(f"  B. External AUC {e['AUC']:.3f}, patient-level 95% CI {e['AUC_CI_by_patient'][0]:.3f} to {e['AUC_CI_by_patient'][1]:.3f}")
    if "workflow" in R:
        w, wo = R["workflow"]["with_quality_gate"], R["workflow"]["without_quality_gate"]
        print(f"  C. Referable all-clears without/with quality check: {wo['referable_given_all_clear']} / {w['referable_given_all_clear']}; "
              f"ungradable (grade 5) photos in this set: {R['workflow']['ungradable_photos']}")


def fill(text, R):
    missing = []

    def sub(m):
        path, spec = m.group(1).strip(), (m.group(2) or "").strip() or None
        try:
            if path.startswith("table."):
                return R["table"][path.split(".", 1)[1]]
            return fmt(get(R, path), spec)
        except (Missing, KeyError, TypeError, ValueError) as e:
            missing.append(f"{path} ({e})")
            return m.group(0)
    return TOKEN.sub(sub, text), missing


def main():
    a = argparse.ArgumentParser()
    a.add_argument("--results", default="results")
    a.add_argument("--allow-quick", action="store_true")
    a.add_argument("--out-dir", default=".")
    a.add_argument("--team", default="TEAM_INPUTS.md", help="the team's written pieces (optional)")
    x = a.parse_args()
    team = read_team(x.team)
    for k, v in team.items():
        n = len(v.split())
        if k in LIMITS and n > LIMITS[k]:
            print(f"Note: team piece '{k}' has {n} words; the limit is {LIMITS[k]}. Please shorten it.")
    R = build(x.results)
    problems = checks(R, x.results, x.allow_quick)
    if problems:
        print("STOPPED, nothing written:\n  " + "\n  ".join(problems))
        sys.exit(1)
    outputs, all_missing = {}, []
    for tpl, out in (("README.template.md", "README.md"), ("VIDEO_SCRIPT.template.md", "VIDEO_SCRIPT.md")):
        text, used = apply_team(open(tpl, encoding="utf-8").read(), team)
        text = re.sub(r"<!-- if:v2 -->(.*?)<!-- /if:v2 -->", (lambda m: m.group(1)) if "v2" in R else "", text, flags=re.S)
        # results/non_retinal.json comes from non_retinal_check.py (needs the model); in a fresh training run it may not exist yet
        text = re.sub(r"<!-- if:nr -->(.*?)<!-- /if:nr -->", (lambda m: m.group(1)) if "nr_n" in R.get("derived", {}) else "", text, flags=re.S)
        text, missing = fill(text, R)
        outputs[out] = text
        all_missing += [f"{tpl}: {m}" for m in missing]
    if all_missing:
        print("STOPPED, nothing written. These tokens could not be filled:\n  " + "\n  ".join(all_missing))
        sys.exit(1)
    os.makedirs(x.out_dir, exist_ok=True)
    for out, text in outputs.items():
        with open(os.path.join(x.out_dir, out), "w", encoding="utf-8") as f:
            f.write(text)
        left = TEAM.findall(text)
        print(f"Wrote {out}" + (f"; still for the team to write: {len(left)}" if left else ""))
        for t in left:
            print("   ", t[:110])
    words = len(re.sub(r"<!--.*?-->|\*\(.*?\)\*|#.*", "", outputs["VIDEO_SCRIPT.md"]).split())
    print(f"Video script: about {words} spoken words (aim for 260 to 300).")
    gates(R)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile README.template.md
# Roshni: open, offline diabetic eye triage for primary-care clinics in Pakistan

**Track:** Health
**Team:** Varun Bajpai (India), Laziza Farrukhovna (Uzbekistan), Zainab Jamshed (Pakistan)
**Try it:** [TEAM_TO_WRITE(demo_link): link to the browser demo, e.g. https://USERNAME.github.io/roshni/web/] · **Code:** [TEAM_TO_WRITE(repo_link): link to the public GitHub repository]

> This project is a prototype and is not medical advice.

![](docs/roshni_demo.png)

**Main takeaway:** Roshni ranks disease well at unseen hospitals (AUC [FROM_RESULTS_JSON: results.external_all.AUC | .3f]), but each clinic must re-set its cut-off locally, which an open tool allows.

<!-- Generated from README.template.md by fill_report.py. Edit the template, not this file. Numbers come only from results/. -->

## The problem
<!-- team:problem -->Pakistan has the world's highest diabetes prevalence: 31.4% of adults, about 34.5 million people, and a 2018 review of Pakistani studies found about 29% of people with diabetes had diabetic retinopathy (9% sight-threatening). It is treatable if caught early, but Pakistan has about 15 ophthalmologists per million, mostly in cities (high-income countries: 76). Screening is not available at primary level, travel to city hospitals is costly and harder for women, and surveyed eye hospitals had no screening policy.<!-- /team:problem -->

## What we built
**User:** a doctor or nurse in a Pakistani primary-care diabetes clinic with a retinal camera but no eye specialist. **In:** one retinal photo. **Out:** one next step for the doctor to confirm: **REFER** (moderate or worse, grade 2 to 4), **RETAKE** (too unclear to rule disease out) or **RESCREEN** in 12 months.
**Safety rule:** a photo judged too poor to read gets a retake instead of an all-clear, and never blocks a referral. It also shows the estimated grade, a heat map and an English and Urdu note.

## What we added
Our own idea (track 4.1.4).

- **What already exists:** FDA-cleared LumineticsCore and EyeArt, and Medios AI (offline, smartphone camera, India): all commercial and closed (appendix); UK estimates for AI graders were GBP 0.35 to 3.02 per patient (NICE 2021).
- **What we add:** an open, safety-first version anyone can re-test: thresholds frozen before testing, failures published, a retake rule that never blocks a referral, a pre-planned stress test, and a browser demo that keeps the photo on the device and works offline ([FROM_RESULTS_JSON: derived.cpu_short]).
- **Own question:** does the photo-quality check stop false all-clears? On [FROM_RESULTS_JSON: derived.val_photos | int] external photos, referable photos given an all-clear fell from [FROM_RESULTS_JSON: workflow.without_quality_gate.referable_given_all_clear | int] to [FROM_RESULTS_JSON: workflow.with_quality_gate.referable_given_all_clear | int] (severe or proliferative: [FROM_RESULTS_JSON: workflow.without_quality_gate.sight_threatening_given_all_clear | int] to [FROM_RESULTS_JSON: workflow.with_quality_gate.sight_threatening_given_all_clear | int]), costing [FROM_RESULTS_JSON: workflow.with_quality_gate.RETAKE | int] retakes.
- **New source:** trained on rural India (APTOS 2019), tested once on unseen Chinese hospital photos (DeepDRiD).

## How it works
- **Data:** APTOS 2019, 3,662 photos graded 0 to 4 (Aravind Eye Hospital, India), split by grade into [FROM_RESULTS_JSON: derived.split_counts], with [FROM_RESULTS_JSON: duplicates.pairs | int] near-duplicate pairs kept on one side of the split. External: [FROM_RESULTS_JSON: derived.ext_photos | int] DeepDRiD photos from [FROM_RESULTS_JSON: derived.ext_patients | int] patients with quality labels.
- **Model:** EfficientNet-B0 (ImageNet weights), fine-tuned for [FROM_RESULTS_JSON: derived.epochs_run | int] epochs at [FROM_RESULTS_JSON: frozen.size | int] px; referable score = probability of grade 2 or worse.
- **Frozen before testing:** the referral threshold ([FROM_RESULTS_JSON: frozen.ref_threshold | .3f]) was set for [FROM_RESULTS_JSON: frozen.target_sensitivity | pct0] sensitivity on validation; the quality check (logistic regression on image features, cut-off [FROM_RESULTS_JSON: frozen.poor_threshold | .2f]) was trained only on DeepDRiD's 1,200 training photos.
- **Baselines:** colour histogram, and untuned pretrained features, each + logistic regression, thresholded the same way.

## Results
Held-out APTOS test ([FROM_RESULTS_JSON: derived.aptos_test_n | int] photos, [FROM_RESULTS_JSON: derived.aptos_test_referable | int] referable) and external DeepDRiD:

[FROM_RESULTS_JSON: table.main]

- **By severity (APTOS test):** [FROM_RESULTS_JSON: derived.severe_sent | int] of [FROM_RESULTS_JSON: derived.severe_total | int] severe and [FROM_RESULTS_JSON: derived.prolif_sent | int] of [FROM_RESULTS_JSON: derived.prolif_total | int] proliferative photos were referred.
- **External photos:** at the frozen cut-off sensitivity fell to [FROM_RESULTS_JSON: results.external_all.sensitivity | pct] though AUC stayed [FROM_RESULTS_JSON: results.external_all.AUC | .3f]: the ranking carried over, the cut-off did not. Re-set on about [FROM_RESULTS_JSON: derived.lc_photos | int] local labelled photos, it gave [FROM_RESULTS_JSON: derived.lc_sens | .1f]% sensitivity and [FROM_RESULTS_JSON: derived.lc_spec | .1f]% specificity (post-hoc; appendix).
<!-- if:v2 -->- **v2, more cameras (pre-registered):** adding [FROM_RESULTS_JSON: v2.oiaddr.photos | int] photos from 147 hospitals raised DeepDRiD sensitivity to [FROM_RESULTS_JSON: v2.results.external_all.sensitivity | pct] (false all-clears [FROM_RESULTS_JSON: workflow.with_quality_gate.referable_given_all_clear | int] → [FROM_RESULTS_JSON: v2.workflow.with_quality_gate.referable_given_all_clear | int]) but cut specificity to [FROM_RESULTS_JSON: v2.results.external_all.specificity | pct], with [FROM_RESULTS_JSON: derived.v2_near_copies | int] near-copies of DeepDRiD photos among them: [FROM_RESULTS_JSON: derived.v2_verdict] (appendix).<!-- /if:v2 -->
- **Worse photos (stress test):** [FROM_RESULTS_JSON: derived.stress_brief] (appendix).
- **Where it fails:** [TEAM_TO_WRITE(where_it_fails): two missed eyes from results/missed_external.png (or the team-only APTOS gallery) and the reason you see in each, e.g. blur, small lesions near the edge, or a grade you would dispute]
- **What we learned:** [TEAM_TO_WRITE(what_we_learned): two or three sentences on what surprised you, e.g. how the external photos, the poor-quality photos or the stress test changed what you believed, and what you would do differently]

**Limitations.** Not tested on Pakistani patients or low-cost cameras; the stress test only imitates worse photos. APTOS grades come from single graders. Unlike commercial tools, we do not detect macular oedema, use one photo per eye and have no prospective trial. No age or sex is recorded; sensitivity differs by hospital source (appendix).

## What we'd do next
<!-- team:next_steps -->1. Test the frozen tool on the expert-graded Pakistani dataset registered at Al-Ibrahim Eye Hospital, Karachi (50,000 patients planned, NCT06287645), then in a primary-care clinic that already screens.
2. Test it with the low-cost cameras such clinics can afford, and ask doctors and camera operators whether the three actions and the Urdu note fit their routine.<!-- /team:next_steps -->

## Sources
- IDF Diabetes Atlas, Pakistan: https://diabetesatlas.org/data-by-location/country/pakistan
- Mumtaz et al. 2018, diabetic retinopathy in Pakistan, systematic review: https://pjms.com.pk/index.php/pjms/article/view/13819
- Akhtar et al. 2026, diabetic retinopathy in type 2 diabetes in Pakistan, systematic review and meta-analysis (Front Clin Diabetes Healthc; pooled prevalence 32.9%, 60 studies): https://pubmed.ncbi.nlm.nih.gov/41982556/
- Ophthalmologists per million in Pakistan (The News, 2026): https://www.thenews.pk/print/1422954-rising-cataract-burden-outpaces-pakistan-s-healthcare-capacity
- Sightsavers 2021, diabetic retinopathy services in Pakistan (primary level, cities, barriers for women): https://www.sightsavers.org/blogs/2021/06/diabetic-retinopathy-in-pakistan
- Shah et al. 2019, screening policy in Pakistani eye hospitals: https://www.ijo.cn/gjyken/article/pdf/20190517
- Siddiqui, Zafar and Qazi 2023, AI and the future of healthcare in Pakistan (J Pak Med Assoc): https://jpma.org.pk/index.php/public_html/article/view/10917/2701
- Pakistan public primary-care spending (World Bank, Health Financing System Assessment): https://openknowledge.worldbank.org/bitstreams/716ee589-8e2b-4c9c-9759-b2d45c68ee01/download
- Barriers to eye care in Pakistan (Malik et al. 2025): https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12260732/
- Pakistani expert-graded retinal dataset registered on ClinicalTrials.gov (NCT06287645; site: Al-Ibrahim Eye Hospital, Karachi): https://clinicaltrials.gov/study/NCT06287645
- NICE MIB265 (2021), AI technologies for detecting diabetic retinopathy (cost per patient): https://www.nice.org.uk/advice/mib265/chapter/summary
- Jawa et al. 2016, integrated primary-care model for DR detection in Pakistan (Pak J Med Sci; cites the 223-patient clinic study): https://pjms.com.pk/index.php/pjms/article/view/10955
- Ahsan et al. 2021, one-window diabetes eye care facility at Al-Ibrahim Eye Hospital (Pak J Med Sci): https://pjms.org.pk/index.php/pjms/article/download/3946/1100
- Topcon TRC-NW400 camera about US$15,000 to 22,000; IDx-DR per-image fee not public (AAFP 2020): https://www.aafp.org/pubs/afp/issues/2020/0301/p307.html
- Google's Thailand deployment (MIT Technology Review 2020): https://www.technologyreview.com/2020/04/27/1000658/google-medical-ai-accurate-lab-real-life-clinic-covid-diabetes-retina-disease/amp/ and (TechCrunch 2020): https://techcrunch.com/2020/04/27/google-medical-researchers-humbled-when-ai-screening-tool-falls-short-in-real-life-testing/
- LumineticsCore (IDx-DR) FDA 510(k) summary K203629: https://www.accessdata.fda.gov/cdrh_docs/pdf20/K203629.pdf
- EyeArt FDA 510(k) summary K200667: https://www.accessdata.fda.gov/cdrh_docs/pdf20/K200667.pdf
- LumineticsCore pivotal trial (Abramoff et al. 2018): https://www.nature.com/articles/s41746-018-0040-6
- EyeArt pivotal trial (Ipp et al. 2021): https://pubmed.ncbi.nlm.nih.gov/34779843/
- Medios AI, offline smartphone screening in India (Natarajan et al., JAMA Ophthalmology 2019): https://pubmed.ncbi.nlm.nih.gov/31393538/ and in Dominica (Kemp et al., BMJ Open Ophthalmology 2023): https://bmjophth.bmj.com/content/8/1/e001491
- APTOS 2019 Blindness Detection: https://www.kaggle.com/c/aptos2019-blindness-detection
- DeepDRiD (Liu et al., Patterns 2022): https://github.com/deepdrdoc/DeepDRiD
- OIA-DDR (Li et al., Information Sciences 2019; used only in the v2 experiment): https://github.com/nkicsl/DDR-dataset and https://doi.org/10.1016/j.ins.2019.06.011
- Pretrained model: EfficientNet-B0 with ImageNet weights (torchvision `IMAGENET1K_V1`): https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html; EfficientNet paper (Tan and Le 2019): https://arxiv.org/abs/1905.11946
- Grad-CAM (Selvaraju et al. 2017): https://arxiv.org/abs/1610.02391
- Libraries: PyTorch https://pytorch.org, scikit-learn https://scikit-learn.org, pandas https://pandas.pydata.org, Pillow https://python-pillow.org, matplotlib https://matplotlib.org, Gradio https://www.gradio.app, ONNX Runtime and ONNX Runtime Web https://onnxruntime.ai, NumPy https://numpy.org, scikit-image (sample photos in `non_retinal_check.py`) https://scikit-image.org
- Tools for tests and the video only: Playwright https://playwright.dev, FFmpeg https://ffmpeg.org, Node.js https://nodejs.org
- AI tools: [TEAM_TO_WRITE(ai_tools): every assistant used, e.g. Claude (Anthropic) for code and research support]

---

## Appendix: technical details

**How it works, end to end (everything runs inside the browser; the photo is never uploaded)**

```mermaid
flowchart LR
  A["Retinal photo<br/>(clinic camera or file)"] --> B["Crop the black border,<br/>resize to 384 px<br/>(same arithmetic as Python)"]
  B --> C["EfficientNet-B0<br/>ONNX Runtime Web (WebAssembly)"]
  C --> D["Referable score<br/>P(grade 2 or worse)"]
  C --> E["Image features"]
  E --> F["Quality check<br/>(logistic regression)"]
  C --> H["Grad-CAM heat map"]
  D --> G{"Decision rule<br/>(thresholds frozen<br/>before testing)"}
  F --> G
  G -->|"score at or above [FROM_RESULTS_JSON: frozen.ref_threshold | .3f]"| R["REFER"]
  G -->|"otherwise, poor photo"| T["RETAKE"]
  G -->|"otherwise"| S["RESCREEN"]
  R --> N["Note for the doctor<br/>in English and Urdu"]
  T --> N
  S --> N
```

After the first visit, a service worker keeps the page, the model and the runtime on the device, so screening also works with no internet.

**How this maps to the Round 1 rubric** (highschoolhackathon.org/judges)

| Criterion (points) | Evidence in this project |
|---|---|
| Technical execution (30) | A demo that runs end to end in the browser and offline, with the browser model checked against PyTorch; EfficientNet-B0 fine-tuned with a near-duplicate-safe split; two baselines; threshold frozen on validation; held-out test; untouched external set from another country with patient-level confidence intervals |
| Originality (20) | Our own question (does a quality check stop false all-clears?); a retake rule that never blocks a referral; a proven screening workflow rebuilt in the open and re-settable by each clinic; tested on deliberately worse photos and on pictures that are not retinal photos |
| Impact and track fit (20) | A named user and moment in Pakistani primary care; three actions a clinic can act on; sourced diabetes and eye-workforce figures for Pakistan; no server, account or licence; a note in Urdu checked by a native speaker |
| Testing and honesty (15) | Missed cases and false alarms reported separately; named missed photos; a stress test planned before results; the drop at new hospitals reported ([FROM_RESULTS_JSON: results.external_all.sensitivity | pct]); a pre-registered experiment (v2) reported although it failed; post-hoc analyses labelled as post-hoc; a real-world yardstick that none of our settings meets yet |
| Communication (15) | A screenshot and one main takeaway at the top; three actions (refer, retake, rescreen); a two-minute video of the real demo; every number generated from `results/` |

**What already exists, and where Roshni differs** (from FDA summaries and published studies; their accuracy comes from prospective studies, ours from public photos)

| | LumineticsCore (IDx-DR) | EyeArt | Medios AI | Roshni |
|---|---|---|---|---|
| Who takes the photo | Primary-care staff | Primary-care or eye-care staff | A minimally trained health worker | Primary-care doctor or nurse |
| Camera | 2 photos per eye, Topcon NW400 only | 2 photos per eye, named Canon cameras | Remidio smartphone-based camera | 1 photo, no camera model required; not yet tested on low-cost cameras |
| Referral line | More than mild DR, including macular oedema | Moderate or worse, or macular oedema | More than mild DR, with or without macular oedema | Moderate or worse (grade 2 to 4); no macular oedema |
| Poor photo | "Insufficient image quality" | "Ungradable" | Not stated in the studies we read | Retake, or refer if disease is already visible |
| Where it runs | Remote servers | Remote servers | Offline, on the camera's smartphone | A web browser, offline after the first visit; or a CPU alone ([FROM_RESULTS_JSON: derived.cpu_short]) |
| Code and model | Commercial | Commercial | Proprietary | Open |
| Evidence | Prospective trial: 900 patients, 10 primary-care sites | Prospective trial: 893 patients, 15 sites | 213 participants in India (sensitivity 100%, specificity 88.4%); in real-world use in Dominica, 77.5% and 91.5% | Retrospective: held-out and external public photos, plus a stress test; not tested in Pakistan |

<!-- team:adoption --><!-- /team:adoption -->

**Context, not head-to-head:** in prospective trials graded by a reading centre and covering macular oedema, LumineticsCore reported 87.2% sensitivity and 90.7% specificity (900 patients) and EyeArt 95.5% and 85.0% (893 patients).

**What a clinic needs to run it**

| Need | Roshni |
|---|---|
| Device | A phone or computer with a current web browser (tested in Chrome) |
| Internet | Only for the first visit, about 31 MB (model 16 MB + ONNX Runtime 15 MB); after that it works with the network switched off (tested) |
| Photo | One retinal photo (JPEG or PNG) from any fundus camera |
| Server, account, licence fee, graphics card | None |
| Speed | [FROM_RESULTS_JSON: derived.cpu_sentence]; the browser demo shows the time on each device |

**Why each design choice (evidence behind it)**

| Choice | Evidence |
|---|---|
| Refer, retake, or rescreen in 12 months | LumineticsCore's outputs and 12-month advice |
| A photo too poor to read is retaken, not cleared, and never blocks a referral | Google's Thailand deployment rejected more than a fifth of photos and left patients waiting; EyeArt could not grade 12.6% of undilated eyes |
| Runs in a browser or on a CPU, offline | Internet outages halved screening at one Thai clinic; Pakistan's public primary-care spending is about US$16 per person |
| One photo, any camera, no licence fee | LumineticsCore needs a US$15,000 to 22,000 Topcon camera and two photos per eye, with a per-image fee that is not public; Medios is tied to Remidio's camera |
| Open, frozen thresholds | No public expert-graded Pakistani dataset yet, so a local clinic must be able to re-test the tool itself |
| Urdu note | Illiteracy and female gender are documented barriers to eye care in Pakistan |

**Stress test (planned before any result).** Every held-out APTOS test photo and every DeepDRiD validation photo was made worse in four fixed ways, at two strengths: blur (Gaussian radius 1.5 and 3 px), lower resolution (shrunk to 192 and 96 px, then enlarged back), under-exposure (brightness ×0.6 and ×0.35), and messaging-app compression (JPEG quality 30 and 8). The frozen tool was scored on each copy without re-tuning anything. It imitates worse photos; it is not a test on real low-cost cameras. Result: [FROM_RESULTS_JSON: derived.stress_sentence]. See `results/stress_test.png` and `results/stress_examples.png`.

[FROM_RESULTS_JSON: table.stress]

**External results by photo quality and hospital source (DeepDRiD).** Sensitivity was [FROM_RESULTS_JSON: results.external_good_photos.sensitivity | pct] on good photos and [FROM_RESULTS_JSON: results.external_poor_photos.sensitivity | pct] on poor ones. It does not perform equally across hospital sources: sensitivity ranged from [FROM_RESULTS_JSON: derived.src_sens_min | .1f]% to [FROM_RESULTS_JSON: derived.src_sens_max | .1f]% (two of the three sources are small).

[FROM_RESULTS_JSON: table.by_source]

**APTOS test, by grade** (five-grade agreement, quadratic weighted kappa: [FROM_RESULTS_JSON: aptos_grades.QWK | .3f])

[FROM_RESULTS_JSON: table.by_grade]

**Workflow with and without the quality check ([FROM_RESULTS_JSON: derived.val_photos | int] DeepDRiD validation photos)**

[FROM_RESULTS_JSON: table.workflow]

<!-- if:v2 -->**v2: more cameras in training (pre-registered).** The plan, the tests and the decision rule were committed to GitHub (`PREREGISTRATION_v2.md`) before v2 was trained. v2 is the same pipeline with the OIA-DDR grading photos (Li et al. 2019; [FROM_RESULTS_JSON: v2.oiaddr.photos | int] gradable photos from 147 hospitals in China, 42 camera models; CC BY-NC-SA 4.0) added to training and validation, split the same near-duplicate-safe way ([FROM_RESULTS_JSON: v2.oiaddr.split.train | int] / [FROM_RESULTS_JSON: v2.oiaddr.split.val | int] / [FROM_RESULTS_JSON: v2.oiaddr.split.test | int]). Its referral cut-off ([FROM_RESULTS_JSON: v2.frozen.ref_threshold | .3f]) was frozen on the combined validation photos. More camera variety closed most of the missed-referral gap on the unseen hospitals but cost specificity, and the hash check found [FROM_RESULTS_JSON: derived.v2_near_copies | int] DeepDRiD photos with near-copies among the added photos (removing them changes little). Verdict: [FROM_RESULTS_JSON: derived.v2_verdict]. Both OIA-DDR and DeepDRiD are from China, so neither model is shown to work in Pakistan. Files: `results_v2/`, `Roshni_v2_OIA-DDR.ipynb`.

[FROM_RESULTS_JSON: table.v2_compare]

[FROM_RESULTS_JSON: table.v2_tests]<!-- /if:v2 -->

**Re-setting the cut-off locally (added after seeing the external result; the frozen results above stay the main ones).** We drew a "clinic sample" of DeepDRiD training patients (whole patients, all their photos), set the referral cut-off for [FROM_RESULTS_JSON: frozen.target_sensitivity | pct0] sensitivity on it with the same function used to freeze the original, and scored the full workflow on the 400 validation photos from [FROM_RESULTS_JSON: derived.lc_eval_patients | int] other patients; the grading model never saw any DeepDRiD photo. Over [FROM_RESULTS_JSON: derived.lc_draws | int] random samples of [FROM_RESULTS_JSON: derived.lc_patients | int] patients: sensitivity [FROM_RESULTS_JSON: derived.lc_sens | .1f]% (10th to 90th percentile [FROM_RESULTS_JSON: derived.lc_sens_range]) instead of [FROM_RESULTS_JSON: derived.lc_frozen_sens | .1f]%, specificity [FROM_RESULTS_JSON: derived.lc_spec | .1f]% ([FROM_RESULTS_JSON: derived.lc_spec_range]) instead of [FROM_RESULTS_JSON: derived.lc_frozen_spec | .1f]%; referable photos given an all-clear [FROM_RESULTS_JSON: derived.lc_allclear | int] instead of [FROM_RESULTS_JSON: derived.lc_frozen_allclear | int] (severe or proliferative: [FROM_RESULTS_JSON: derived.lc_sev_allclear | int]), at the cost of [FROM_RESULTS_JSON: derived.lc_refer | int] referrals instead of [FROM_RESULTS_JSON: derived.lc_frozen_refer | int]. The same hospitals supply both sets, so this shows the procedure works, not that one cut-off fits every clinic. See `results/local_calibration.png` and `local_calibration.py`.

**Two photos per eye (added after seeing the external result; nothing re-trained or re-tuned).** DeepDRiD has two photos of every eye, as LumineticsCore and EyeArt ask for. Referring an eye when either of its photos scores at or above the frozen cut-off raised sensitivity on the unseen hospitals from [FROM_RESULTS_JSON: derived.tp_one_sens | .1f]% per photo to [FROM_RESULTS_JSON: derived.tp_two_sens | .1f]% per eye ([FROM_RESULTS_JSON: derived.tp_eyes | int] eyes of [FROM_RESULTS_JSON: derived.tp_patients | int] patients; 95% interval [FROM_RESULTS_JSON: derived.tp_two_ci | str], resampling patients), while specificity went from [FROM_RESULTS_JSON: derived.tp_one_spec | .1f]% to [FROM_RESULTS_JSON: derived.tp_two_spec | .1f]%. On the [FROM_RESULTS_JSON: derived.tp_val_eyes | int] validation eyes, taking the more cautious of the two decisions left [FROM_RESULTS_JSON: derived.tp_val_allclear | int] referable eyes with an all-clear (severe or proliferative: [FROM_RESULTS_JSON: derived.tp_val_sev | int]) but sent [FROM_RESULTS_JSON: derived.tp_val_retake | int] eyes to a retake. The unit changes from photos to eyes, so this points to a clinic protocol (two photos per eye) rather than replacing the frozen results. See `two_photo.py`.

<!-- if:nr -->**Pictures that are not retinal photos.** Roshni has no separate "is this a retina?" check, so we screened [FROM_RESULTS_JSON: derived.nr_n | int] non-retinal pictures (plain and noisy images, a page of text, a chart, everyday photos): [FROM_RESULTS_JSON: derived.nr_allclear_words | str] got an all-clear; [FROM_RESULTS_JSON: derived.nr_retake | int] were sent to a retake and [FROM_RESULTS_JSON: derived.nr_refer | int] to a referral (a wasted referral, not a missed one). See `non_retinal_check.py`.<!-- /if:nr -->

**Against a real-world bar.** LumineticsCore's pivotal trial fixed its targets in advance: sensitivity above 85% and specificity above 82.5% (it reached 87.2% and 90.7%; Abramoff et al. 2018). Its reference standard, unit and patients differ from ours, so this is a yardstick, not a head-to-head test. [FROM_RESULTS_JSON: derived.bar_met_words | str], which is why Roshni stays a research prototype and why the next step is a local test with targets fixed in advance.

[FROM_RESULTS_JSON: table.bar]

**Photo-quality check:** found poor photos with AUC [FROM_RESULTS_JSON: quality_gate.AUC_detecting_poor_photos | .3f]; flags [FROM_RESULTS_JSON: quality_gate.poor_photos_flagged | pct] of poor photos and [FROM_RESULTS_JSON: quality_gate.good_photos_flagged | pct] of good ones ([FROM_RESULTS_JSON: quality_gate.flagged_share | pct] of all photos sent to retake or refer on quality).

**External AUC interval** resampling whole patients (each has up to four photos): [FROM_RESULTS_JSON: results.external_all.AUC_CI_by_patient | ci].

**Code and how to run**

| File | What it is |
|---|---|
| `Roshni_Results.ipynb` | **Start here.** Saved with its outputs: recomputes every number above from the per-photo predictions, runs the trained model on the demo photos with heat maps, checks the browser model against PyTorch, and (with DeepDRiD downloaded) re-scores the 400 external validation photos. Runs on a CPU in minutes |
| `Roshni_DR_Screening.ipynb` | The full training run on a free Kaggle GPU (the experiment itself ran in [FROM_RESULTS_JSON: derived.run_minutes | int] minutes): data, near-duplicate split, baselines, training, frozen thresholds, external test, quality check, stress test, CPU timing, web export |
| `drscreen.py` | The core: preprocessing, model, metrics, the decision rule, stress-test changes, Grad-CAM, the English and Urdu note |
| `run_dr.py` | The experiment, end to end; writes everything in `results/` |
| `PREREGISTRATION_v2.md`, `Roshni_v2_OIA-DDR.ipynb` | v2: the same pipeline with the OIA-DDR photos (147 hospitals, 42 camera models) added to training, with its success tests fixed and committed before it was run |
| `local_calibration.py` | Re-setting the cut-off on a small local sample (added after the external result; needs no GPU) |
| `two_photo.py`, `non_retinal_check.py` | Two photos per eye, and pictures that are not retinal photos (both added after the main results; the second needs the model, no GPU) |
| `results/` | Outputs of the run: `results.json`, per-photo predictions, the model (`model_best.pt`) and quality check (`quality_gate.pkl`), figures (no APTOS photos) |
| `story/` | A one-page scroll story of the project that links to the demo (no outside requests; numbers from `results/`) |
| `web/` | The browser demo (ONNX Runtime Web; works offline after the first visit); `export_web.py` builds it from the run |
| `app.py` | An optional Gradio version of the demo |
| `fill_report.py`, `README.template.md` | Writes this README from `results/` so no number is typed by hand; `check_submission.py` checks it against the rules |
| `tests/` | Unit tests (`pytest -q tests`) and browser checks |

**Reproduce:** open `Roshni_DR_Screening.ipynb` on Kaggle (APTOS 2019 attached, GPU and Internet on) and run all; then `python fill_report.py`. To check the results without a GPU: `pip install -r requirements.txt`, then run `Roshni_Results.ipynb`.

**Licences:** our code is MIT (`LICENSE`). The demo photos in `web/examples` and the screenshot in `docs/` are from DeepDRiD (CC BY-SA 4.0, Liu et al. 2022). ONNX Runtime Web is MIT (`web/ort/NOTICE.md`). No APTOS image is included, following the competition's rules; the trained weights were fit on APTOS 2019 data, so check those rules before using them beyond research. v2's added training data (OIA-DDR) is CC BY-NC-SA 4.0, non-commercial. Roshni is a research prototype: not a medical device, not registered with Pakistan's drug regulator (DRAP), and not for decisions about real patients.


In [ ]:
%%writefile VIDEO_SCRIPT.template.md
# Roshni: 2-minute video script

<!-- Generated by fill_report.py. Follows the Hacker Guide's structure: problem (15 s), demo (60 s), how it works and the baseline (30 s), limitation and next step (15 s). About 270 spoken words; each part has a word budget of about 2.5 words per second. Rehearse once with a timer. -->

## 0:00 to 0:15 · Zainab · The problem
*(On screen: title card, then the three Pakistan numbers.)*

In Pakistan, about one adult in three has diabetes. Diabetic eye disease is treatable when caught early, but screening sits mostly in eye hospitals, and there are only about fifteen eye specialists per million people.

## 0:15 to 1:15 · Varun · The demo
*(On screen: the browser demo.)*

This is Roshni, running entirely inside a web browser. At a routine diabetes visit, the nurse photographs the back of the eye. Roshni gives the doctor one of three next steps to confirm, and the photo never leaves the device. This photo shows referable disease, so it says refer; the heat map shows where it looked. This next photo is too poor to read. Roshni asks for a retake instead of an all-clear. But if a poor photo already shows serious disease, it still says refer: a bad photo never blocks a referral. The note prints in Urdu, so the patient knows what happens next. Underneath: an open EfficientNet model trained on photos from rural India, plus a photo-quality check.

## 1:15 to 1:45 · Laziza · How we know it works
*(On screen: the results table, then the stress-test chart.)*

We froze every threshold before testing. On [FROM_RESULTS_JSON: derived.aptos_test_n | int] unseen Indian photos, Roshni caught [FROM_RESULTS_JSON: results.fine_tuned.sensitivity | pct] of referable photos at [FROM_RESULTS_JSON: results.fine_tuned.specificity | pct] specificity: AUC [FROM_RESULTS_JSON: results.fine_tuned.AUC | .2f], against [FROM_RESULTS_JSON: results.colour_baseline.AUC | .2f] for a colour baseline. On [FROM_RESULTS_JSON: derived.ext_photos | int] Chinese hospital photos it had never seen, the AUC was [FROM_RESULTS_JSON: results.external_all.AUC | .2f], but it caught only [FROM_RESULTS_JSON: results.external_all.sensitivity | pct], so a clinic must re-check the threshold on its own photos. The quality check [FROM_RESULTS_JSON: derived.gate_short]. [FROM_RESULTS_JSON: derived.stress_short]

## 1:45 to 2:00 · Zainab · Limits and next step
*(On screen: results/missed_external.png, then the limits slide. Both show only DeepDRiD photos, which may be shared with credit; keep APTOS photos off the public video.)*

Roshni still misses disease, including [TEAM_TO_WRITE(video_miss_reason): the main reason found in the missed photos, in a few words], and we had no Pakistani photos to test on. Next: graded Pakistani photos and low-cost cameras. Roshni is a prototype, not medical advice.


In [ ]:
%%writefile export_web.py
"""Export the trained, frozen Roshni tool for the in-browser demo (web/).

  python export_web.py --results results --out web

Writes:
  web/model.onnx        the fine-tuned EfficientNet-B0: outputs grade logits, image features and the last feature map
  web/roshni_web.json   frozen thresholds, the photo-quality check, the classifier weights (for the heat map),
                        and the note texts, all copied from the run's own files
Then checks that ONNX Runtime gives the same answers as PyTorch on a few photos.
The demo page (web/index.html) runs everything in the browser: the photo never leaves the device.
"""
import argparse
import json
import os
import pickle

import numpy as np
import torch

import drscreen as ds


class _Export(torch.nn.Module):
    """Same network, returning (logits, pooled features, last feature map) in one forward pass."""

    def __init__(self, m):
        super().__init__()
        self.features, self.avgpool, self.classifier = m.features, m.avgpool, m.classifier

    def forward(self, x):
        fmap = self.features(x)
        emb = torch.flatten(self.avgpool(fmap), 1)
        return self.classifier(emb), emb, fmap


def _examples(results_dir, out_dir, n_each=1):
    """Copy a few DeepDRiD validation photos (CC BY-SA 4.0) for the demo, with the experts' grade and quality label.
    One photo per rule the demo shows, each from a different patient, chosen well away from both frozen cut-offs so
    small numeric differences between browsers cannot flip the answer. The last one is a referable eye the tool misses,
    shown on purpose. APTOS photos are not copied: the competition's rules do not allow redistributing its images."""
    import shutil
    import pandas as pd
    p = os.path.join(results_dir, "deepdrid_predictions.csv")
    rj = os.path.join(results_dir, "results.json")
    if not os.path.exists(p) or not os.path.exists(rj):
        return []
    fr = json.load(open(rj))["frozen"]
    rt, pt = fr["ref_threshold"], fr["poor_threshold"]
    dd = pd.read_csv(p)
    v = dd[(dd.part == "validation") & (dd.grade <= 4)].copy()
    v["m_ref"], v["m_poor"] = (v.p_referable - rt).abs(), (v.p_poor - pt).abs()
    safe = v[(v.m_ref > 0.15) & (v.m_poor > 0.15)]
    picks = [  # (rows, sort key, note)
        (safe[(safe.decision == "REFER") & (safe.grade >= 3) & (safe.poor_quality == 0)], "p_referable", False, ""),
        (safe[(safe.decision == "REFER") & (safe.grade >= 2) & (safe.p_poor > pt)], "p_poor", False, ""),  # check says poor, still refers
        (safe[(safe.decision == "RETAKE") & (safe.poor_quality == 1) & (safe.grade <= 1)], "p_poor", False, ""),
        (safe[(safe.decision == "RESCREEN") & (safe.grade == 0) & (safe.poor_quality == 0)], "p_poor", True, ""),
        (safe[(safe.decision == "RESCREEN") & (safe.grade == 2) & (safe.poor_quality == 0)], "p_referable", True,
         ". Shown on purpose: one of the referable photos Roshni misses"),
    ]
    os.makedirs(os.path.join(out_dir, "examples"), exist_ok=True)
    out, patients = [], set()
    for rows, key, asc, note in picks:
        for r in rows.sort_values(key, ascending=asc).itertuples():
            if r.patient_id in patients or not os.path.exists(r.cached):
                continue
            patients.add(r.patient_id)
            name = f"examples/{r.image_id}.jpg"
            shutil.copy(r.cached, os.path.join(out_dir, name))
            out.append(dict(file=name, caption=f"Expert grade {r.grade} ({ds.GRADES[r.grade]}), "
                                                 f"photo quality {'poor' if r.poor_quality else 'good'}{note}"))
            break
    return out


def export(results_dir="results", out_dir="web", check_paths=(), repo_url=None):
    with open(os.path.join(results_dir, "results.json")) as f:
        res = json.load(f)
    fr = res["frozen"]
    if fr["arch"] != "efficientnet_b0":
        raise ValueError("web export supports efficientnet_b0 only")
    model = ds.build_model(fr["arch"], pretrained=False)
    model.load_state_dict(torch.load(os.path.join(results_dir, "model_best.pt"), map_location="cpu"))
    model.eval()
    net = _Export(model).eval()
    os.makedirs(out_dir, exist_ok=True)
    size = int(fr["size"])
    dummy = torch.zeros(1, 3, size, size)
    kw = dict(input_names=["image"], output_names=["logits", "features", "feature_map"], opset_version=17)
    try:
        torch.onnx.export(net, dummy, os.path.join(out_dir, "model.onnx"), dynamo=False, **kw)
    except TypeError:  # older torch without the dynamo argument
        torch.onnx.export(net, dummy, os.path.join(out_dir, "model.onnx"), **kw)

    gate = None
    gp = os.path.join(results_dir, "quality_gate.pkl")
    if os.path.exists(gp):
        with open(gp, "rb") as f:
            pipe = pickle.load(f)
        sc, lr = pipe.steps[0][1], pipe.steps[-1][1]
        gate = dict(mean=sc.mean_.tolist(), scale=sc.scale_.tolist(), coef=lr.coef_[0].tolist(),
                    intercept=float(lr.intercept_[0]))
    lin = model.classifier[1]
    cfg = dict(
        size=size, mean=list(ds.MEAN), std=list(ds.STD), crop_tol=10, referable_from=ds.REFERABLE_FROM,
        ref_threshold=float(fr["ref_threshold"]), poor_threshold=float(fr["poor_threshold"]),
        target_sensitivity=float(fr["target_sensitivity"]), grades=ds.GRADES, gate=gate,
        head_weight=lin.weight.detach().numpy().round(6).tolist(), head_bias=lin.bias.detach().numpy().round(6).tolist(),
        langs={k: dict(name=ds.LANGS[k]["name"], dir=ds.LANGS[k]["dir"], checked=ds.LANGS[k]["checked"]) for k in ds.shown_langs()},
        default_langs=list(ds.DEFAULT_LANGS),
        report={d: {k: t for k, t in v.items() if k in ds.shown_langs()} for d, v in ds.REPORT.items()}, summary=_summary(res), repo_url=repo_url,
        examples=_examples(results_dir, out_dir),
        examples_credit="Example photos: DeepDRiD (Liu et al., Patterns 2022), CC BY-SA 4.0. "
                        "Captions give the experts' grade and photo-quality label, not the tool's answer.",
    )
    with open(os.path.join(out_dir, "roshni_web.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False)
    _stamp_offline_cache(out_dir)
    return _check(model, pipe if gate else None, fr, out_dir, check_paths)


def _stamp_offline_cache(out_dir):
    """Name the offline cache after every file of the demo, so any change (model, config, page, examples) replaces the old cache."""
    import hashlib
    import re
    sw = os.path.join(out_dir, "sw.js")
    if not os.path.exists(sw):
        return
    h = hashlib.sha256()
    for d, _, files in sorted(os.walk(out_dir)):
        for f in sorted(files):
            p = os.path.join(d, f)
            if os.path.abspath(p) == os.path.abspath(sw):
                continue
            h.update(os.path.relpath(p, out_dir).encode())
            with open(p, "rb") as fh:
                h.update(fh.read())
    text = open(sw, encoding="utf-8").read()
    text = re.sub(r'const VERSION = "[^"]*";', f'const VERSION = "roshni-{h.hexdigest()[:12]}";', text)
    open(sw, "w", encoding="utf-8").write(text)


def _summary(res):
    """A few headline numbers for the page footer, copied from results.json (nothing typed by hand)."""
    r = res.get("results", {})
    out = {}
    for k in ("fine_tuned", "external_all"):
        if k in r:
            out[k] = dict(n=r[k]["n"], AUC=r[k]["AUC"], sensitivity=r[k]["sensitivity"], specificity=r[k]["specificity"])
    return out


def _check(model, gate, fr, out_dir, paths):
    """PyTorch and ONNX Runtime must agree on the same preprocessed photos."""
    try:
        import onnxruntime as ort
    except ImportError:
        print("onnxruntime not installed: skipped the agreement check")
        return None
    sess = ort.InferenceSession(os.path.join(out_dir, "model.onnx"), providers=["CPUExecutionProvider"])
    tf = ds.get_transforms(fr["size"], False)
    worst = 0.0
    for p in list(paths)[:5]:
        x = tf(ds.prep_image(p, fr["size"])).unsqueeze(0)
        d, p_ref, p_poor, P, e = ds.triage_one(model, gate, p, fr["size"], fr["ref_threshold"], fr["poor_threshold"], tf)
        logits, emb, _ = sess.run(None, {"image": x.numpy()})
        Po = np.exp(logits - logits.max(1, keepdims=True))
        Po /= Po.sum(1, keepdims=True)
        worst = max(worst, float(np.abs(Po[0] - P).max()), float(np.abs(emb - e).max()))
    print(f"ONNX vs PyTorch, largest difference on {min(5, len(list(paths)))} photos: {worst:.2e}")
    if worst > 1e-3:
        raise RuntimeError("ONNX export does not match PyTorch")
    return worst


if __name__ == "__main__":
    a = argparse.ArgumentParser()
    a.add_argument("--results", default="results")
    a.add_argument("--out", default="web")
    a.add_argument("--check", nargs="*", default=[])
    a.add_argument("--repo-url", default=None)
    x = a.parse_args()
    export(x.results, x.out, x.check, x.repo_url)


In [ ]:
%%writefile local_calibration.py
"""Local re-calibration: does re-setting the referral cut-off on a few labelled photos from a new clinic fix the
external drop?

Added AFTER we saw the external result (sensitivity fell to about 56% at the frozen cut-off), to test our own
advice that "each clinic must re-set the cut-off on its own graded photos". The model, the quality check and the
frozen results are untouched and stay the main results. Uses only the saved per-photo predictions, so it needs
no GPU and no photos:

  * calibration photos: DeepDRiD *training* patients (the grading model never saw any DeepDRiD photo)
  * evaluation photos:  the 400 DeepDRiD *validation* photos, from 100 other patients
  * a "clinic sample" is drawn by patient (all of a patient's photos together), the cut-off is set for 90%
    sensitivity on it with the same function used to freeze the original cut-off, and the full workflow
    (refer / retake / rescreen, with the quality check) is scored on the validation photos. Repeated many times.

Writes results/local_calibration.json and results/local_calibration.png.
Run:  python local_calibration.py [--results results] [--draws 500]
"""
import argparse
import json
import os

import numpy as np
import pandas as pd

import drscreen as ds

SAMPLE_PATIENTS = (12, 25, 50, 100)      # about 4 photos per patient: 48, 100, 200, 400 photos


def workflow(d, ref_thr, poor_thr):
    y, sev = (d.grade >= 2).values, (d.grade >= 3).values
    ref = d.p_referable.values >= ref_thr
    dec = np.asarray(ds.decide(d.p_referable.values, d.p_poor.values, ref_thr, poor_thr))
    return dict(threshold=float(ref_thr), sensitivity=float((ref & y).sum() / y.sum()),
                specificity=float((~ref & ~y).sum() / (~y).sum()),
                REFER=int((dec == "REFER").sum()), RETAKE=int((dec == "RETAKE").sum()), RESCREEN=int((dec == "RESCREEN").sum()),
                referable_given_all_clear=int(((dec == "RESCREEN") & y).sum()),
                sight_threatening_given_all_clear=int(((dec == "RESCREEN") & sev).sum()))


def run(results_dir="results", draws=500, seed=0):
    res = json.load(open(os.path.join(results_dir, "results.json")))
    fr = res["frozen"]
    dd = pd.read_csv(os.path.join(results_dir, "deepdrid_predictions.csv"))
    dd = dd[dd.grade <= 4]
    cal, val = dd[dd.part == "training"], dd[dd.part == "validation"]
    assert not set(cal.patient_id) & set(val.patient_id), "calibration and evaluation patients must differ"
    out = dict(note="Added after seeing the external result; the frozen results stay the main results.",
               target_sensitivity=fr["target_sensitivity"], evaluation_photos=len(val),
               evaluation_patients=int(val.patient_id.nunique()), calibration_pool_patients=int(cal.patient_id.nunique()),
               draws=draws, frozen=workflow(val, fr["ref_threshold"], fr["poor_threshold"]), samples=[])
    rng = np.random.default_rng(seed)
    pats = cal.patient_id.unique()
    for n in SAMPLE_PATIENTS:
        if n > len(pats):                            # cannot draw more patients than the pool has
            continue
        rows, tries = [], 0
        while len(rows) < draws and tries < draws * 5:
            tries += 1
            sub = cal[cal.patient_id.isin(rng.choice(pats, n, replace=False))]
            if (sub.grade >= 2).sum() < 3:          # a clinic sample with almost no disease cannot set a cut-off
                continue
            t = ds.threshold_for_sensitivity((sub.grade >= 2).astype(int).values, sub.p_referable.values, fr["target_sensitivity"])
            rows.append(workflow(val, t, fr["poor_threshold"]))
        r = pd.DataFrame(rows)
        q = lambda c: {k: float(v) for k, v in zip(("p10", "median", "p90"), r[c].quantile([0.1, 0.5, 0.9]))}
        out["samples"].append(dict(patients=n, photos_median=int(round(cal.groupby("patient_id").size().median() * n)),
                                   draws=len(r), skipped_too_little_disease=tries - len(r),
                                   sensitivity=q("sensitivity"), specificity=q("specificity"), threshold=q("threshold"),
                                   referable_given_all_clear=q("referable_given_all_clear"),
                                   sight_threatening_given_all_clear=q("sight_threatening_given_all_clear"),
                                   REFER=q("REFER"), RETAKE=q("RETAKE"), RESCREEN=q("RESCREEN")))
    out["all_calibration_photos"] = workflow(
        val, ds.threshold_for_sensitivity((cal.grade >= 2).astype(int).values, cal.p_referable.values, fr["target_sensitivity"]),
        fr["poor_threshold"])
    json.dump(out, open(os.path.join(results_dir, "local_calibration.json"), "w"), indent=2)
    figure(out, os.path.join(results_dir, "local_calibration.png"))
    return out


def figure(out, path):
    from matplotlib.figure import Figure          # no pyplot: leaves the caller's backend (e.g. a notebook) alone
    s = out["samples"]
    x = [f"{d['patients']} patients\n(~{d['photos_median']} photos)" for d in s]
    fig = Figure(figsize=(11, 4.2), dpi=150)
    axes = fig.subplots(1, 2)
    for ax, key, title in ((axes[0], "sensitivity", "Sensitivity on 100 other patients"),
                           (axes[1], "specificity", "Specificity on 100 other patients")):
        med = [100 * d[key]["median"] for d in s]
        lo = [100 * (d[key]["median"] - d[key]["p10"]) for d in s]
        hi = [100 * (d[key]["p90"] - d[key]["median"]) for d in s]
        ax.errorbar(range(len(s)), med, yerr=[lo, hi], fmt="o", color="#2f6fdb", ms=8, capsize=5, lw=2,
                    label="cut-off re-set on a local sample\n(median, 10th to 90th percentile)")
        ax.axhline(100 * out["frozen"][key], color="#6e7781", ls="--", lw=1.5, label="frozen cut-off (set on Indian photos)")
        ax.set_xticks(range(len(s)), x, fontsize=9)
        ax.set_ylim(0, 100)
        ax.set_ylabel("%", color="#4b5563")
        ax.set_title(title, loc="left", fontsize=11)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#e1e4e8")
        ax.set_axisbelow(True)
    axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    fig.suptitle("DeepDRiD: re-setting the referral cut-off on labelled photos from the new hospitals "
                 "(added after seeing the external result)", fontsize=10, x=0.01, ha="left", color="#4b5563")
    fig.tight_layout()
    fig.savefig(path)


if __name__ == "__main__":
    a = argparse.ArgumentParser()
    a.add_argument("--results", default="results")
    a.add_argument("--draws", type=int, default=500)
    x = a.parse_args()
    o = run(x.results, x.draws)
    f = o["frozen"]
    print(f"frozen cut-off: sensitivity {f['sensitivity']:.1%}, specificity {f['specificity']:.1%}, "
          f"referable all-clears {f['referable_given_all_clear']}")
    for d in o["samples"]:
        print(f"{d['patients']:>3} patients: sensitivity {d['sensitivity']['median']:.1%} "
              f"({d['sensitivity']['p10']:.1%} to {d['sensitivity']['p90']:.1%}), specificity {d['specificity']['median']:.1%} "
              f"({d['specificity']['p10']:.1%} to {d['specificity']['p90']:.1%}), referable all-clears "
              f"{d['referable_given_all_clear']['median']:.0f}")


In [ ]:
%%writefile two_photo.py
"""Two photos per eye: does referring when EITHER photo of an eye is suspicious recover missed referrals?

Added AFTER we saw the external result, like local_calibration.py. Nothing is re-trained or re-tuned: the frozen
model, the frozen referral cut-off and the frozen quality check are used exactly as they are. DeepDRiD takes two
photos of every eye (file names like 12_l1 and 12_l2), as the commercial systems do (LumineticsCore and EyeArt
both ask for two photos per eye), so the saved per-photo predictions are enough to answer it. No GPU, no photos.

Rules (chosen to keep Roshni's safety rule: a poor photo never blocks a referral):
  * referral:  an eye is referred if either of its two photos scores at or above the frozen cut-off;
  * workflow:  REFER if either photo says REFER; otherwise RETAKE if either says RETAKE; otherwise RESCREEN.
  * truth:     an eye is referable if its expert grade is 2 or worse (the higher grade if its two photos differ).

One patient (4 photos) whose file names do not pair into two photos per eye is left out of both columns.
Units change: results are per EYE (two photos), not per photo, so they are not directly comparable with the
per-photo results; both are reported. Confidence intervals resample whole patients (each has two eyes).

Writes results/two_photo.json.
Run:  python two_photo.py [--results results]
"""
import argparse
import json
import os

import numpy as np
import pandas as pd

import drscreen as ds


def per_eye(d, ref_thr, poor_thr):
    d = d.copy()
    d["eye"] = d.image_id.str.extract(r"_([lr])\d")[0]
    d["dec"] = ds.decide(d.p_referable.values, d.p_poor.values, ref_thr, poor_thr)
    rank = {"RESCREEN": 0, "RETAKE": 1, "REFER": 2}
    d["rank"] = d.dec.map(rank)
    e = d.groupby(["patient_id", "eye"]).agg(grade=("grade", "max"), p_max=("p_referable", "max"),
                                             rank=("rank", "max"), photos=("image_id", "count")).reset_index()
    e["dec"] = e["rank"].map({v: k for k, v in rank.items()})
    return d, e


def binary(y, ref):
    y, ref = np.asarray(y, bool), np.asarray(ref, bool)
    tp, fn, tn, fp = int((ref & y).sum()), int((~ref & y).sum()), int((~ref & ~y).sum()), int((ref & ~y).sum())
    return dict(n=int(len(y)), positives=int(y.sum()), sensitivity=tp / (tp + fn), specificity=tn / (tn + fp),
                missed=fn, false_alarms=fp)


def patient_ci(e, ref_thr, n=2000, seed=0):
    """Percentile intervals for eye-level sensitivity and specificity, resampling whole patients."""
    rng = np.random.default_rng(seed)
    pats = e.patient_id.unique()
    groups = {p: g for p, g in e.groupby("patient_id")}
    sens, spec = [], []
    for _ in range(n):
        s = pd.concat([groups[p] for p in rng.choice(pats, len(pats), replace=True)])
        r = binary(s.grade >= 2, s.p_max >= ref_thr)
        sens.append(r["sensitivity"]); spec.append(r["specificity"])
    q = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
    return q(sens), q(spec)


def workflow(units, sev_col="grade"):
    y, sev = (units[sev_col] >= 2).values, (units[sev_col] >= 3).values
    dec = units.dec.values
    return dict(REFER=int((dec == "REFER").sum()), RETAKE=int((dec == "RETAKE").sum()),
                RESCREEN=int((dec == "RESCREEN").sum()),
                referable_given_all_clear=int(((dec == "RESCREEN") & y).sum()),
                sight_threatening_given_all_clear=int(((dec == "RESCREEN") & sev).sum()))


def run(results_dir="results"):
    res = json.load(open(os.path.join(results_dir, "results.json")))
    fr = res["frozen"]
    ref_thr, poor_thr = fr["ref_threshold"], fr["poor_threshold"]
    d = pd.read_csv(os.path.join(results_dir, "deepdrid_predictions.csv"))
    photos, eyes = per_eye(d, ref_thr, poor_thr)
    odd = eyes[eyes.photos != 2]             # one DeepDRiD patient's file names do not split into 2 + 2 photos
    skipped = sorted(set(odd.patient_id))
    eyes = eyes[~eyes.patient_id.isin(skipped)]
    photos = photos[~photos.patient_id.isin(skipped)]
    out = dict(rule="refer if either photo of the eye scores at or above the frozen cut-off; "
                    "workflow takes the most cautious of the two decisions",
               post_hoc=True, ref_threshold=ref_thr, poor_threshold=poor_thr,
               grades_differ_within_eye=int((photos.groupby(["patient_id", "eye"]).grade.nunique() > 1).sum()),
               patients_skipped=[int(x) for x in skipped], photos_skipped=int(len(d) - len(photos)))
    for name, part in (("all", None), ("validation", "validation")):
        p = photos if part is None else photos[photos.part == part]
        e = eyes[eyes.patient_id.isin(p.patient_id.unique())]
        one = binary(p.grade >= 2, p.p_referable >= ref_thr)
        two = binary(e.grade >= 2, e.p_max >= ref_thr)
        two["sensitivity_ci"], two["specificity_ci"] = patient_ci(e, ref_thr)
        out[name] = dict(per_photo=one, per_eye_two_photos=two, patients=int(e.patient_id.nunique()))
        if part == "validation":
            out[name]["workflow_per_photo"] = workflow(p)
            out[name]["workflow_per_eye_two_photos"] = workflow(e)
    with open(os.path.join(results_dir, "two_photo.json"), "w") as f:
        json.dump(out, f, indent=2)
    return out


if __name__ == "__main__":
    a = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    a.add_argument("--results", default="results")
    o = run(a.parse_args().results)
    for k in ("all", "validation"):
        one, two = o[k]["per_photo"], o[k]["per_eye_two_photos"]
        print(f"{k:10s} one photo: sens {100*one['sensitivity']:.1f}% spec {100*one['specificity']:.1f}% (n={one['n']} photos) | "
              f"two photos per eye: sens {100*two['sensitivity']:.1f}% "
              f"({100*two['sensitivity_ci'][0]:.1f} to {100*two['sensitivity_ci'][1]:.1f}) "
              f"spec {100*two['specificity']:.1f}% (n={two['n']} eyes)")
    print("validation workflow, per photo:", o["validation"]["workflow_per_photo"])
    print("validation workflow, per eye:  ", o["validation"]["workflow_per_eye_two_photos"])


In [ ]:
%%writefile web_index.html
<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Roshni: diabetic eye screening that runs on your device</title>
<link rel="manifest" href="manifest.json">
<meta name="theme-color" content="#12232e">
<link rel="icon" href="icons/icon-192.png">
<style>
  /* Clinical decision-support look. Colour carries meaning only (red refer, amber retake, teal rescreen).
     No web fonts or outside files, so the page keeps working offline. */
  :root {
    --bg:#faf8f4; --paper:#ffffff; --tint:#f2eee6; --line:#e4ded3; --line2:#d6cfc2;
    --ink:#12232e; --ink2:#3e4c56; --muted:#5f6b74; --faint:#5f6b74;   /* small grey text kept at 4.5:1 contrast or more */
    --teal:#1f6f6b; --teal2:#2a8a84; --tealbg:#e8f2f0;
    --refer:#b4382c; --referbg:#fbeeec; --retake:#a8650f; --retakebg:#fbf2e4; --rescreen:#1f6f6b; --rescreenbg:#e8f2f0;
    --r:16px; --shadow:0 1px 2px rgba(18,35,46,.04), 0 18px 40px -24px rgba(18,35,46,.22);
  }
  * { box-sizing:border-box; }
  html { scroll-behavior:smooth; }
  body { margin:0; background:var(--bg); color:var(--ink);
    font:15.5px/1.6 ui-sans-serif,-apple-system,BlinkMacSystemFont,"SF Pro Text","Inter","Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif;
    -webkit-font-smoothing:antialiased; text-rendering:optimizeLegibility; }
  html[lang="ur"] body { font-family:"Noto Naskh Arabic","Geeza Pro","Segoe UI",Tahoma,ui-sans-serif,sans-serif; }
  html[lang="ur"] .longtext { font-family:"Noto Nastaliq Urdu","Jameel Noori Nastaleeq","Urdu Typesetting","Noto Naskh Arabic",serif; line-height:2.15; }
  a { color:var(--teal); }
  button { font:inherit; }
  :focus-visible { outline:2px solid var(--teal); outline-offset:2px; }
  .wrap { max-width:1160px; margin:0 auto; padding:0 28px; }
  .machine { display:none; }   /* English originals for the automated tests and the video script */

  /* top bar */
  .top { border-bottom:1px solid var(--line); background:var(--bg); }
  .top .wrap { display:flex; align-items:center; justify-content:space-between; gap:16px; height:64px; }
  .brand { display:flex; align-items:center; gap:12px; min-width:0; }
  .mark { width:26px; height:26px; border-radius:50%; border:1.5px solid var(--teal); position:relative; flex:none; }
  .mark::after { content:""; position:absolute; inset:7px; border-radius:50%; background:var(--teal); }
  .brand b { font-size:1.02rem; letter-spacing:.01em; font-weight:650; }
  .brand small { display:block; color:var(--muted); font-size:.75rem; line-height:1.2; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
  .seg { display:inline-flex; border:1px solid var(--line2); border-radius:10px; padding:3px; background:var(--paper); gap:2px; }
  .seg button { border:0; background:transparent; color:var(--ink2); padding:6px 12px; border-radius:7px; cursor:pointer; font-size:.84rem; white-space:nowrap; }
  .seg button[aria-pressed="true"] { background:var(--ink); color:#fff; }
  .seg button:hover:not([aria-pressed="true"]) { background:var(--tint); }
  .langwrap { display:flex; align-items:center; gap:10px; }
  .langwrap > span { font-size:.72rem; text-transform:uppercase; letter-spacing:.12em; color:var(--muted); }

  /* intro */
  body::before { content:""; display:block; height:3px; background:linear-gradient(90deg,var(--teal),var(--ink)); }
  .intro { display:grid; grid-template-columns:minmax(0,1.25fr) minmax(0,.75fr); gap:48px; padding:64px 0 40px; align-items:center; }
  .kicker { font-size:.74rem; text-transform:uppercase; letter-spacing:.18em; color:var(--teal); font-weight:650; margin:0 0 16px; display:flex; align-items:center; gap:10px; }
  .kicker::before { content:""; width:22px; height:1.5px; background:var(--teal); }
  html[lang="ur"] .kicker { letter-spacing:0; font-size:.9rem; }
  h1 { font-size:clamp(2.3rem,5vw,3.9rem); line-height:1.02; letter-spacing:-.035em; font-weight:300; margin:0 0 20px; color:var(--ink); }
  html[lang="ur"] h1 { font-weight:500; letter-spacing:0; line-height:1.5; }
  .lede { color:var(--ink2); font-size:1.08rem; margin:0; max-width:600px; }
  .cta { display:flex; flex-wrap:wrap; gap:10px; margin:26px 0 18px; }
  .cta a { text-decoration:none; }
  .disc { font-size:.82rem; color:#7a4a0c; background:var(--retakebg); border:1px solid #efdcbc; border-radius:10px; padding:9px 12px; max-width:600px; }
  .hero-vis { margin:0; display:grid; justify-items:center; gap:14px; }
  .orbit { position:relative; width:min(320px,70vw); aspect-ratio:1; border-radius:50%; display:grid; place-items:center;
    background:radial-gradient(circle, var(--paper) 58%, var(--tealbg) 59% 100%); box-shadow:0 0 0 1px var(--line), var(--shadow); }
  .orbit::before, .orbit::after { content:""; position:absolute; inset:-18px; border-radius:50%; border:1px solid var(--line); }
  .orbit::after { inset:-38px; border-style:dashed; border-color:var(--line2); opacity:.6; }
  .orbit img { width:78%; height:78%; border-radius:50%; object-fit:cover; display:block; background:#000; box-shadow:0 18px 40px -20px rgba(18,35,46,.6); }
  .hero-vis figcaption { font-size:.72rem; color:var(--faint); margin-top:26px; }
  .hl { display:grid; grid-template-columns:repeat(4,1fr); gap:0; margin:0 0 26px; border-top:1px solid var(--line2); border-bottom:1px solid var(--line2); }
  .hl > div { padding:18px 20px 18px 0; border-inline-end:1px solid var(--line); padding-inline-start:20px; }
  .hl > div:first-child { padding-inline-start:0; }
  .hl > div:last-child { border-inline-end:0; }
  .hl b { display:block; font-size:.95rem; font-weight:650; color:var(--ink); margin-bottom:3px; }
  .hl span { font-size:.82rem; color:var(--muted); line-height:1.45; display:block; }

  /* workflow stages */
  .steps { list-style:none; margin:0 0 22px; padding:0; display:grid; grid-template-columns:repeat(4,1fr); border:1px solid var(--line); border-radius:var(--r); background:var(--paper); overflow:hidden; box-shadow:var(--shadow); }
  .steps li { position:relative; padding:16px 18px 16px; display:flex; gap:12px; align-items:flex-start; border-inline-end:1px solid var(--line); transition:background .3s; }
  .steps li:last-child { border-inline-end:0; }
  .steps .num { width:26px; height:26px; border-radius:50%; border:1px solid var(--line2); color:var(--muted); font-size:.78rem; font-weight:600; display:grid; place-items:center; flex:none; font-variant-numeric:tabular-nums; transition:all .3s; }
  .steps b { display:block; font-size:.9rem; font-weight:600; color:var(--ink2); line-height:1.3; }
  .steps small { display:block; color:var(--faint); font-size:.76rem; margin-top:2px; line-height:1.3; }
  .steps li::after { content:""; position:absolute; inset-inline:0; bottom:0; height:2px; background:transparent; transition:background .3s; }
  /* idle: step 1 is current */
  body[data-stage="idle"] .steps li[data-step="0"] .num, body:not([data-stage]) .steps li[data-step="0"] .num { border-color:var(--teal); color:var(--teal); }
  body[data-stage="idle"] .steps li[data-step="0"]::after { background:var(--teal); }
  body[data-stage="idle"] .steps li[data-step="0"] b { color:var(--ink); }
  /* working: 1 done, 2 and 3 running (they run together, in one pass of the model) */
  body[data-stage="working"] .steps li[data-step="0"] .num,
  body[data-stage="done"] .steps li:not([data-step="3"]) .num { background:var(--teal); border-color:var(--teal); color:#fff; }
  body[data-stage="working"] .steps li[data-step="1"], body[data-stage="working"] .steps li[data-step="2"] { background:var(--tealbg); }
  body[data-stage="working"] .steps li[data-step="1"] .num, body[data-stage="working"] .steps li[data-step="2"] .num { border-color:var(--teal); color:var(--teal); animation:pulse 1s ease-in-out infinite; }
  body[data-stage="working"] .steps li[data-step="1"]::after, body[data-stage="working"] .steps li[data-step="2"]::after { background:var(--teal); }
  body[data-stage="done"] .steps li[data-step="3"] .num { border-color:var(--ink); background:var(--ink); color:#fff; }
  body[data-stage="done"] .steps li[data-step="3"]::after { background:var(--ink); }
  body[data-stage="done"] .steps b, body[data-stage="working"] .steps li:not([data-step="3"]) b { color:var(--ink); }
  @keyframes pulse { 50% { opacity:.45; } }

  /* tool */
  .tool { display:grid; grid-template-columns:minmax(0,.9fr) minmax(0,1.1fr); gap:22px; align-items:start; scroll-margin-top:84px; }
  .panel { background:var(--paper); border:1px solid var(--line); border-radius:var(--r); padding:24px; box-shadow:var(--shadow); }
  .label { font-size:.72rem; text-transform:uppercase; letter-spacing:.14em; color:var(--muted); font-weight:600; margin:0 0 12px; }
  html[lang="ur"] .label { letter-spacing:0; font-size:.86rem; }
  .label.gap { margin-top:26px; }

  #drop { position:relative; display:block; border:1.5px dashed var(--line2); border-radius:12px; background:var(--tint); cursor:pointer; transition:border-color .2s, background .2s; overflow:hidden; }
  #drop:hover, #drop.over, #drop:focus-within { border-color:var(--teal); background:#eef3f2; }
  #drop:has(input:disabled) { cursor:wait; opacity:.6; }   /* greyed out until the model has loaded */
  .dz { display:grid; justify-items:center; gap:8px; padding:34px 18px; text-align:center; }
  .dz .ring { width:54px; height:54px; border-radius:50%; border:1.5px solid var(--teal); display:grid; place-items:center; }
  .dz .ring::after { content:""; width:16px; height:16px; border-radius:50%; background:var(--teal); opacity:.85; }
  .dz strong { font-size:1rem; font-weight:600; color:var(--ink); }
  .dz small { color:var(--muted); font-size:.82rem; max-width:340px; }
  .btn { display:inline-flex; align-items:center; justify-content:center; gap:8px; border-radius:10px; padding:10px 16px; font-size:.9rem; font-weight:600; cursor:pointer; border:1px solid transparent; transition:background .2s, border-color .2s; }
  .btn.primary { background:var(--ink); color:#fff; }
  .btn.primary:hover { background:var(--teal); }
  .btn.ghost { background:var(--paper); color:var(--ink); border-color:var(--line2); }
  .btn.ghost:hover { border-color:var(--ink2); }
  #file { position:absolute; width:1px; height:1px; opacity:0; overflow:hidden; }
  .pv { display:none; grid-template-columns:120px 1fr; gap:16px; align-items:center; padding:16px; }
  .pv img { width:120px; height:120px; object-fit:cover; border-radius:50%; background:#000; box-shadow:0 0 0 1px var(--line2), 0 0 0 6px var(--paper), 0 0 0 7px var(--line); }
  .pv .cap2 { font-size:.72rem; text-transform:uppercase; letter-spacing:.12em; color:var(--muted); margin-bottom:8px; }
  body.has-image .dz { display:none; }
  body.has-image .pv { display:grid; }

  .stat { display:flex; align-items:center; gap:9px; margin:14px 2px 2px; font-size:.86rem; color:var(--ink2); min-height:1.5em; }
  .stat::before { content:""; width:8px; height:8px; border-radius:50%; background:var(--faint); flex:none; }
  #statusText[data-kind="ready"]::before, #statusText[data-kind="done"]::before { background:var(--teal); }
  #statusText[data-kind="working"]::before, #statusText[data-kind="loading"]::before { background:var(--retake); animation:pulse 1s ease-in-out infinite; }
  #statusText[data-kind^="err"] { color:var(--refer); }
  #statusText[data-kind^="err"]::before { background:var(--refer); }
  #offline { font-size:.78rem; color:var(--teal); margin:2px 2px 0; min-height:1.2em; }

  #examples { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:8px; }
  @media (max-width: 420px) { #examples { grid-template-columns:1fr; } }
  .ex { display:grid; grid-template-columns:46px 1fr; gap:10px; align-items:center; border:1px solid var(--line); background:var(--paper); border-radius:10px; padding:7px 10px 7px 7px; cursor:pointer; text-align:start; color:inherit; transition:border-color .2s, box-shadow .2s; }
  .ex:hover { border-color:var(--line2); box-shadow:var(--shadow); }
  .ex.on { border-color:var(--ink); box-shadow:0 0 0 1px var(--ink); }
  .ex img { width:46px; height:46px; object-fit:cover; border-radius:50%; display:block; background:#000; }
  .ex span { display:block; font-size:.74rem; color:var(--ink2); line-height:1.35; }
  #excredit { font-size:.72rem; color:var(--faint); margin-top:10px; }
  #langs { display:flex; flex-wrap:wrap; gap:8px; }
  #langs label { display:inline-flex; align-items:center; gap:8px; font-size:.86rem; border:1px solid var(--line2); border-radius:999px; padding:6px 12px; cursor:pointer; background:var(--paper); }
  #langs label:has(input:checked) { border-color:var(--teal); background:var(--tealbg); }
  #langs input { accent-color:var(--teal); margin:0; }

  /* result */
  .empty { border:1px dashed var(--line2); border-radius:12px; padding:44px 20px; text-align:center; color:var(--muted); font-size:.92rem; background:var(--tint); }
  body:has(#card:not([hidden])) .empty { display:none; }
  #card { --c:var(--teal); --cb:var(--tealbg); border:1px solid var(--line); border-radius:12px; overflow:hidden; background:var(--paper); animation:rise .35s ease both; }
  #card[data-decision="REFER"] { --c:var(--refer); --cb:var(--referbg); }
  #card[data-decision="RETAKE"] { --c:var(--retake); --cb:var(--retakebg); }
  #card[data-decision="RESCREEN"] { --c:var(--rescreen); --cb:var(--rescreenbg); }
  .rtop { background:var(--cb); border-bottom:1px solid var(--line); padding:18px 20px 16px; border-top:3px solid var(--c); }
  .kick2 { font-size:.7rem; text-transform:uppercase; letter-spacing:.14em; color:var(--muted); font-weight:600; }
  html[lang="ur"] .kick2 { letter-spacing:0; font-size:.82rem; }
  #rTag { display:inline-block; margin-top:10px; font-size:.74rem; font-weight:700; letter-spacing:.12em; color:#fff; background:var(--c); border-radius:6px; padding:4px 9px; }
  html[lang="ur"] #rTag { letter-spacing:0; }
  #rHead { font-size:clamp(1.3rem,2.3vw,1.7rem); line-height:1.25; font-weight:600; letter-spacing:-.015em; color:var(--ink); margin:10px 0 6px; }
  html[lang="ur"] #rHead { letter-spacing:0; line-height:1.7; }
  #rBody { color:var(--ink2); margin:0; }
  .rbody { padding:16px 20px 18px; }
  #rNext { margin:0; padding:12px 14px; border-radius:10px; border:1px solid var(--line); border-inline-start:3px solid var(--c); font-weight:600; color:var(--ink); background:var(--paper); }
  .tech { margin-top:14px; font-size:.8rem; color:var(--muted); font-variant-numeric:tabular-nums; display:grid; gap:2px; }
  .tech span { font-family:ui-monospace,"SF Mono",Menlo,Consolas,monospace; font-size:.76rem; }
  html[lang="ur"] .tech span { font-family:inherit; font-size:.82rem; }
  @keyframes rise { from { opacity:0; transform:translateY(6px); } to { opacity:1; transform:none; } }
  #notebox, #heatbox { animation:rise .4s ease both; }
  .n { margin:10px 0; padding:14px 16px; border-radius:10px; border:1px solid var(--line); background:var(--tint); color:var(--ink); line-height:1.65; }
  .n b { font-weight:600; }
  .n small { color:var(--muted); display:inline-block; margin-top:4px; }
  .n[dir="rtl"] { font-family:"Noto Nastaliq Urdu","Jameel Noori Nastaleeq","Urdu Typesetting","Noto Naskh Arabic",serif; font-size:1.06rem; line-height:2.2; }
  .imgs { display:grid; grid-template-columns:1fr 1fr; gap:14px; }
  .imgs > div { border:1px solid var(--line); border-radius:12px; padding:14px; background:var(--tint); }
  canvas { width:100%; max-width:384px; display:block; margin:0 auto; border-radius:50%; background:#000; }
  .cap { font-size:.76rem; color:var(--muted); margin-top:10px; text-align:center; }
  .why { font-size:.8rem; color:var(--muted); margin:10px 0 0; }
  .g { display:grid; grid-template-columns:130px 1fr 42px; align-items:center; gap:10px; font-size:.84rem; margin:6px 0; color:var(--ink2); font-variant-numeric:tabular-nums; }
  .bar { height:6px; background:var(--tint); border:1px solid var(--line); border-radius:999px; overflow:hidden; display:block; }
  .bar span { display:block; height:100%; background:var(--ink2); }
  .pct { text-align:end; color:var(--muted); }
  #newBtn { display:none; margin-top:20px; width:100%; }
  body[data-stage="done"] #newBtn { display:flex; }

  /* evidence */
  footer { padding:34px 0 60px; }
  .ev { display:grid; grid-template-columns:220px 1fr; gap:28px; border-top:1px solid var(--line2); padding-top:24px; }
  .ev p { margin:0 0 8px; color:var(--ink2); font-size:.88rem; }
  .ev .lim { color:var(--ink); }
  .fine { color:var(--faint); font-size:.74rem; margin-top:28px; }

  @media (max-width: 900px) {
    .intro { grid-template-columns:1fr; gap:30px; padding:40px 0 30px; }
    .hero-vis { order:-1; }
    .orbit { width:min(220px,60vw); }
    .hl { grid-template-columns:1fr 1fr; }
    .hl > div:nth-child(2) { border-inline-end:0; }
    .hl > div:nth-child(3) { padding-inline-start:0; }
    .hl > div:nth-child(-n+2) { border-bottom:1px solid var(--line); }
    .tool { grid-template-columns:1fr; }
    .steps { grid-template-columns:1fr 1fr; }
    .steps li:nth-child(2) { border-inline-end:0; }
    .steps li:nth-child(-n+2) { border-bottom:1px solid var(--line); }
    .ev { grid-template-columns:1fr; gap:10px; }
  }
  @media (max-width: 560px) {
    .wrap { padding:0 16px; }
    .top .wrap { height:auto; padding-top:10px; padding-bottom:10px; flex-wrap:wrap; }
    .langwrap > span { display:none; }
    .panel { padding:18px; }
    .steps li { padding:12px; gap:9px; }
    .steps small { display:none; }
    .imgs { gap:8px; } .imgs > div { padding:8px; }
    .g { grid-template-columns:96px 1fr 36px; }
    .pv { grid-template-columns:88px 1fr; } .pv img { width:88px; height:88px; }
  }
  @media (prefers-reduced-motion: reduce) { * { animation:none !important; transition:none !important; } html { scroll-behavior:auto; } }
</style>
</head>
<body data-stage="idle">
<header class="top">
  <div class="wrap">
    <div class="brand"><span class="mark" aria-hidden="true"></span><div><b>Roshni</b><small data-i18n="brandSub">Diabetic eye screening · research prototype</small></div></div>
    <div class="langwrap"><span data-i18n="langLabel">Language</span><div id="uilang" class="seg" role="group" aria-label="Language"></div></div>
  </div>
</header>

<div class="wrap">
  <section class="intro">
    <div>
      <p class="kicker" data-i18n="kicker">Primary care · Pakistan</p>
      <h1><span data-i18n="title">Retinal image assessment</span></h1>
      <p class="lede longtext" data-i18n="lede">For a primary-care diabetes clinic without an eye specialist. Upload one retinal photograph; Roshni suggests one next step for the doctor to confirm.</p>
      <div class="cta"><a class="btn primary" href="#tool" data-i18n="ctaStart">Start an assessment</a><a class="btn ghost" href="#evidence" data-i18n="ctaEvidence">Evidence and limits</a></div>
      <div class="disc" role="note" data-i18n="disclaimer">Research prototype for a student hackathon. Not medical advice, not a diagnosis, and not tested on patients in Pakistan.</div>
    </div>
    <figure class="hero-vis">
      <div class="orbit"><img src="examples/410_l1.jpg" alt="" width="320" height="320"></div>
      <figcaption data-i18n="heroCap">Example retinal photograph · DeepDRiD, CC BY-SA 4.0</figcaption>
    </figure>
  </section>

  <section class="hl" aria-label="Highlights">
    <div><b data-i18n="h1t">On-device</b><span data-i18n="h1d">The photo is analysed in this browser and is not uploaded.</span></div>
    <div><b data-i18n="h2t">Offline-ready</b><span data-i18n="h2d">After the first visit, it works without internet.</span></div>
    <div><b data-i18n="h3t">Three languages</b><span data-i18n="h3d">English, Urdu and Uzbek interface; the Uzbek patient note is being checked.</span></div>
    <div><b data-i18n="h4t">Open code</b><span data-i18n="h4d">Code and results are public under the MIT licence.</span></div>
  </section>

  <ol class="steps" aria-label="Workflow">
    <li data-step="0"><span class="num">1</span><span><b></b><small></small></span></li>
    <li data-step="1"><span class="num">2</span><span><b></b><small></small></span></li>
    <li data-step="2"><span class="num">3</span><span><b></b><small></small></span></li>
    <li data-step="3"><span class="num">4</span><span><b></b><small></small></span></li>
  </ol>

  <main class="tool" id="tool">
    <section class="panel" aria-labelledby="upTitle">
      <p class="label" id="upTitle" data-i18n="uploadTitle">Upload a retinal photograph</p>
      <label id="drop">
        <div class="dz">
          <span class="ring" aria-hidden="true"></span>
          <strong data-i18n="uploadTitle">Upload a retinal photograph</strong>
          <small data-i18n="uploadHint">Fundus photo from the clinic camera or your files · JPG or PNG · drag and drop or click</small>
          <span class="btn primary" data-i18n="uploadBtn">Choose image</span>
        </div>
        <div class="pv">
          <img id="preview" alt="">
          <div><div class="cap2" data-i18n="previewCap">Selected image</div><span class="btn ghost" data-i18n="replaceBtn">Replace image</span></div>
        </div>
        <input id="file" type="file" accept="image/*" disabled>
      </label>
      <div class="machine" aria-hidden="true"><div id="status">Loading…</div></div>
      <div id="statusText" class="stat" role="status" aria-live="polite"></div>
      <div id="offline"></div>

      <p class="label gap" data-i18n="examplesTitle">Or try an example photo</p>
      <div id="examples"></div>
      <div id="excredit"></div>

      <p class="label gap" data-i18n="noteLangs">Languages on the patient note</p>
      <div id="langs"></div>
    </section>

    <section class="panel" aria-labelledby="resTitle">
      <p class="label" id="resTitle" data-i18n="resultTitle">Result</p>
      <div class="empty" data-i18n="empty">Upload an image to begin. The result will appear here.</div>
      <div id="card" hidden aria-live="polite">
        <div class="machine" aria-hidden="true"><div id="decision"></div><div id="scores"></div><div id="timing"></div></div>
        <div class="rtop">
          <div class="kick2" data-i18n="kicker2">Suggested next step · for the doctor to confirm</div>
          <span id="rTag"></span>
          <div id="rHead"></div>
          <p id="rBody" class="longtext"></p>
        </div>
        <div class="rbody">
          <p id="rNext" class="longtext"></p>
          <div class="tech"><span id="rScores"></span><span id="rTiming"></span></div>
        </div>
      </div>
      <div id="notebox" hidden>
        <p class="label gap" data-i18n="noteTitle">Note for the doctor and patient</p>
        <div id="note"></div>
      </div>
      <div id="heatbox" hidden>
        <p class="label gap" data-i18n="imagesTitle">Original photograph and heat map</p>
        <div class="imgs">
          <div><canvas id="photo"></canvas><div class="cap" data-i18n="photoCap">Photo as the model sees it</div></div>
          <div><canvas id="heat"></canvas><div class="cap" data-i18n="heatCap">Heat map (Grad-CAM) for referable disease</div></div>
        </div>
        <p class="why" data-i18n="heatWhy">The heat map shows which parts of the photo drove the score. It is not proof that disease is present or absent.</p>
        <p class="label gap" data-i18n="gradesTitle">Estimated severity (model output)</p>
        <div id="grades"></div>
      </div>
      <button id="newBtn" class="btn ghost" type="button" data-i18n="newBtn">Start new assessment</button>
    </section>
  </main>

  <footer id="evidence">
    <div class="ev">
      <p class="label" data-i18n="evidenceTitle">Evidence and limits</p>
      <div>
        <p id="results"></p>
        <p id="thr"></p>
        <p class="lim" data-i18n="model">Model: EfficientNet-B0 fine-tuned on APTOS 2019 (India), tested on DeepDRiD (China). Not tested on Pakistani patients or low-cost cameras. Does not detect macular oedema.</p>
        <p><a id="repo" hidden data-i18n="repo">Code, results and limitations</a></p>
      </div>
    </div>
    <p class="fine" data-i18n="fine">Roshni · VERITAS AI Hackathon 2026 · Research prototype, not a medical device</p>
  </footer>
</div>
<script src="ort/ort.wasm.min.js"></script>
<script src="roshni.js"></script>
</body>
</html>


In [ ]:
%%writefile web_roshni.js
/* Roshni in the browser: the same frozen tool as the Python code, run with ONNX Runtime Web.
   The photo is processed on this device and never uploaded.
   Pure functions at the top are also tested in Node (tests/test_web.mjs). */

const R = {};

/* PIL "L" conversion, used to find the round fundus inside its black border. */
R.cropBox = function (rgba, w, h, tol) {
  let x0 = w, y0 = h, x1 = -1, y1 = -1, count = 0;
  for (let y = 0; y < h; y++) {
    for (let x = 0; x < w; x++) {
      const i = (y * w + x) * 4;
      const L = (rgba[i] * 19595 + rgba[i + 1] * 38470 + rgba[i + 2] * 7471 + 0x8000) >> 16;
      if (L > tol) {
        count++;
        if (x < x0) x0 = x;
        if (x > x1) x1 = x;
        if (y < y0) y0 = y;
        if (y > y1) y1 = y;
      }
    }
  }
  if (count < 0.05 * w * h) return { x: 0, y: 0, w: w, h: h };   // almost all dark: leave it to the quality check
  return { x: x0, y: y0, w: x1 - x0 + 1, h: y1 - y0 + 1 };
};

/* Resize exactly like Pillow's Image.resize(..., BILINEAR), which the Python tool uses:
   a triangle filter widened when shrinking, horizontal pass then vertical, 22-bit fixed point. */
R._coeffs = function (inSize, outSize) {
  const scale = inSize / outSize, fscale = Math.max(scale, 1), support = fscale, ss = 1 / fscale;
  const PREC = 22, list = [];
  for (let xx = 0; xx < outSize; xx++) {
    const center = (xx + 0.5) * scale;
    let xmin = Math.trunc(center - support + 0.5); if (xmin < 0) xmin = 0;
    let xmax = Math.trunc(center + support + 0.5); if (xmax > inSize) xmax = inSize;
    const n = xmax - xmin, k = new Float64Array(n);
    let ww = 0;
    for (let x = 0; x < n; x++) { const t = Math.abs((x + xmin - center + 0.5) * ss); k[x] = t < 1 ? 1 - t : 0; ww += k[x]; }
    const ki = new Int32Array(n);
    for (let x = 0; x < n; x++) { const v = (ww !== 0 ? k[x] / ww : k[x]) * (1 << PREC); ki[x] = Math.trunc(v < 0 ? v - 0.5 : v + 0.5); }
    list.push({ xmin: xmin, k: ki });
  }
  return list;
};

R.resizePIL = function (src, sw, sh, dw, dh) {
  const PREC = 22, half = 1 << (PREC - 1), div = 1 << PREC;
  const clip = v => (v < 0 ? 0 : v > 255 ? 255 : v);
  let w = sw, h = sh, buf = src;
  if (dw !== sw) {
    const cs = R._coeffs(sw, dw), out = new Uint8ClampedArray(dw * h * 4);
    for (let y = 0; y < h; y++) for (let xx = 0; xx < dw; xx++) {
      const { xmin, k } = cs[xx];
      let r = half, g = half, b = half;
      for (let x = 0; x < k.length; x++) { const i = (y * w + xmin + x) * 4; r += buf[i] * k[x]; g += buf[i + 1] * k[x]; b += buf[i + 2] * k[x]; }
      const o = (y * dw + xx) * 4;
      out[o] = clip(Math.floor(r / div)); out[o + 1] = clip(Math.floor(g / div)); out[o + 2] = clip(Math.floor(b / div)); out[o + 3] = 255;
    }
    buf = out; w = dw;
  }
  if (dh !== sh) {
    const cs = R._coeffs(sh, dh), out = new Uint8ClampedArray(w * dh * 4);
    for (let yy = 0; yy < dh; yy++) {
      const { xmin, k } = cs[yy];
      for (let x = 0; x < w; x++) {
        let r = half, g = half, b = half;
        for (let y = 0; y < k.length; y++) { const i = ((xmin + y) * w + x) * 4; r += buf[i] * k[y]; g += buf[i + 1] * k[y]; b += buf[i + 2] * k[y]; }
        const o = (yy * w + x) * 4;
        out[o] = clip(Math.floor(r / div)); out[o + 1] = clip(Math.floor(g / div)); out[o + 2] = clip(Math.floor(b / div)); out[o + 3] = 255;
      }
    }
    buf = out; h = dh;
  }
  return buf;
};

R.softmax = function (z) {
  const m = Math.max(...z);
  const e = z.map(v => Math.exp(v - m));
  const s = e.reduce((a, b) => a + b, 0);
  return e.map(v => v / s);
};

R.referableProb = function (P, from) {
  let s = 0;
  for (let k = from; k < P.length; k++) s += P[k];
  return s;
};

/* StandardScaler + logistic regression, exported from scikit-learn. */
R.poorProb = function (emb, gate) {
  if (!gate) return 0;
  let z = gate.intercept;
  for (let c = 0; c < emb.length; c++) z += ((emb[c] - gate.mean[c]) / gate.scale[c]) * gate.coef[c];
  return 1 / (1 + Math.exp(-z));
};

/* Refer whenever referable disease is suspected (a poor photo never blocks a referral);
   otherwise a poor photo gets RETAKE instead of an all-clear. */
R.decide = function (pRef, pPoor, refThr, poorThr) {
  if (pRef >= refThr) return "REFER";
  if (pPoor >= poorThr) return "RETAKE";
  return "RESCREEN";
};

/* Grad-CAM for the referable score, computed exactly without gradients: the network ends in
   global average pooling and one linear layer, so the gradient at the last feature map is the same at
   every position: sum_k (dscore/dlogit_k) * W[k][c]. Matches drscreen.gradcam up to a positive scale. */
R.heatmap = function (fmap, C, H, W, P, headW, from) {
  let sRef = 0;
  for (let k = from; k < P.length; k++) sRef += P[k];
  const g = P.map((p, k) => (k >= from ? p / sRef : 0) - p);
  const alpha = new Float32Array(C);
  for (let c = 0; c < C; c++) {
    let a = 0;
    for (let k = 0; k < P.length; k++) a += g[k] * headW[k][c];
    alpha[c] = a;
  }
  const cam = new Float32Array(H * W);
  for (let c = 0; c < C; c++) {
    const a = alpha[c], off = c * H * W;
    if (a === 0) continue;
    for (let i = 0; i < H * W; i++) cam[i] += a * fmap[off + i];
  }
  let mn = Infinity, mx = -Infinity;
  for (let i = 0; i < cam.length; i++) {
    cam[i] = Math.max(0, cam[i]);
    if (cam[i] < mn) mn = cam[i];
    if (cam[i] > mx) mx = cam[i];
  }
  for (let i = 0; i < cam.length; i++) cam[i] = (cam[i] - mn) / (mx - mn + 1e-8);
  return cam;
};

if (typeof module !== "undefined") module.exports = R;

/* ------------------------------------------------------------------ browser app
   The screening itself (prepare, run, the decision rule, Grad-CAM) is unchanged.
   What is new here is only how the page shows it: one interface language for the whole page
   (English, Urdu, Uzbek), an image preview, the workflow stages and a "new assessment" button.
   For the automated tests and the video script, #status, #decision, #scores and #timing still hold
   the original English text (hidden); the visible text is translated. */
if (typeof window !== "undefined") {
  const COLORS = { REFER: "#b42318", RETAKE: "#b54708", RESCREEN: "#067647" };
  const TITLES = { REFER: "Refer to an eye specialist", RETAKE: "Retake the photo", RESCREEN: "Rescreen in 12 months" };
  const STATUS_EN = {
    loading: "Loading the model onto this device…",
    ready: "Ready. Choose a retinal photo or an example.",
    working: "Screening on this device…",
    done: "Done. Nothing was uploaded.",
    errModel: "Could not load the model: ",
    errImage: "Could not open that image.",
    errRun: "Something went wrong: ",
  };

  /* Interface text. Urdu: to be checked by Zainab. Uzbek: to be checked by Laziza.
     The patient-note texts (REFER/RETAKE/RESCREEN/DISCLAIMER) still come from roshni_web.json for
     English and Urdu; the Uzbek note below is shown with "(translation being checked)" until approved. */
  const UI = {
    en: {
      dir: "ltr", name: "English",
      brandSub: "Diabetic eye screening · research prototype",
      langLabel: "Language",
      chipLocal: "Runs in this browser", chipPhoto: "Photo is not uploaded", chipProto: "Research prototype",
      kicker: "Primary care · Pakistan",
      title: "Retinal image assessment",
      lede: "For a primary-care diabetes clinic without an eye specialist. Upload one retinal photograph; Roshni suggests one next step for the doctor to confirm.",
      privacy: "The photo is processed inside this browser and is not uploaded. After the first visit, the tool also works without internet.",
      disclaimer: "Research prototype for a student hackathon. Not medical advice, not a diagnosis, and not tested on patients in Pakistan.",
      steps: ["Upload retinal image", "Check image quality", "Analyse", "View result"],
      stepHints: ["JPG or PNG fundus photo", "Is the photo clear enough?", "Referable-disease score", "Next step for the doctor"],
      uploadTitle: "Upload a retinal photograph",
      uploadHint: "Fundus photo from the clinic camera or your files · JPG or PNG · drag and drop or click",
      uploadBtn: "Choose image", replaceBtn: "Replace image", previewCap: "Selected image",
      examplesTitle: "Or try an example photo",
      examplesCredit: "Example photos: DeepDRiD (Liu et al., Patterns 2022), CC BY-SA 4.0. Captions give the experts' grade and photo-quality label, not the tool's answer.",
      exGrade: "Expert grade", exQuality: "photo quality", good: "good", poor: "poor",
      exMissed: "Shown on purpose: one of the referable photos Roshni misses",
      noteLangs: "Languages on the patient note", checking: "translation being checked",
      status: {
        loading: "Loading the model onto this device…", ready: "Ready. Upload a retinal (fundus) photo or choose an example. Other kinds of picture give meaningless results.",
        working: "Checking image quality and analysing on this device…", done: "Done. The photo was not uploaded.",
        errModel: "Could not load the model: ", errImage: "Could not open that image. Please choose a JPG or PNG retinal photo.", errRun: "Something went wrong: ",
      },
      offlineReady: "Saved on this device: works without internet from now on.",
      resultTitle: "Result", empty: "Upload an image to begin. The result will appear here.",
      kicker2: "Suggested next step · for the doctor to confirm",
      R: {
        REFER: { tag: "REFER", head: "Referable diabetic retinopathy suspected", body: "Moderate or worse diabetic retinopathy may be present. Clinician assessment is needed.", next: "Next step: refer to an eye specialist, after the doctor confirms." },
        RETAKE: { tag: "RETAKE", head: "The image needs to be captured again", body: "The photo is not clear enough to rule out disease.", next: "Next step: retake the photo. If it is still unclear, refer to an eye specialist." },
        RESCREEN: { tag: "RESCREEN", head: "Referable diabetic retinopathy not detected in this gradable image", body: "This is not a guarantee that the eye is healthy, and other eye diseases are not checked.", next: "Next step: continue routine eye check-ups as the doctor advises (usually every 12 months)." },
      },
      scores: (r, rt, p, pt) => `Referable-disease score ${r} (refer at ${rt} or above) · Poor-photo score ${p} (retake at ${pt} or above)`,
      timing: s => `Processed on this device in ${s} s.`,
      noteTitle: "Note for the doctor and patient",
      imagesTitle: "Original photograph and heat map",
      photoCap: "Photo as the model sees it", heatCap: "Heat map (Grad-CAM) for referable disease",
      heatWhy: "The heat map shows which parts of the photo drove the score. It is not proof that disease is present or absent.",
      gradesTitle: "Estimated severity (model output)",
      grades: ["No DR", "Mild", "Moderate", "Severe", "Proliferative"],
      newBtn: "Start new assessment",
      evidenceTitle: "Evidence and limits",
      results: (a, b) => `On ${a.n} unseen photos from India: sensitivity ${a.se}%, specificity ${a.sp}%. On ${b.n} photos from Chinese hospitals it never saw: sensitivity ${b.se}%, specificity ${b.sp}%.`,
      thr: (t, s) => `Referral threshold ${t}, frozen before testing (set for ${s}% sensitivity on validation photos).`,
      model: "Model: EfficientNet-B0 fine-tuned on APTOS 2019 (India), tested on DeepDRiD (China). Not tested on Pakistani patients or low-cost cameras. Does not detect macular oedema.",
      repo: "Code, results and limitations",
      ctaStart: "Start an assessment", ctaEvidence: "Evidence and limits",
      heroCap: "Example retinal photograph · DeepDRiD, CC BY-SA 4.0",
      h1t: "On-device", h1d: "The photo is analysed in this browser and is not uploaded.",
      h2t: "Offline-ready", h2d: "After the first visit, it works without internet.",
      h3t: "Three languages", h3d: "English, Urdu and Uzbek interface; the Uzbek patient note is being checked.",
      h4t: "Open code", h4d: "Code and results are public under the MIT licence.",
      fine: "Roshni · VERITAS AI Hackathon 2026 · Research prototype, not a medical device",
    },
    ur: {
      dir: "rtl", name: "اردو",
      brandSub: "ذیابیطس سے آنکھ کی اسکریننگ · تحقیقی نمونہ",
      langLabel: "زبان",
      chipLocal: "اسی براؤزر میں چلتا ہے", chipPhoto: "تصویر اپ لوڈ نہیں ہوتی", chipProto: "تحقیقی نمونہ",
      kicker: "بنیادی صحت مرکز · پاکستان",
      title: "ریٹینا کی تصویر کا جائزہ",
      lede: "ایسے ذیابیطس کلینک کے لیے جہاں آنکھوں کا ماہر موجود نہ ہو۔ ریٹینا کی ایک تصویر اپ لوڈ کریں؛ روشنی ڈاکٹر کی تصدیق کے لیے ایک اگلا قدم تجویز کرتی ہے۔",
      privacy: "تصویر اسی براؤزر کے اندر پراسیس ہوتی ہے اور اپ لوڈ نہیں کی جاتی۔ پہلی بار کھولنے کے بعد یہ انٹرنیٹ کے بغیر بھی کام کرتی ہے۔",
      disclaimer: "طلبہ کے ہیکاتھون کے لیے تحقیقی نمونہ۔ یہ طبی مشورہ یا تشخیص نہیں، اور پاکستان میں مریضوں پر اس کی جانچ نہیں ہوئی۔",
      steps: ["ریٹینا کی تصویر اپ لوڈ کریں", "تصویر کا معیار جانچیں", "تجزیہ", "نتیجہ دیکھیں"],
      stepHints: ["JPG یا PNG تصویر", "کیا تصویر کافی واضح ہے؟", "بیماری کا اسکور", "ڈاکٹر کے لیے اگلا قدم"],
      uploadTitle: "ریٹینا کی تصویر اپ لوڈ کریں",
      uploadHint: "کلینک کے کیمرے یا اپنی فائلوں سے فنڈس تصویر · JPG یا PNG · یہاں گھسیٹیں یا کلک کریں",
      uploadBtn: "تصویر منتخب کریں", replaceBtn: "تصویر تبدیل کریں", previewCap: "منتخب تصویر",
      examplesTitle: "یا کوئی مثال آزمائیں",
      examplesCredit: "مثالی تصاویر: DeepDRiD (Liu et al., Patterns 2022)، CC BY-SA 4.0۔ عنوان ماہرین کا درجہ اور تصویر کا معیار بتاتے ہیں، آلے کا جواب نہیں۔",
      exGrade: "ماہرین کا درجہ", exQuality: "تصویر کا معیار", good: "اچھا", poor: "کمزور",
      exMissed: "جان بوجھ کر دکھائی گئی: ایسی آنکھ جسے روشنی نہیں پکڑ پاتی",
      noteLangs: "مریض کے نوٹ کی زبانیں", checking: "ترجمہ زیرِ جانچ",
      status: {
        loading: "ماڈل اس ڈیوائس پر لوڈ ہو رہا ہے…", ready: "تیار۔ ریٹینا کی تصویر اپ لوڈ کریں یا کوئی مثال منتخب کریں۔",
        working: "اسی ڈیوائس پر تصویر کا معیار جانچا اور تجزیہ کیا جا رہا ہے…", done: "مکمل۔ تصویر اپ لوڈ نہیں کی گئی۔",
        errModel: "ماڈل لوڈ نہیں ہو سکا: ", errImage: "یہ تصویر نہیں کھل سکی۔ براہِ کرم JPG یا PNG ریٹینا تصویر منتخب کریں۔", errRun: "کچھ غلط ہو گیا: ",
      },
      offlineReady: "اس ڈیوائس پر محفوظ: اب انٹرنیٹ کے بغیر بھی کام کرے گا۔",
      resultTitle: "نتیجہ", empty: "شروع کرنے کے لیے تصویر اپ لوڈ کریں۔ نتیجہ یہاں ظاہر ہوگا۔",
      kicker2: "تجویز کردہ اگلا قدم · ڈاکٹر کی تصدیق کے لیے",
      R: {
        REFER: { tag: "ریفر", head: "ذیابیطس سے ریٹینا کی ایسی بیماری کا شبہ جس پر ریفر کرنا ضروری ہے", body: "درمیانے یا اس سے زیادہ درجے کی بیماری ہو سکتی ہے۔ معالج کا معائنہ ضروری ہے۔", next: "اگلا قدم: ڈاکٹر کی تصدیق کے بعد آنکھوں کے ماہر کے پاس بھیجیں۔" },
        RETAKE: { tag: "دوبارہ تصویر", head: "تصویر دوبارہ لینی ہوگی", body: "تصویر اتنی واضح نہیں کہ بیماری کو رد کیا جا سکے۔", next: "اگلا قدم: تصویر دوبارہ لیں۔ اگر پھر بھی واضح نہ ہو تو آنکھوں کے ماہر کے پاس بھیجیں۔" },
        RESCREEN: { tag: "دوبارہ اسکریننگ", head: "اس قابلِ جائزہ تصویر میں ریفر کے قابل بیماری نہیں ملی", body: "یہ اس بات کی ضمانت نہیں کہ آنکھ صحت مند ہے، اور آنکھ کی دوسری بیماریاں نہیں جانچی جاتیں۔", next: "اگلا قدم: ڈاکٹر کی ہدایت کے مطابق آنکھوں کے معمول کے معائنے جاری رکھیں (عام طور پر ہر 12 ماہ بعد)۔" },
      },
      scores: (r, rt, p, pt) => `بیماری کا اسکور ${r} (${rt} یا زیادہ پر ریفر) · کمزور تصویر کا اسکور ${p} (${pt} یا زیادہ پر دوبارہ تصویر)`,
      timing: s => `اس ڈیوائس پر ${s} سیکنڈ میں پراسیس ہوا۔`,
      noteTitle: "ڈاکٹر اور مریض کے لیے نوٹ",
      imagesTitle: "اصل تصویر اور ہیٹ میپ",
      photoCap: "تصویر جیسے ماڈل دیکھتا ہے", heatCap: "ریفر کے قابل بیماری کا ہیٹ میپ (Grad-CAM)",
      heatWhy: "ہیٹ میپ دکھاتا ہے کہ تصویر کے کن حصوں نے اسکور پر اثر ڈالا۔ یہ بیماری کے ہونے یا نہ ہونے کا ثبوت نہیں۔",
      gradesTitle: "اندازاً شدت (ماڈل کا نتیجہ)",
      grades: ["کوئی DR نہیں", "ہلکی", "درمیانی", "شدید", "پرولیفریٹو"],
      newBtn: "نیا جائزہ شروع کریں",
      evidenceTitle: "شواہد اور حدود",
      results: (a, b) => `بھارت کی ${a.n} نئی تصاویر پر: حساسیت ${a.se}%، مخصوصیت ${a.sp}%۔ چینی ہسپتالوں کی ${b.n} ایسی تصاویر پر جو ماڈل نے پہلے نہیں دیکھیں: حساسیت ${b.se}%، مخصوصیت ${b.sp}%۔`,
      thr: (t, s) => `ریفرل کی حد ${t}، جانچ سے پہلے طے کی گئی (ویلیڈیشن تصاویر پر ${s}% حساسیت کے لیے)۔`,
      model: "ماڈل: EfficientNet-B0، APTOS 2019 (بھارت) پر تربیت یافتہ، DeepDRiD (چین) پر جانچا گیا۔ پاکستانی مریضوں یا کم قیمت کیمروں پر نہیں جانچا گیا۔ میکولر اوڈیما کی نشاندہی نہیں کرتا۔",
      repo: "کوڈ، نتائج اور حدود",
      ctaStart: "جائزہ شروع کریں", ctaEvidence: "شواہد اور حدود",
      heroCap: "ریٹینا کی مثالی تصویر · DeepDRiD، CC BY-SA 4.0",
      h1t: "اسی ڈیوائس پر", h1d: "تصویر کا تجزیہ اسی براؤزر میں ہوتا ہے اور اسے اپ لوڈ نہیں کیا جاتا۔",
      h2t: "آف لائن تیار", h2d: "پہلی بار کھولنے کے بعد انٹرنیٹ کے بغیر کام کرتا ہے۔",
      h3t: "تین زبانیں", h3d: "انگریزی، اردو اور ازبک انٹرفیس؛ ازبک میں مریض کا نوٹ زیرِ جانچ ہے۔",
      h4t: "اوپن کوڈ", h4d: "کوڈ اور نتائج MIT لائسنس کے تحت عوامی ہیں۔",
      fine: "روشنی · VERITAS AI ہیکاتھون 2026 · تحقیقی نمونہ، طبی آلہ نہیں",
    },
    uz: {
      dir: "ltr", name: "Oʻzbekcha",
      brandSub: "Diabetik koʻz skriningi · tadqiqot prototipi",
      langLabel: "Til",
      chipLocal: "Shu brauzerda ishlaydi", chipPhoto: "Rasm yuklab yuborilmaydi", chipProto: "Tadqiqot prototipi",
      kicker: "Birlamchi tibbiy yordam · Pokiston",
      title: "Toʻr parda tasvirini baholash",
      lede: "Koʻz mutaxassisi boʻlmagan birlamchi diabet klinikasi uchun. Bitta toʻr parda (retina) suratini yuklang; Roshni shifokor tasdiqlashi uchun bitta keyingi qadamni taklif qiladi.",
      privacy: "Rasm shu brauzer ichida qayta ishlanadi va hech qayerga yuklanmaydi. Birinchi tashrifdan keyin vosita internetsiz ham ishlaydi.",
      disclaimer: "Talabalar xakatoni uchun tadqiqot prototipi. Tibbiy maslahat yoki tashxis emas va Pokistondagi bemorlarda sinovdan oʻtkazilmagan.",
      steps: ["Toʻr parda rasmini yuklash", "Rasm sifatini tekshirish", "Tahlil", "Natijani koʻrish"],
      stepHints: ["JPG yoki PNG fundus rasm", "Rasm yetarlicha aniqmi?", "Kasallik balli", "Shifokor uchun keyingi qadam"],
      uploadTitle: "Toʻr parda rasmini yuklang",
      uploadHint: "Klinika kamerasidan yoki fayllaringizdan fundus rasm · JPG yoki PNG · bu yerga torting yoki bosing",
      uploadBtn: "Rasm tanlash", replaceBtn: "Rasmni almashtirish", previewCap: "Tanlangan rasm",
      examplesTitle: "Yoki namunaviy rasmni sinab koʻring",
      examplesCredit: "Namunaviy rasmlar: DeepDRiD (Liu et al., Patterns 2022), CC BY-SA 4.0. Izohlar vosita javobini emas, mutaxassislar bahosi va rasm sifatini koʻrsatadi.",
      exGrade: "Mutaxassis bahosi", exQuality: "rasm sifati", good: "yaxshi", poor: "past",
      exMissed: "Ataylab koʻrsatilgan: Roshni oʻtkazib yuboradigan koʻzlardan biri",
      noteLangs: "Bemor eslatmasi tillari", checking: "tarjima tekshirilmoqda",
      status: {
        loading: "Model shu qurilmaga yuklanmoqda…", ready: "Tayyor. Toʻr parda rasmini yuklang yoki namunani tanlang.",
        working: "Shu qurilmada rasm sifati tekshirilmoqda va tahlil qilinmoqda…", done: "Tayyor. Rasm yuklab yuborilmadi.",
        errModel: "Modelni yuklab boʻlmadi: ", errImage: "Bu rasmni ochib boʻlmadi. JPG yoki PNG toʻr parda rasmini tanlang.", errRun: "Xatolik yuz berdi: ",
      },
      offlineReady: "Shu qurilmada saqlandi: endi internetsiz ham ishlaydi.",
      resultTitle: "Natija", empty: "Boshlash uchun rasm yuklang. Natija shu yerda chiqadi.",
      kicker2: "Tavsiya etilgan keyingi qadam · shifokor tasdiqlashi uchun",
      R: {
        REFER: { tag: "YUBORISH", head: "Yuborishni talab qiluvchi diabetik retinopatiya gumon qilinmoqda", body: "Oʻrtacha yoki undan ogʻir diabetik retinopatiya boʻlishi mumkin. Shifokor koʻrigi zarur.", next: "Keyingi qadam: shifokor tasdiqlagach, koʻz mutaxassisiga yuboring." },
        RETAKE: { tag: "QAYTA OLISH", head: "Rasmni qayta olish kerak", body: "Rasm kasallikni istisno qilish uchun yetarlicha aniq emas.", next: "Keyingi qadam: rasmni qayta oling. Hali ham noaniq boʻlsa, koʻz mutaxassisiga yuboring." },
        RESCREEN: { tag: "QAYTA SKRINING", head: "Baholash mumkin boʻlgan bu rasmda yuborishni talab qiluvchi diabetik retinopatiya aniqlanmadi", body: "Bu koʻz sogʻlom ekanining kafolati emas, boshqa koʻz kasalliklari tekshirilmaydi.", next: "Keyingi qadam: shifokor maslahatiga koʻra muntazam koʻz tekshiruvlarini davom ettiring (odatda har 12 oyda)." },
      },
      scores: (r, rt, p, pt) => `Kasallik balli ${r} (${rt} va undan yuqori boʻlsa yuborish) · Past sifat balli ${p} (${pt} va undan yuqori boʻlsa qayta olish)`,
      timing: s => `Shu qurilmada ${s} soniyada qayta ishlandi.`,
      noteTitle: "Shifokor va bemor uchun eslatma",
      imagesTitle: "Asl rasm va issiqlik xaritasi",
      photoCap: "Model koʻradigan rasm", heatCap: "Yuborishni talab qiluvchi kasallik uchun issiqlik xaritasi (Grad-CAM)",
      heatWhy: "Issiqlik xaritasi rasmning qaysi qismlari ballga taʼsir qilganini koʻrsatadi. Bu kasallik bor yoki yoʻqligining isboti emas.",
      gradesTitle: "Taxminiy ogʻirlik darajasi (model natijasi)",
      grades: ["DR yoʻq", "Yengil", "Oʻrtacha", "Ogʻir", "Proliferativ"],
      newBtn: "Yangi baholashni boshlash",
      evidenceTitle: "Dalillar va cheklovlar",
      results: (a, b) => `Hindistondan ${a.n} ta yangi rasmda: sezgirlik ${a.se}%, oʻziga xoslik ${a.sp}%. Model koʻrmagan Xitoy shifoxonalaridan ${b.n} ta rasmda: sezgirlik ${b.se}%, oʻziga xoslik ${b.sp}%.`,
      thr: (t, s) => `Yuborish chegarasi ${t}, sinovdan oldin belgilangan (validatsiya rasmlarida ${s}% sezgirlik uchun).`,
      model: "Model: EfficientNet-B0, APTOS 2019 (Hindiston) asosida oʻqitilgan, DeepDRiD (Xitoy) da sinovdan oʻtkazilgan. Pokistonlik bemorlarda yoki arzon kameralarda sinalmagan. Makula shishini aniqlamaydi.",
      repo: "Kod, natijalar va cheklovlar",
      ctaStart: "Baholashni boshlash", ctaEvidence: "Dalillar va cheklovlar",
      heroCap: "Namunaviy toʻr parda rasmi · DeepDRiD, CC BY-SA 4.0",
      h1t: "Qurilmaning oʻzida", h1d: "Rasm shu brauzerda tahlil qilinadi va yuklab yuborilmaydi.",
      h2t: "Oflayn ishlaydi", h2d: "Birinchi tashrifdan keyin internetsiz ishlaydi.",
      h3t: "Uch til", h3d: "Ingliz, urdu va oʻzbek tilidagi interfeys; oʻzbekcha bemor eslatmasi tekshirilmoqda.",
      h4t: "Ochiq kod", h4d: "Kod va natijalar MIT litsenziyasi ostida ochiq.",
      fine: "Roshni · VERITAS AI Hackathon 2026 · Tadqiqot prototipi, tibbiy qurilma emas",
    },
  };
  /* Uzbek interface and patient note: drafts until Laziza (native speaker) approves them.
     Our rule: no language is shown until a native speaker has checked it. Set to true once she replies "approved". */
  const SHOW_UZBEK = false;
  if (!SHOW_UZBEK) {
    delete UI.uz;
    UI.en.h3t = "English and Urdu"; UI.en.h3d = "Interface and patient note in English and Urdu; the Urdu was checked by a native speaker.";
    UI.ur.h3t = "انگریزی اور اردو"; UI.ur.h3d = "انٹرفیس اور مریض کا نوٹ انگریزی اور اردو میں؛ اردو کی جانچ اہلِ زبان نے کی ہے۔";
  }
  if (SHOW_UZBEK) {   // once Laziza has approved: say so, in every interface language (UR line: Zainab to check; UZ line: Laziza)
    UI.en.h3d = "Interface and patient note in English, Urdu and Uzbek, each checked by a native speaker on our team.";
    UI.ur.h3d = "انٹرفیس اور مریض کا نوٹ انگریزی، اردو اور ازبک میں؛ ہر زبان کی جانچ ہماری ٹیم کے اہلِ زبان رکن نے کی ہے۔";
    UI.uz.h3d = "Interfeys va bemor eslatmasi ingliz, urdu va oʻzbek tillarida; har bir tilni jamoamizdagi ona tili egasi tekshirgan.";
  }
  /* Uzbek patient note (draft; used only if roshni_web.json has no Uzbek note yet). */
  const NOTE_UZ = {
    REFER: "Yuborishni talab qiluvchi diabetik retinopatiya gumon qilinmoqda (oʻrtacha yoki ogʻirroq). Shifokor tasdiqlashi uchun tavsiya etilgan keyingi qadam: koʻz mutaxassisiga yuborish.",
    RETAKE: "Rasm kasallikni istisno qilish uchun yetarlicha aniq emas. Tavsiya etilgan keyingi qadam: rasmni qayta oling; hali ham noaniq boʻlsa, koʻz mutaxassisiga yuboring.",
    RESCREEN: "Bu rasmda yuborishni talab qiluvchi diabetik retinopatiya aniqlanmadi; bu koʻz kasalligini istisno qilmaydi. Tavsiya etilgan keyingi qadam: shifokor maslahatiga koʻra muntazam koʻz tekshiruvlarini davom ettiring (odatda har 12 oyda).",
    DISCLAIMER: "Talabalar xakatoni uchun prototip. Tibbiy maslahat yoki tashxis emas.",
  };

  let cfg = null, session = null, lastDecision = null, lastScores = null, statusKey = "loading", statusExtra = "", offlineOK = false;
  let runSeq = 0, objURL = null;   // runSeq: show only the latest photo's result; objURL: the chosen file, released when replaced
  let lang = "en";
  try { const s = localStorage.getItem("roshni-lang"); if (s && UI[s]) lang = s; } catch (e) { /* storage unavailable */ }
  const $ = id => document.getElementById(id);
  const T = () => UI[lang];

  function setStage(s) { document.body.dataset.stage = s; }
  function setStatus(key, extra) {
    statusKey = key; statusExtra = extra || "";
    $("status").textContent = STATUS_EN[key] + statusExtra;   // English, read by tests and the video script
    $("statusText").textContent = T().status[key] + statusExtra;
    $("statusText").dataset.kind = key;
  }

  /* ---------- interface language ---------- */
  function applyLang() {
    const t = T();
    document.documentElement.lang = lang;
    document.documentElement.dir = t.dir;
    document.querySelectorAll("[data-i18n]").forEach(el => { el.textContent = t[el.dataset.i18n]; });
    document.querySelectorAll("[data-step]").forEach(el => {
      const i = +el.dataset.step;
      el.querySelector("b").textContent = t.steps[i];
      el.querySelector("small").textContent = t.stepHints[i];
    });
    document.querySelectorAll("#uilang button").forEach(b => b.setAttribute("aria-pressed", b.dataset.lang === lang));
    $("statusText").textContent = t.status[statusKey] + statusExtra;
    if (offlineOK) $("offline").textContent = t.offlineReady;
    if (cfg) { buildFooter(); translateExamples(); relabelNoteLangs(); }
    if (lastDecision) { renderResultText(lastDecision); renderNote(lastDecision); }
    if (window.roshniLast) renderGrades(window.roshniLast.P);
  }
  function buildUILang() {
    const box = $("uilang");
    for (const code of Object.keys(UI)) {
      const b = document.createElement("button");
      b.type = "button"; b.dataset.lang = code; b.textContent = UI[code].name;
      b.lang = code;
      b.addEventListener("click", () => {
        lang = code;
        try { localStorage.setItem("roshni-lang", code); } catch (e) { /* ignore */ }
        applyLang();
      });
      box.appendChild(b);
    }
  }

  async function init() {
    buildUILang();
    applyLang();
    setStage("idle");
    try {
      setStatus("loading");
      cfg = await (await fetch("roshni_web.json")).json();
      // Uzbek patient note: a draft that the page marks "translation being checked"
      if (SHOW_UZBEK && !cfg.langs.uz) {
        cfg.langs.uz = { name: "Oʻzbekcha", dir: "ltr", checked: false };
        for (const k of Object.keys(NOTE_UZ)) if (cfg.report[k]) cfg.report[k].uz = NOTE_UZ[k];
      }
      ort.env.wasm.numThreads = 1;
      // First visit downloads about 31 MB (model + runtime) once; show how far it has got, so a slow
      // connection does not look like a frozen page. Later visits load from this device's cache.
      const [modelBytes, wasmBytes] = await fetchAll(["model.onnx", "ort/ort-wasm-simd-threaded.wasm"],
                                                    pct => setStatus("loading", ` ${pct}%`));
      ort.env.wasm.wasmBinary = wasmBytes.buffer;
      session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
      buildLanguagePicker();
      buildExamples();
      buildFooter();
      setStatus("ready");
      $("file").disabled = false;
      enableOffline();
    } catch (e) {
      setStatus("errModel", e.message);
      console.error(e);
    }
  }

  /* Download several files together and report one combined percentage. Chunks are collected and joined at the
     end, so a missing or compressed Content-Length can never break the download (the percentage then just stays hidden). */
  async function fetchAll(urls, onPct) {
    const resps = await Promise.all(urls.map(async u => {
      const r = await fetch(u);
      if (!r.ok) throw new Error(`${u}: HTTP ${r.status}`);
      return r;
    }));
    const totals = resps.map(r => +r.headers.get("Content-Length") || 0);
    const total = totals.every(t => t > 0) ? totals.reduce((a, b) => a + b, 0) : 0;
    let got = 0, last = -1;
    return Promise.all(resps.map(async r => {
      if (!r.body || !total) return new Uint8Array(await r.arrayBuffer());
      const reader = r.body.getReader(), parts = [];
      let n = 0;
      for (;;) {
        const { done, value } = await reader.read();
        if (done) break;
        parts.push(value); n += value.length; got += value.length;
        const pct = Math.min(99, Math.floor(100 * got / total));
        if (pct !== last) { last = pct; onPct(pct); }
      }
      const out = new Uint8Array(n);
      let o = 0;
      for (const p of parts) { out.set(p, o); o += p.length; }
      return out;
    }));
  }

  /* Keep the page, model and runtime on this device so screening works with no internet next time. */
  async function enableOffline() {
    if (!("serviceWorker" in navigator) || !window.caches) return;
    try {
      await navigator.serviceWorker.register("sw.js");
      await navigator.serviceWorker.ready;
      for (let i = 0; i < 60; i++) {   // the first visit downloads about 30 MB into the cache
        if (await caches.match("model.onnx") && await caches.match("ort/ort-wasm-simd-threaded.wasm")) {
          offlineOK = true;
          $("offline").textContent = T().offlineReady;
          window.roshniOfflineReady = true;
          return;
        }
        await new Promise(r => setTimeout(r, 1000));
      }
    } catch (e) {
      console.warn("offline mode unavailable", e);
    }
  }

  function buildLanguagePicker() {
    const box = $("langs");
    for (const [code, v] of Object.entries(cfg.langs)) {
      const lab = document.createElement("label");
      const cb = document.createElement("input");
      cb.type = "checkbox";
      cb.value = code;
      cb.checked = cfg.default_langs.includes(code);
      cb.addEventListener("change", () => { if (lastDecision) renderNote(lastDecision); });
      lab.appendChild(cb);
      const span = document.createElement("span");
      span.dataset.code = code;
      lab.appendChild(span);
      box.appendChild(lab);
    }
    relabelNoteLangs();
  }
  function relabelNoteLangs() {
    document.querySelectorAll("#langs span[data-code]").forEach(s => {
      const v = cfg.langs[s.dataset.code];
      s.textContent = v.name + (v.checked ? "" : ` (${T().checking})`);
    });
  }

  function captionFor(c) {
    const t = T();
    const m = /^Expert grade (\d) \((.+?)\), photo quality (good|poor)(\. Shown on purpose.*)?$/.exec(c || "");
    if (!m || lang === "en") return c;
    return `${t.exGrade} ${m[1]} (${t.grades[+m[1]]})، ${t.exQuality} ${t[m[3]]}${m[4] ? ". " + t.exMissed : ""}`.replace("،", lang === "ur" ? "،" : ",");
  }
  function buildExamples() {
    const box = $("examples");
    for (const ex of cfg.examples || []) {
      const b = document.createElement("button");
      b.className = "ex";
      b.type = "button";
      b.dataset.caption = ex.caption;
      const img = document.createElement("img");
      img.src = ex.file;
      img.alt = ex.caption;
      b.appendChild(img);
      const cap = document.createElement("span");
      b.appendChild(cap);
      b.addEventListener("click", () => { document.querySelectorAll(".ex").forEach(x => x.classList.remove("on")); b.classList.add("on"); loadURL(ex.file); });
      box.appendChild(b);
    }
    translateExamples();
  }
  function translateExamples() {
    document.querySelectorAll(".ex").forEach(b => {
      const c = captionFor(b.dataset.caption);
      b.title = c;
      b.querySelector("span").textContent = c;
    });
    $("excredit").textContent = cfg.examples_credit ? T().examplesCredit : "";
  }

  function buildFooter() {
    const s = cfg.summary || {}, t = T();
    const f = (x) => ({ n: x.n.toLocaleString("en-US"), se: (100 * x.sensitivity).toFixed(1), sp: (100 * x.specificity).toFixed(1) });
    $("results").textContent = (s.fine_tuned && s.external_all) ? t.results(f(s.fine_tuned), f(s.external_all)) : "";
    $("thr").textContent = t.thr(cfg.ref_threshold.toFixed(3), (100 * cfg.target_sensitivity).toFixed(0));
    if (cfg.repo_url) { $("repo").href = cfg.repo_url; $("repo").hidden = false; }
  }

  /* ---------- choosing a photo ---------- */
  function showPreview(url) {
    $("preview").src = url;
    document.body.classList.add("has-image");
  }
  function loadURL(url) {
    const img = new Image();
    img.onload = () => { showPreview(url); run(img); };
    img.onerror = () => { setStage("idle"); setStatus("errImage"); };
    img.src = url;
  }

  function fileURL(f) {
    if (objURL) URL.revokeObjectURL(objURL);
    objURL = URL.createObjectURL(f);
    return objURL;
  }
  $("file").addEventListener("change", e => {
    const f = e.target.files[0];
    if (!f) return;
    document.querySelectorAll(".ex").forEach(x => x.classList.remove("on"));
    loadURL(fileURL(f));
    e.target.value = "";   // choosing the same file again still runs
  });
  const drop = $("drop");
  ["dragenter", "dragover"].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); if (!$("file").disabled) drop.classList.add("over"); }));
  ["dragleave", "drop"].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.remove("over"); }));
  drop.addEventListener("drop", e => {
    if ($("file").disabled) return;
    const f = e.dataTransfer.files && e.dataTransfer.files[0];
    if (f) { document.querySelectorAll(".ex").forEach(x => x.classList.remove("on")); loadURL(fileURL(f)); }
  });

  $("newBtn").addEventListener("click", () => {
    lastDecision = null; window.roshniLast = undefined;
    ["card", "notebox", "heatbox"].forEach(id => { $(id).hidden = true; });
    $("file").value = "";
    $("preview").removeAttribute("src");
    if (objURL) { URL.revokeObjectURL(objURL); objURL = null; }
    document.body.classList.remove("has-image");
    document.querySelectorAll(".ex").forEach(x => x.classList.remove("on"));
    setStage("idle");
    setStatus("ready");
    $("tool").scrollIntoView({ behavior: "smooth", block: "start" });
  });

  /* ---------- screening (unchanged) ---------- */
  function prepare(img) {
    const S = cfg.size;
    const c0 = document.createElement("canvas");
    c0.width = img.naturalWidth; c0.height = img.naturalHeight;
    const x0 = c0.getContext("2d", { willReadFrequently: true });
    x0.drawImage(img, 0, 0);
    const box = R.cropBox(x0.getImageData(0, 0, c0.width, c0.height).data, c0.width, c0.height, cfg.crop_tol);
    const side = Math.max(box.w, box.h);
    const c1 = document.createElement("canvas");
    c1.width = side; c1.height = side;
    const x1 = c1.getContext("2d", { willReadFrequently: true });
    x1.fillStyle = "#000"; x1.fillRect(0, 0, side, side);
    x1.drawImage(c0, box.x, box.y, box.w, box.h, Math.floor((side - box.w) / 2), Math.floor((side - box.h) / 2), box.w, box.h);
    const src = x1.getImageData(0, 0, side, side).data;
    const px = side === S ? src : R.resizePIL(src, side, side, S, S);
    const c2 = $("photo");
    c2.width = S; c2.height = S;
    c2.getContext("2d").putImageData(new ImageData(new Uint8ClampedArray(px), S, S), 0, 0);
    const t = new Float32Array(3 * S * S);
    for (let i = 0; i < S * S; i++) {
      for (let ch = 0; ch < 3; ch++) t[ch * S * S + i] = (px[i * 4 + ch] / 255 - cfg.mean[ch]) / cfg.std[ch];
    }
    return { tensor: new ort.Tensor("float32", t, [1, 3, S, S]), pixels: px };
  }

  async function run(img) {
    const my = ++runSeq;   // if photos are chosen quickly one after another, show only the last one's result
    try {
      setStage("working");
      setStatus("working");
      await new Promise(r => requestAnimationFrame(() => setTimeout(r, 0)));   // let the stage indicator paint first
      const t0 = performance.now();
      const { tensor, pixels } = prepare(img);
      const out = await session.run({ image: tensor });
      if (my !== runSeq) return;
      const P = R.softmax(Array.from(out.logits.data));
      const emb = out.features.data;
      const pRef = R.referableProb(P, cfg.referable_from);
      const pPoor = R.poorProb(emb, cfg.gate);
      const d = R.decide(pRef, pPoor, cfg.ref_threshold, cfg.poor_threshold);
      const ms = performance.now() - t0;
      window.roshniLast = { P: P, features: Array.from(emb) };   // for automated checks
      const [, C, H, W] = out.feature_map.dims;
      const cam = R.heatmap(out.feature_map.data, C, H, W, P, cfg.head_weight, cfg.referable_from);
      drawHeat(cam, H, W, pixels);
      renderCard(d, pRef, pPoor, ms);
      renderGrades(P);
      renderNote(d);
      lastDecision = d;
      setStage("done");
      setStatus("done");
      // On a phone the result sits below the photo list: bring it into view so the answer is not missed.
      const card = $("card");
      if (card && card.getBoundingClientRect().top > window.innerHeight * 0.6) card.scrollIntoView({ behavior: "smooth", block: "start" });
    } catch (e) {
      if (my !== runSeq) return;
      setStage("idle");
      setStatus("errRun", e.message);
      console.error(e);
    }
  }

  function drawHeat(cam, H, W, pixels) {
    const S = cfg.size;
    const small = document.createElement("canvas");
    small.width = W; small.height = H;
    const xs = small.getContext("2d");
    const im = xs.createImageData(W, H);
    for (let i = 0; i < H * W; i++) { const v = Math.round(cam[i] * 255); im.data.set([v, v, v, 255], i * 4); }
    xs.putImageData(im, 0, 0);
    const big = document.createElement("canvas");
    big.width = S; big.height = S;
    const xb = big.getContext("2d", { willReadFrequently: true });
    xb.imageSmoothingEnabled = true;
    xb.drawImage(small, 0, 0, S, S);
    const h = xb.getImageData(0, 0, S, S).data;
    const c = $("heat");
    c.width = S; c.height = S;
    const xc = c.getContext("2d");
    const o = xc.createImageData(S, S);
    for (let i = 0; i < S * S; i++) {
      const a = 0.45 * h[i * 4] / 255;
      o.data[i * 4] = pixels[i * 4] * (1 - a) + 255 * a;
      o.data[i * 4 + 1] = pixels[i * 4 + 1] * (1 - a) + 40 * a;
      o.data[i * 4 + 2] = pixels[i * 4 + 2] * (1 - a);
      o.data[i * 4 + 3] = 255;
    }
    xc.putImageData(o, 0, 0);
    $("heatbox").hidden = false;
  }

  /* ---------- showing the result ---------- */
  function renderCard(d, pRef, pPoor, ms) {
    const card = $("card");
    card.hidden = false;
    card.dataset.decision = d;
    // English originals (hidden), unchanged, for the tests
    $("decision").textContent = TITLES[d];
    $("decision").style.color = COLORS[d];
    const poor = cfg.gate ? `${pPoor.toFixed(2)} (retake at ${cfg.poor_threshold.toFixed(2)} or above)` : "not available";
    $("scores").textContent = `Referable-disease score ${pRef.toFixed(2)} (refer at ${cfg.ref_threshold.toFixed(2)} or above) · Poor-photo score ${poor}`;
    $("timing").textContent = `Screened on this device in ${(ms / 1000).toFixed(2)} s.`;
    lastScores = { r: pRef.toFixed(2), rt: cfg.ref_threshold.toFixed(2), p: cfg.gate ? pPoor.toFixed(2) : "–", pt: cfg.poor_threshold.toFixed(2), s: (ms / 1000).toFixed(2) };
    renderResultText(d);
  }
  function renderResultText(d) {
    const t = T(), r = t.R[d];
    $("rTag").textContent = r.tag;
    $("rHead").textContent = r.head;
    $("rBody").textContent = r.body;
    $("rNext").textContent = r.next;
    if (lastScores) {
      $("rScores").textContent = t.scores(lastScores.r, lastScores.rt, lastScores.p, lastScores.pt);
      $("rTiming").textContent = t.timing(lastScores.s);
    }
  }

  function renderGrades(P) {
    const box = $("grades");
    box.replaceChildren();
    P.forEach((p, k) => {
      const row = document.createElement("div");
      row.className = "g";
      const lab = document.createElement("span");
      lab.textContent = `${k} ${T().grades[k] || cfg.grades[k]}`;
      const bar = document.createElement("span");
      bar.className = "bar";
      const fill = document.createElement("span");
      fill.style.width = `${(100 * p).toFixed(1)}%`;
      bar.appendChild(fill);
      const val = document.createElement("span");
      val.className = "pct";
      val.textContent = `${(100 * p).toFixed(0)}%`;
      row.append(lab, bar, val);
      box.appendChild(row);
    });
  }

  function renderNote(d) {
    const chosen = Array.from(document.querySelectorAll("#langs input:checked")).map(x => x.value);
    const langs = chosen.length ? chosen : ["en"];
    const box = $("note");
    box.replaceChildren();
    for (const code of Object.keys(cfg.langs)) {
      if (!langs.includes(code)) continue;
      if (!cfg.report[d][code]) continue;
      const div = document.createElement("div");
      div.dir = cfg.langs[code].dir;
      div.lang = code;
      div.className = "n";
      const b = document.createElement("b");
      b.textContent = cfg.langs[code].name + (cfg.langs[code].checked ? "" : ` (${T().checking})`) + ": ";
      div.appendChild(b);
      div.appendChild(document.createTextNode(cfg.report[d][code]));
      const small = document.createElement("small");
      small.textContent = cfg.report.DISCLAIMER[code];
      div.appendChild(document.createElement("br"));
      div.appendChild(small);
      box.appendChild(div);
    }
    $("notebox").hidden = false;
  }

  init();
}


In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=False)   # for the browser-demo export
import torch, torchvision
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this will be slow)')

## 2. Data
**Training data:** APTOS 2019 (3,662 graded photos, Aravind Eye Hospital, rural India).  
**External test, never used to train the grading model:** DeepDRiD regular fundus photos, training and validation parts (1,600 graded photos from 400 patients, with photo-quality labels; China; CC BY-SA 4.0).

In [ ]:
import glob
APTOS = os.environ.get('APTOS_DIR') or next((c for c in ['/kaggle/input/aptos2019-blindness-detection',
        '/kaggle/input/competitions/aptos2019-blindness-detection', '/content/aptos', 'aptos'] if os.path.exists(os.path.join(c, 'train.csv'))), None)
if APTOS is None:   # Colab route through the Kaggle API
    from google.colab import files
    print('Upload kaggle.json'); files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.replace('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json')); os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    os.system('kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos_zip')
    os.system('cd /content/aptos_zip && unzip -q -o aptos2019-blindness-detection.zip "train_images/*" train.csv -d /content/aptos')
    APTOS = '/content/aptos'
print('APTOS folder:', APTOS, '| photos:', len(glob.glob(os.path.join(APTOS, 'train_images', '*'))))

DDR = os.environ.get('DDR_ROOT', 'DeepDRiD')
if not os.path.exists(os.path.join(DDR, 'regular_fundus_images', 'regular-fundus-validation', 'regular-fundus-validation.csv')):
    os.system(f'git clone -q --depth 1 --filter=blob:none --sparse https://github.com/deepdrdoc/DeepDRiD.git {DDR}')
    os.system(f'cd {DDR} && git sparse-checkout set regular_fundus_images/regular-fundus-training regular_fundus_images/regular-fundus-validation')
print('DeepDRiD photos:', len(glob.glob(os.path.join(DDR, 'regular_fundus_images', '*', 'Images', '*', '*.jpg'))))

## 3. Quick check (about 5 minutes)
Runs everything on 300 photos for 2 epochs, so a broken step shows up early. Its numbers mean nothing; do not report them.

In [ ]:
import run_dr
WORKERS = min(4, os.cpu_count() or 2)
SIZE = int(os.environ.get('SIZE', 384)); PRETRAINED = os.environ.get('PRETRAINED', '1') == '1'
QUICK_EPOCHS = int(os.environ.get('QUICK_EPOCHS', 2))
quick = run_dr.run(run_dr.RunConfig(f'{APTOS}/train.csv', f'{APTOS}/train_images', DDR, out='quick_check', cache='cache',
                                    size=SIZE, pretrained=PRETRAINED, epochs=QUICK_EPOCHS, workers=WORKERS, max_images=300))
quick['summary']

## 4. Full run
The 90% sensitivity target, the photo-quality cut-off (0.5) and every training setting were fixed before any result was seen. The referral threshold is set on the validation split and frozen; the APTOS test split and DeepDRiD are scored once, after that.

In [ ]:
EPOCHS = int(os.environ.get('EPOCHS', 12))
cfg = run_dr.RunConfig(f'{APTOS}/train.csv', f'{APTOS}/train_images', DDR, out='results', cache='cache',
                       size=SIZE, pretrained=PRETRAINED, epochs=EPOCHS, workers=WORKERS)
out = run_dr.run(cfg)
out['summary']

## 5. Results the Health track asks for: missed cases and false alarms, by severity and by group

In [ ]:
from IPython.display import Image as Show, display
import pandas as pd, json, traceback
try:
    print('Frozen settings:', out['frozen'])
    print('Near-duplicate photos found in APTOS:', out['duplicates'])
    display(out['aptos_flagged_by_grade'])          # how many of each grade were sent to a specialist
    print('5-grade agreement (quadratic weighted kappa):', round(out['aptos_grades']['QWK'], 3))
    if 'workflow' in out:
        display(pd.DataFrame(out['workflow']).drop(index=[], errors='ignore'))
        print('Photo-quality check:', out['quality_gate'])
        display(pd.DataFrame(out['external_by_source']))
    print('Speed on the CPU alone (no graphics card):', out.get('cpu_speed'))
    display(Show('results/roc.png'))
    if os.path.exists('results/stress_test.csv'):
        print('Stress test (planned before results): the frozen tool on deliberately worse photos')
        display(pd.read_csv('results/stress_test.csv'))
        display(Show('results/stress_examples.png')); display(Show('results/stress_test.png'))
except Exception:
    traceback.print_exc()   # a display problem must never stop the run

## 6. Where it fails
Look at each missed eye and write down a reason (blur, small lesions, edge of the photo, label doubt).

In [ ]:
te = pd.read_csv('results/aptos_test_predictions.csv')
try:
    for f in ['results/missed_cases.png', 'results/false_alarms.png', 'results/missed_external.png']:
        if os.path.exists(f): display(Show(f))
    thr = out['frozen']['ref_threshold']
    display(te[(te.grade >= 2) & (te.p_referable < thr)].sort_values('p_referable').head(10))
except Exception:
    traceback.print_exc()

## 7. Where the model looked (Grad-CAM) on a few test photos

In [ ]:
import drscreen as ds, matplotlib.pyplot as plt
from PIL import Image as PILImage
try:
    pick = pd.concat([te[te.grade == g].head(1) for g in range(5)])
    fig, axes = plt.subplots(1, len(pick), figsize=(3 * len(pick), 3.3))
    for ax, r in zip(axes, pick.itertuples()):
        ax.imshow(ds.gradcam(out['model'], PILImage.open(r.cached), cfg.size)); ax.axis('off')
        ax.set_title(f'grade {r.grade}, score {r.p_referable:.2f}', fontsize=9)
    plt.tight_layout(); plt.savefig('results/gradcam.png', dpi=120); plt.show()
except Exception:
    traceback.print_exc()

## 8. Write-up and video script, filled from results/
Every number comes from the files this run wrote. It refuses quick-check numbers and writes nothing if any value is missing. Then fill the [TEAM_TO_WRITE] parts by hand.

In [ ]:
r = subprocess.run([sys.executable, 'fill_report.py'] + (['--allow-quick'] if os.environ.get('ALLOW_QUICK') == '1' else []),
                   capture_output=True, text=True)
print(r.stdout, r.stderr)

## 9. Browser demo (the main demo)
Exports the frozen model to ONNX and checks that it gives the same answers as PyTorch. The `web/` folder is a complete website: the photo is screened inside the visitor's browser and never uploaded. Put the folder in the GitHub repository and switch on GitHub Pages to get a permanent link.

In [ ]:
import shutil
try:
    import export_web
    os.makedirs('web', exist_ok=True)
    shutil.copy('web_index.html', 'web/index.html'); shutil.copy('web_roshni.js', 'web/roshni.js')
    check = list(pd.read_csv('results/deepdrid_predictions.csv').query("part == 'validation'").path[:5]) if os.path.exists('results/deepdrid_predictions.csv') else list(te.path[:5])
    export_web.export('results', 'web', check)
    print(sorted(os.listdir('web')))
except Exception:
    traceback.print_exc()   # the results are already saved; the browser demo can be rebuilt from them later

## 10. Download everything
One zip with the results, the filled write-up and video script, and the browser demo. On a committed (Save & Run All) run, download it from the version's Output tab.

In [ ]:
pack = 'roshni_outputs'
shutil.rmtree(pack, ignore_errors=True); os.makedirs(pack)
for f in ('README.md', 'VIDEO_SCRIPT.md'):
    if os.path.exists(f): shutil.copy(f, pack)
shutil.copytree('results', f'{pack}/results')
if os.path.isdir('web'): shutil.copytree('web', f'{pack}/web')
shutil.make_archive(pack, 'zip', '.', pack)
print(f'{pack}.zip', round(os.path.getsize(pack + '.zip') / 1e6, 1), 'MB')
# keep the saved output small: the raw DeepDRiD copy and the image cache are no longer needed
if os.environ.get('KEEP_DATA') != '1':
    shutil.rmtree('DeepDRiD', ignore_errors=True); shutil.rmtree('cache', ignore_errors=True); shutil.rmtree(pack, ignore_errors=True)

## 11. Gradio app (optional, interactive sessions only)
The main demo is the browser version in section 9, which keeps working after this notebook stops. This app gives a public link for 72 hours only. To try it, set `LAUNCH_GRADIO = True`, set `KEEP_DATA=1` before running section 10 (so the example photos are kept), and run this cell.

In [ ]:
LAUNCH_GRADIO = False
if LAUNCH_GRADIO:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=False)
        os.makedirs('examples', exist_ok=True)
        if os.path.exists('results/deepdrid_predictions.csv'):
            dd = pd.read_csv('results/deepdrid_predictions.csv')
            v = dd[dd.part == 'validation']
            for name in ('RETAKE', 'REFER', 'RESCREEN'):
                for r in v[v.decision == name].head(2).itertuples():
                    if os.path.exists(r.cached): shutil.copy(r.cached, f'examples/deepdrid_{name.lower()}_{os.path.basename(r.cached)}')
        import app
        app.build_app('results', 'examples').launch(share=True)
    except Exception:
        traceback.print_exc()

## Credits
APTOS 2019 Blindness Detection (Aravind Eye Hospital, Kaggle); DeepDRiD (Liu et al., Patterns 2022, CC BY-SA 4.0); PyTorch and torchvision (EfficientNet-B0, ImageNet weights); scikit-learn; Gradio; Grad-CAM (Selvaraju et al. 2017). AI tools used: list every assistant here.